<a href="https://colab.research.google.com/github/RafaelMuniz94/RaruRevival/blob/main/Raru_Revival_Faster_Whisper_FINAL_Phase1_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Raru Revival — Etapa 1 FINAL v3
## Faster-Whisper + Repair Pass v3

Arquitetura validada para Google Colab gratuito:

1. **Passada 1:** `large-v3-turbo`, multilíngue, `chunk_length=3`, VAD desligado.
2. **Quality Analyzer:** identifica trechos suspeitos sem alterar o RAW.
3. **Repair Groups v3:** agrupa segmentos suspeitos realmente adjacentes.
4. **Passada 2:** `large-v3` com contexto maior e candidatos AUTO/PT/EN.
5. **Decision v3:** rejeita loops/scripts absurdos, trata CRITICAL e registra confiança.
6. **Outputs preservados:** RAW e `.REPAIRED` separados.

> O vídeo original nunca é modificado ou recomprimido.


### v2
- recuperação automática de GPU;
- retry em `int8_float16` e `int8` antes de CPU;
- liberação explícita de `large-v3-turbo` antes do Repair;
- diagnóstico de VRAM e erro CUDA real.


### v3
- guard para empty alignment do faster-whisper 1.2.1;
- preservação de segmentos sem word timestamps;
- sinal `missing_word_alignment` para a Passada 2.



In [1]:
# CÉLULA 1 — Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/RaruRevival')
DIRS = {
    'lives': DRIVE_ROOT / 'Lives',
    'transcriptions': DRIVE_ROOT / 'Transcriptions',
    'checkpoints': DRIVE_ROOT / 'Checkpoints',
    'logs': DRIVE_ROOT / 'Logs',
    'temp_drive': DRIVE_ROOT / 'Temp',
}

for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

print("Estrutura:")
for k, p in DIRS.items():
    print(f"  {k:15s} -> {p}")

Mounted at /content/drive
Estrutura:
  lives           -> /content/drive/MyDrive/RaruRevival/Lives
  transcriptions  -> /content/drive/MyDrive/RaruRevival/Transcriptions
  checkpoints     -> /content/drive/MyDrive/RaruRevival/Checkpoints
  logs            -> /content/drive/MyDrive/RaruRevival/Logs
  temp_drive      -> /content/drive/MyDrive/RaruRevival/Temp


In [2]:
# ============================================================
# CÉLULA 2 — Instalar dependências
# ============================================================
#
# Compatibilidade validada para o Raru Revival:
#
#   faster-whisper 1.2.1
#   CTranslate2 4.8.2
#   PyAV 18.0.0
#
# Também instalamos as bibliotecas CUDA 12 usadas pelo
# CTranslate2:
#
#   libcublas.so.12
#   cuDNN 9
#
# IMPORTANTE:
# Os pacotes nvidia.* podem ser namespace packages.
# Portanto NÃO usamos módulo.__file__, pois ele pode ser None.
# ============================================================


# ------------------------------------------------------------
# 1. Dependências
# ------------------------------------------------------------

%pip install -q \
    --no-cache-dir \
    "faster-whisper==1.2.1" \
    "ctranslate2==4.8.2" \
    "av==18.0.0" \
    "nvidia-cublas-cu12" \
    "nvidia-cudnn-cu12==9.*" \
    "psutil>=5.9" \
    "tqdm>=4.66"


# ------------------------------------------------------------
# 2. FFmpeg / ffprobe
# ------------------------------------------------------------

!ffmpeg -version | head -n 1
!ffprobe -version | head -n 1


# ------------------------------------------------------------
# 3. Imports usados apenas para preparação do runtime
# ------------------------------------------------------------

import os
import glob
import ctypes

from pathlib import Path
from importlib import import_module
from importlib.metadata import version


# ------------------------------------------------------------
# 4. Localizar diretório real de um namespace package NVIDIA
# ------------------------------------------------------------

def find_namespace_lib_dir(
    module_name
):
    module = import_module(
        module_name
    )

    # --------------------------------------------------------
    # Método principal:
    # namespace packages normalmente expõem __path__
    # --------------------------------------------------------

    package_paths = list(
        getattr(
            module,
            "__path__",
            []
        )
    )

    for package_path in (
        package_paths
    ):
        path = Path(
            package_path
        ).resolve()

        if (
            path.exists()
        ):
            return str(
                path
            )


    # --------------------------------------------------------
    # Fallback:
    # pacote tradicional com __file__
    # --------------------------------------------------------

    module_file = getattr(
        module,
        "__file__",
        None
    )

    if (
        module_file
        is not None
    ):
        path = Path(
            module_file
        ).resolve().parent

        if (
            path.exists()
        ):
            return str(
                path
            )


    raise RuntimeError(
        "Não foi possível localizar "
        f"{module_name}"
    )


CUBLAS_LIB_DIR = (
    find_namespace_lib_dir(
        "nvidia.cublas.lib"
    )
)

CUDNN_LIB_DIR = (
    find_namespace_lib_dir(
        "nvidia.cudnn.lib"
    )
)


print()
print("=" * 70)
print("BIBLIOTECAS NVIDIA")
print("=" * 70)

print(
    "cuBLAS:",
    CUBLAS_LIB_DIR
)

print(
    "cuDNN :",
    CUDNN_LIB_DIR
)


# ------------------------------------------------------------
# 5. Mostrar arquivos encontrados
# ------------------------------------------------------------

cublas_files = sorted(
    glob.glob(
        os.path.join(
            CUBLAS_LIB_DIR,
            "libcublas*.so*"
        )
    )
)

cudnn_files = sorted(
    glob.glob(
        os.path.join(
            CUDNN_LIB_DIR,
            "libcudnn*.so*"
        )
    )
)


print()
print(
    "cuBLAS encontrados:"
)

for path in (
    cublas_files
):
    print(
        " ",
        os.path.basename(
            path
        )
    )


print()
print(
    "cuDNN encontrados:"
)

for path in (
    cudnn_files
):
    print(
        " ",
        os.path.basename(
            path
        )
    )


# ------------------------------------------------------------
# 6. Garantir que encontramos as bibliotecas essenciais
# ------------------------------------------------------------

def find_library_file(
    directory,
    filename
):
    candidates = glob.glob(
        os.path.join(
            directory,
            filename
        )
    )

    if not candidates:

        raise RuntimeError(
            "\nBiblioteca não encontrada:\n"
            f"  {filename}\n\n"
            "Diretório pesquisado:\n"
            f"  {directory}"
        )

    return sorted(
        candidates
    )[0]


CUBLAS_LT_PATH = (
    find_library_file(
        CUBLAS_LIB_DIR,
        "libcublasLt.so.12"
    )
)

CUBLAS_PATH = (
    find_library_file(
        CUBLAS_LIB_DIR,
        "libcublas.so.12"
    )
)

CUDNN_PATH = (
    find_library_file(
        CUDNN_LIB_DIR,
        "libcudnn.so.9"
    )
)


# ------------------------------------------------------------
# 7. Registrar diretórios
#
# O faster-whisper recomenda LD_LIBRARY_PATH para instalações
# via pip.
#
# No notebook já estamos dentro de um processo Python.
# Portanto, além de atualizar a variável, fazemos preload
# explícito abaixo.
# ------------------------------------------------------------

existing_paths = [

    path

    for path
    in os.environ.get(
        "LD_LIBRARY_PATH",
        ""
    ).split(":")

    if path
]


for required_path in [

    CUBLAS_LIB_DIR,
    CUDNN_LIB_DIR,

]:

    if (
        required_path
        not in existing_paths
    ):
        existing_paths.insert(
            0,
            required_path
        )


os.environ[
    "LD_LIBRARY_PATH"
] = ":".join(
    existing_paths
)


# ------------------------------------------------------------
# 8. Preload explícito
#
# CTranslate2 só tenta usar cuBLAS efetivamente durante a
# inferência. Por isso carregar o modelo sozinho não detecta
# necessariamente o problema.
#
# RTLD_GLOBAL deixa as bibliotecas disponíveis para os
# componentes carregados depois.
# ------------------------------------------------------------

print()
print("=" * 70)
print("PRELOAD CUDA")
print("=" * 70)


def preload(
    path
):
    ctypes.CDLL(
        path,
        mode=ctypes.RTLD_GLOBAL
    )

    print(
        "✓",
        os.path.basename(
            path
        )
    )


# cuBLAS depende de cuBLASLt.
preload(
    CUBLAS_LT_PATH
)

preload(
    CUBLAS_PATH
)

# cuDNN principal.
preload(
    CUDNN_PATH
)


# ------------------------------------------------------------
# 9. Versões realmente instaladas
# ------------------------------------------------------------

fw_version = version(
    "faster-whisper"
)

ct2_version = version(
    "ctranslate2"
)

av_version = version(
    "av"
)

cublas_version = version(
    "nvidia-cublas-cu12"
)

cudnn_version = version(
    "nvidia-cudnn-cu12"
)


print()
print("=" * 70)
print("VERSÕES")
print("=" * 70)

print(
    "faster-whisper :",
    fw_version
)

print(
    "CTranslate2    :",
    ct2_version
)

print(
    "PyAV           :",
    av_version
)

print(
    "cuBLAS CUDA 12 :",
    cublas_version
)

print(
    "cuDNN CUDA 12  :",
    cudnn_version
)


# ------------------------------------------------------------
# 10. Validações
# ------------------------------------------------------------

assert (
    fw_version
    == "1.2.1"
), (
    "faster-whisper inesperado: "
    + fw_version
)

assert (
    ct2_version
    == "4.8.2"
), (
    "CTranslate2 inesperado: "
    + ct2_version
)

assert (
    av_version
    == "18.0.0"
), (
    "PyAV incompatível: "
    + av_version
)


# ------------------------------------------------------------
# 11. Teste direto das bibliotecas
# ------------------------------------------------------------

print()
print("=" * 70)
print("VALIDAÇÃO CUDA")
print("=" * 70)

print(
    "✓ libcublasLt.so.12 carregada"
)

print(
    "✓ libcublas.so.12 carregada"
)

print(
    "✓ libcudnn.so.9 carregada"
)


# ------------------------------------------------------------
# 12. Testar CTranslate2
# ------------------------------------------------------------

import ctranslate2

print()
print(
    "Compute types CUDA disponíveis:"
)

cuda_compute_types = (
    ctranslate2.get_supported_compute_types(
        "cuda"
    )
)

print(
    cuda_compute_types
)

assert (
    "float16"
    in cuda_compute_types
), (
    "CTranslate2 não reportou float16 na GPU."
)


print()
print("=" * 70)
print("AMBIENTE PRONTO")
print("=" * 70)

print(
    "✓ Faster-Whisper 1.2.1"
)

print(
    "✓ PyAV 18.0.0"
)

print(
    "✓ CTranslate2 4.8.2"
)

print(
    "✓ cuBLAS CUDA 12 disponível"
)

print(
    "✓ cuDNN 9 disponível"
)

print(
    "✓ T4 pronta para Faster-Whisper"
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 165.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 143.6 MB/s eta 0:00:00
ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
ffprobe version 6.1.1-3ubuntu5 Copyright (c) 2007-2023 the FFmpeg developers

BIBLIOTECAS NVIDIA
cuBLAS: /usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib
cuDNN : /usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib

cuBLAS encontrados:
  libcublas.so.12
  libcublasLt.so.12

cuDNN encontrados:
  libcudnn.so.9
  libcudnn_adv.so.9
  libcudnn_cnn.so.9
  libcudnn_engines_precompiled.so.9
  libcudnn_engines_runtime_compiled.so.9
  libcudnn_engines_tensor_ir.so.9
  libcudnn_ext.so.9
  libcudnn_graph.so.9
  libcudnn_heuristic.so.9
  libcudnn_ops.so.9

PRELOAD CUDA
✓ libcublasLt.so.12
✓ libcublas.

In [3]:
# CÉLULA 3 — Detectar hardware e tipos de computação
import os, sys, json, shutil, subprocess, platform, psutil, importlib.metadata
from pathlib import Path

def run_cmd(cmd):
    try:
        return subprocess.check_output(cmd, stderr=subprocess.STDOUT, text=True).strip()
    except Exception:
        return None

def bytes_gb(n):
    return round(n / (1024**3), 2)

def detect_hardware():
    vm = psutil.virtual_memory()
    disk = shutil.disk_usage('/content')

    gpu_name = None
    gpu_vram_mb = 0
    gpu_driver = None

    q = run_cmd([
        'nvidia-smi',
        '--query-gpu=name,memory.total,driver_version',
        '--format=csv,noheader,nounits'
    ])
    if q:
        first = q.splitlines()[0]
        parts = [x.strip() for x in first.split(',')]
        if len(parts) >= 3:
            gpu_name = parts[0]
            gpu_vram_mb = int(float(parts[1]))
            gpu_driver = parts[2]

    cuda_version = run_cmd(['bash', '-lc',
        "nvidia-smi | sed -n 's/.*CUDA Version: \\([0-9.]*\\).*/\\1/p' | head -n1"
    ])

    import ctranslate2
    gpu_types = []
    cpu_types = sorted(ctranslate2.get_supported_compute_types('cpu'))
    if gpu_name:
        try:
            gpu_types = sorted(ctranslate2.get_supported_compute_types('cuda'))
        except Exception as e:
            gpu_types = [f"ERRO: {e}"]

    return {
        'python': sys.version.split()[0],
        'platform': platform.platform(),
        'ram_total_gb': bytes_gb(vm.total),
        'ram_available_gb': bytes_gb(vm.available),
        'disk_total_gb': bytes_gb(disk.total),
        'disk_free_gb': bytes_gb(disk.free),
        'gpu_available': gpu_name is not None,
        'gpu_name': gpu_name,
        'gpu_vram_gb': round(gpu_vram_mb / 1024, 2) if gpu_name else 0,
        'gpu_driver': gpu_driver,
        'cuda_version_reported': cuda_version,
        'ct2_cpu_compute_types': cpu_types,
        'ct2_gpu_compute_types': gpu_types,
        'faster_whisper_version': importlib.metadata.version('faster-whisper'),
        'ctranslate2_version': importlib.metadata.version('ctranslate2'),
    }

HW = detect_hardware()
print(json.dumps(HW, indent=2, ensure_ascii=False))

{
  "python": "3.13.15",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "ram_total_gb": 12.67,
  "ram_available_gb": 11.19,
  "disk_total_gb": 112.64,
  "disk_free_gb": 69.91,
  "gpu_available": true,
  "gpu_name": "Tesla T4",
  "gpu_vram_gb": 15.0,
  "gpu_driver": "580.82.07",
  "cuda_version_reported": "13.0",
  "ct2_cpu_compute_types": [
    "float32",
    "int16",
    "int8",
    "int8_float32"
  ],
  "ct2_gpu_compute_types": [
    "float16",
    "float32",
    "int8",
    "int8_float16",
    "int8_float32"
  ],
  "faster_whisper_version": "1.2.1",
  "ctranslate2_version": "4.8.2"
}


In [4]:
# ============================================================
# CÉLULA 4 — Configurações do projeto
# ============================================================

QUALITY_MODE = "balanced"   # "balanced" | "max_quality"

# ------------------------------------------------------------
# Passada 1 — configuração validada
# ------------------------------------------------------------

PRIMARY_LANGUAGE = "pt"

LANGUAGE = None
MULTILINGUAL = True
TASK = "transcribe"

WORD_TIMESTAMPS = True
BEAM_SIZE = 5

TEMPERATURE = 0.0
CONDITION_ON_PREVIOUS_TEXT = False

# Importante:
# 3 segundos foi o melhor resultado no teste rápido
# PT/EN misto, mas corta contexto linguístico.
# A Passada 2 existe justamente para reparar esses casos.
WHISPER_CHUNK_LENGTH = 3

# Preserva timeline e evita concatenação agressiva.
VAD_FILTER = False

# Mantidos como documentação/config, embora o VAD esteja OFF.
VAD_PARAMETERS = {
    "threshold": 0.40,
    "min_speech_duration_ms": 0,
    "min_silence_duration_ms": 1200,
    "speech_pad_ms": 500,
}

NO_SPEECH_THRESHOLD = None
HALLUCINATION_SILENCE_THRESHOLD = None

# ------------------------------------------------------------
# Checkpoint externo / resiliência do Colab
# ------------------------------------------------------------

CHUNK_SECONDS = 15 * 60
OVERLAP_SECONDS = 2.0

# ------------------------------------------------------------
# Teste rápido
# ------------------------------------------------------------

TEST_MODE = False
TEST_SECONDS = 20 * 60

# ------------------------------------------------------------
# Preparação local
# ------------------------------------------------------------

COPY_VIDEO_TO_LOCAL_IF_POSSIBLE = True
LOCAL_RESERVE_GB = 8.0

# ------------------------------------------------------------
# Modelos
# ------------------------------------------------------------

GPU_BALANCED_MODEL = "large-v3-turbo"
GPU_MAX_MODEL = "large-v3"
CPU_FRESH_JOB_MODEL = "medium"


In [5]:
# ============================================================
# CÉLULA 4B — Configuração da Passada 2 / Repair v3
# ============================================================

ENABLE_REPAIR_PASS = True

REPAIR_MODEL_NAME = "large-v3"

# Contexto antes/depois do grupo suspeito.
REPAIR_PADDING_SECONDS = 4.0

# ------------------------------------------------------------
# Repair Groups v3
#
# Segmentos suspeitos com gap muito pequeno pertencem à mesma
# frase/região e devem ser reparados juntos.
# ------------------------------------------------------------

REPAIR_GROUP_MAX_GAP_SECONDS = 0.15
REPAIR_GROUP_MAX_CORE_SECONDS = 12.0

# ------------------------------------------------------------
# Quality Analyzer
# ------------------------------------------------------------

REPAIR_COMPRESSION_RATIO = 2.4
REPAIR_AVG_LOGPROB = -1.0

REPAIR_SHORT_SEGMENT_MAX_WORDS = 4
REPAIR_SHORT_SEGMENT_MEAN_PROB = 0.45

REPAIR_BOUNDARY_MARGIN = 0.12
REPAIR_BOUNDARY_WORD_PROB = 0.65

REPAIR_REPETITION_MIN_WORDS = 8
REPAIR_REPETITION_RATIO = 0.60

REPAIR_ZERO_DURATION_RATIO = 0.50

# Propaga suspeita para pequenos vizinhos de texto com script estranho.
REPAIR_SCRIPT_NEIGHBOR_GAP_SECONDS = 1.20

# ------------------------------------------------------------
# Validação dos candidatos da Passada 2
# ------------------------------------------------------------

REPAIR_ALLOWED_LANGUAGES = {"pt", "en"}

REPAIR_OTHER_LANGUAGE_REJECT_PROB = 0.70
REPAIR_CANDIDATE_MIN_MEAN_WORD_PROB = 0.35

REPAIR_CANDIDATE_MAX_COMPRESSION_RATIO = 2.4
REPAIR_CANDIDATE_REPETITION_MIN_WORDS = 6
REPAIR_CANDIDATE_REPETITION_RATIO = 0.55

REPAIR_CANDIDATE_MIN_COVERAGE = 0.10

# AUTO tem preferência.
# PT/EN forçado só vence se for significativamente melhor.
REPAIR_FORCED_LANGUAGE_MIN_GAIN = 0.15

# ------------------------------------------------------------
# Decisão de substituição
# ------------------------------------------------------------

REPAIR_MIN_SCORE_GAIN = 0.07

REPAIR_BOUNDARY_TRUST_MIN_SCORE = 0.75
REPAIR_BOUNDARY_MAX_SCORE_DROP = 0.02

REPAIR_CRITICAL_MIN_SCORE = 0.35

# Script inesperado já é evidência forte de que o RAW está ruim.
REPAIR_SCRIPT_MIN_SCORE = 0.65

# ------------------------------------------------------------
# Classificação de confiança para uso posterior
# ------------------------------------------------------------

REPAIR_CONFIDENCE_HIGH = 0.85
REPAIR_CONFIDENCE_MEDIUM = 0.70

# ------------------------------------------------------------
# Overlaps residuais
# ------------------------------------------------------------

REPAIR_MAX_AUTO_OVERLAP_FIX_SECONDS = 0.25


In [6]:
# CÉLULA 5 — Escolha automática de device/model/compute_type

def choose_runtime_config(hw, quality_mode="balanced", existing_progress=None):
    gpu = hw['gpu_available']
    vram = hw['gpu_vram_gb']
    gpu_types = set(x for x in hw['ct2_gpu_compute_types'] if not str(x).startswith("ERRO"))
    cpu_types = set(hw['ct2_cpu_compute_types'])

    # Se já existe checkpoint, preserve o MESMO modelo do job.
    if existing_progress and existing_progress.get("model"):
        model = existing_progress["model"]
    else:
        if gpu:
            model = GPU_MAX_MODEL if quality_mode == "max_quality" else GPU_BALANCED_MODEL
        else:
            model = CPU_FRESH_JOB_MODEL

    if gpu:
        if vram >= 8 and "float16" in gpu_types:
            compute = "float16"
        elif "int8_float16" in gpu_types:
            compute = "int8_float16"
        elif "int8" in gpu_types:
            compute = "int8"
        else:
            compute = "float32"
        return {"device": "cuda", "compute_type": compute, "model": model}
    else:
        compute = "int8" if "int8" in cpu_types else "float32"
        return {"device": "cpu", "compute_type": compute, "model": model}

RUNTIME = choose_runtime_config(HW, QUALITY_MODE)
print("Configuração escolhida:", RUNTIME)
if RUNTIME["device"] == "cpu":
    print("AVISO: GPU não disponível. O processamento continuará em CPU e será significativamente mais lento.")

Configuração escolhida: {'device': 'cuda', 'compute_type': 'float16', 'model': 'large-v3-turbo'}


In [7]:
# CÉLULA 6 — Funções de mídia e preparação local

import hashlib, time, math, re
from pathlib import Path

VIDEO_EXTS = {'.mkv', '.mp4', '.mov', '.webm', '.m4v', '.avi'}

def ffprobe_json(path):
    cmd = [
        'ffprobe', '-v', 'error',
        '-show_entries', 'format=duration,size,format_name:stream=index,codec_type,codec_name,sample_rate,channels',
        '-of', 'json', str(path)
    ]
    out = subprocess.check_output(cmd, text=True)
    return json.loads(out)

def media_duration(path):
    data = ffprobe_json(path)
    return float(data['format']['duration'])

def source_fingerprint(path, block=1024*1024):
    # Evita hash do arquivo inteiro: tamanho + início + fim.
    p = Path(path)
    size = p.stat().st_size
    h = hashlib.sha256()
    h.update(str(size).encode())
    with open(p, 'rb') as f:
        h.update(f.read(block))
        if size > block:
            f.seek(max(0, size - block))
            h.update(f.read(block))
    return h.hexdigest()

def prepare_source(source_path):
    source_path = Path(source_path)
    if not source_path.exists():
        raise FileNotFoundError(source_path)

    local_dir = Path('/content/raru_revive')
    local_dir.mkdir(parents=True, exist_ok=True)

    free_gb = shutil.disk_usage('/content').free / (1024**3)
    source_gb = source_path.stat().st_size / (1024**3)

    if COPY_VIDEO_TO_LOCAL_IF_POSSIBLE and source_gb + LOCAL_RESERVE_GB < free_gb:
        local_video = local_dir / source_path.name
        if not local_video.exists() or local_video.stat().st_size != source_path.stat().st_size:
            print(f"Copiando vídeo para SSD temporário do Colab ({source_gb:.2f} GB)...")
            shutil.copy2(source_path, local_video)
        return local_video, "local_video_copy"

    # Para vídeo grande demais, NÃO criamos um FLAC completo.
    # Cada checkpoint extrairá apenas seu bloco diretamente do Drive.
    # Isso evita refazer horas de extração após uma queda do Colab.
    print("Vídeo grande para cópia local; os blocos serão extraídos diretamente do Drive.")
    return source_path, "drive_direct_chunked"

def extract_chunk(input_path, out_path, start_s, duration_s):
    # -ss antes de -i é rápido; para áudio intermediário é suficiente.
    cmd = [
        'ffmpeg', '-hide_banner', '-loglevel', 'error', '-y',
        '-ss', f'{start_s:.3f}',
        '-i', str(input_path),
        '-t', f'{duration_s:.3f}',
        '-vn', '-ac', '1', '-ar', '16000',
        '-c:a', 'flac',
        str(out_path)
    ]
    subprocess.run(cmd, check=True)

In [8]:
# ============================================================
# CÉLULA 7 — Carregar Faster-Whisper + gestão robusta de VRAM
# ============================================================

from faster_whisper import WhisperModel
import gc


def release_gpu_memory():
    """
    Tenta devolver memória ao runtime entre trocas de modelo.

    CTranslate2 não usa o allocator do PyTorch para tudo,
    então o passo mais importante é remover as referências
    dos WhisperModel antes de chamar esta função.
    """
    gc.collect()

    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

            try:
                torch.cuda.ipc_collect()
            except Exception:
                pass

    except Exception:
        pass

    gc.collect()


def gpu_memory_snapshot(label=None):
    """
    Diagnóstico leve de VRAM via nvidia-smi.
    Nunca interrompe o pipeline se nvidia-smi falhar.
    """
    if label:
        print(f"\nVRAM — {label}")

    try:
        result = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=memory.total,memory.used,memory.free",
                "--format=csv,noheader,nounits",
            ],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            check=False,
        )

        if (
            result.returncode == 0
            and result.stdout.strip()
        ):
            total, used, free = [
                int(x.strip())
                for x in result.stdout.strip().splitlines()[0].split(",")
            ]

            print(
                f"  total={total} MiB | "
                f"usado={used} MiB | "
                f"livre={free} MiB"
            )

            return {
                "total_mib": total,
                "used_mib": used,
                "free_mib": free,
            }

    except Exception:
        pass

    return None


def load_model_exact(runtime):
    """
    Carrega EXATAMENTE o runtime solicitado.
    Não troca silenciosamente para CPU.
    """
    print(
        f"Tentando carregar: "
        f"model={runtime['model']} "
        f"device={runtime['device']} "
        f"compute={runtime['compute_type']}"
    )

    model = WhisperModel(
        runtime["model"],
        device=runtime["device"],
        compute_type=runtime["compute_type"],
        cpu_threads=max(
            1,
            min(
                os.cpu_count() or 2,
                8
            )
        ),
        num_workers=1,
    )

    active = {
        "device": runtime["device"],
        "compute_type": runtime["compute_type"],
        "model": runtime["model"],
    }

    print("OK.")

    return model, active


def load_model_with_fallback(runtime):
    """
    Fallback usado no carregamento inicial.

    Ordem para GPU:
        compute solicitado
        int8_float16
        int8
        float16
        CPU int8

    Importante:
    falha DURANTE transcrição é tratada separadamente
    pela Célula 10, que tenta recuperar a GPU antes da CPU.
    """
    attempts = []

    attempts.append({
        "device": runtime["device"],
        "compute_type": runtime["compute_type"],
        "model": runtime["model"],
    })

    if runtime["device"] == "cuda":

        for compute in (
            "int8_float16",
            "int8",
            "float16",
        ):
            if (
                compute
                != runtime["compute_type"]
            ):
                attempts.append({
                    "device": "cuda",
                    "compute_type": compute,
                    "model": runtime["model"],
                })

        attempts.append({
            "device": "cpu",
            "compute_type": "int8",
            "model": runtime["model"],
        })

    else:

        attempts.append({
            "device": "cpu",
            "compute_type": "float32",
            "model": runtime["model"],
        })

    last_error = None
    seen = set()

    for attempt in attempts:

        key = (
            attempt["device"],
            attempt["compute_type"],
            attempt["model"],
        )

        if key in seen:
            continue

        seen.add(key)

        try:

            release_gpu_memory()

            return load_model_exact(
                attempt
            )

        except Exception as e:

            last_error = e

            print(
                f"Falhou: "
                f"{type(e).__name__}: "
                f"{e}"
            )

    raise RuntimeError(
        "Não foi possível carregar o modelo. "
        f"Último erro: {last_error}"
    )


MODEL, ACTIVE_RUNTIME = (
    load_model_with_fallback(
        RUNTIME
    )
)

print(
    "Runtime ativo:",
    ACTIVE_RUNTIME
)

gpu_memory_snapshot(
    "após carregar modelo inicial"
)


Tentando carregar: model=large-v3-turbo device=cuda compute=float16


OK.
Runtime ativo: {'device': 'cuda', 'compute_type': 'float16', 'model': 'large-v3-turbo'}

VRAM — após carregar modelo inicial
  total=15360 MiB | usado=1673 MiB | livre=13240 MiB


{'total_mib': 15360, 'used_mib': 1673, 'free_mib': 13240}

In [9]:
# ============================================================
# CÉLULA 7B — Guard para alinhamento vazio do faster-whisper
# ============================================================
#
# faster-whisper 1.2.1 pode receber um alinhamento vazio de
# CTranslate2 em uma janela degenerada. Sem este guard,
# find_alignment() quebra em time_indices[jumps].
#
# O comportamento adotado é conservador:
# o segmento continua existindo, mas sem word timestamps.
# A Célula 9 preserva o texto usando timestamps do segmento.
# ============================================================

import numpy as np
from faster_whisper import WhisperModel


def _raru_safe_find_alignment(
    self,
    tokenizer,
    text_tokens,
    encoder_output,
    num_frames,
    median_filter_width=7,
):
    if len(text_tokens) == 0:
        return []

    results = self.model.align(
        encoder_output,
        tokenizer.sot_sequence,
        text_tokens,
        num_frames,
        median_filter_width=median_filter_width,
    )

    return_list = []

    for result, text_token in zip(results, text_tokens):
        text_token_probs = result.text_token_probs
        alignments = result.alignments

        # Fix upstream ainda não presente no release 1.2.1:
        # alinhamento vazio -> sem palavras, sem crash.
        if not alignments:
            return_list.append([])
            continue

        text_indices = np.array([pair[0] for pair in alignments])
        time_indices = np.array([pair[1] for pair in alignments])

        if text_indices.size == 0 or time_indices.size == 0:
            return_list.append([])
            continue

        words, word_tokens = tokenizer.split_to_word_tokens(
            text_token + [tokenizer.eot]
        )

        if len(word_tokens) <= 1:
            return_list.append([])
            continue

        word_boundaries = np.pad(
            np.cumsum([len(t) for t in word_tokens[:-1]]),
            (1, 0),
        )

        if len(word_boundaries) <= 1:
            return_list.append([])
            continue

        jumps = np.pad(
            np.diff(text_indices),
            (1, 0),
            constant_values=1,
        ).astype(bool)

        # Defesa adicional para shapes inconsistentes.
        if len(jumps) != len(time_indices):
            return_list.append([])
            continue

        jump_times = time_indices[jumps] / self.tokens_per_second

        if (
            jump_times.size == 0
            or np.max(word_boundaries) >= jump_times.size
        ):
            return_list.append([])
            continue

        start_times = jump_times[word_boundaries[:-1]]
        end_times = jump_times[word_boundaries[1:]]

        word_probabilities = [
            np.mean(text_token_probs[i:j])
            for i, j in zip(
                word_boundaries[:-1],
                word_boundaries[1:],
            )
        ]

        return_list.append([
            {
                "word": word,
                "tokens": tokens,
                "start": start,
                "end": end,
                "probability": probability,
            }
            for word, tokens, start, end, probability in zip(
                words,
                word_tokens,
                start_times,
                end_times,
                word_probabilities,
            )
        ])

    return return_list


WhisperModel.find_alignment = _raru_safe_find_alignment

print("Guard de alinhamento vazio aplicado ao WhisperModel.")


Guard de alinhamento vazio aplicado ao WhisperModel.


In [10]:
# ============================================================
# CÉLULA 8 — Utilidades de checkpoint e serialização
# ============================================================

from datetime import datetime, timezone

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def safe_stem(path):
    return re.sub(
        r'[^A-Za-z0-9._-]+',
        '_',
        Path(path).stem
    )

def job_paths(source_path, test_mode=False):
    """
    RAW e TEST usam caminhos fisicamente diferentes,
    inclusive para checkpoints.
    """
    stem = safe_stem(source_path)

    suffix = ".TEST" if test_mode else ""
    checkpoint_suffix = "__TEST" if test_mode else ""

    return {
        "final_json": DIRS["transcriptions"] / f"{stem}{suffix}.json",
        "txt": DIRS["transcriptions"] / f"{stem}{suffix}.txt",
        "srt": DIRS["transcriptions"] / f"{stem}{suffix}.srt",

        "progress": (
            DIRS["checkpoints"]
            / f"{stem}{checkpoint_suffix}.progress.json"
        ),

        "chunks_dir": (
            DIRS["checkpoints"]
            / f"{stem}{checkpoint_suffix}"
        ),

        "log": (
            DIRS["logs"]
            / f"{stem}{suffix}.log"
        ),
    }

def atomic_json_dump(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    with open(
        tmp,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            obj,
            f,
            ensure_ascii=False,
            indent=2
        )
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp, path)

def load_json(path, default=None):
    path = Path(path)

    if not path.exists():
        return default

    with open(
        path,
        encoding="utf-8"
    ) as f:
        return json.load(f)

def sec_to_clock(s, srt=False):
    s = max(
        0.0,
        float(s)
    )

    h = int(s // 3600)
    m = int((s % 3600) // 60)
    sec = s % 60

    if srt:
        return (
            f"{h:02d}:"
            f"{m:02d}:"
            f"{sec:06.3f}"
        ).replace(".", ",")

    return (
        f"{h:02d}:"
        f"{m:02d}:"
        f"{sec:06.3f}"
    )


In [11]:
# ============================================================
# CÉLULA 9 — Transcrição de um bloco com overlap
#             + fallback de alinhamento por palavra
# ============================================================

def segment_to_dict(
    seg,
    offset_s,
    core_start,
    core_end
):
    words_out = []

    for w in (seg.words or []):
        abs_start = offset_s + float(w.start)
        abs_end = offset_s + float(w.end)
        midpoint = (abs_start + abs_end) / 2.0

        if core_start <= midpoint < core_end:
            words_out.append({
                "start": round(abs_start, 3),
                "end": round(abs_end, 3),
                "word": w.word,
                "probability": (
                    round(float(w.probability), 5)
                    if w.probability is not None
                    else None
                ),
            })

    # Caminho normal: alinhamento por palavra disponível.
    if WORD_TIMESTAMPS and words_out:
        text = "".join(
            word["word"]
            for word in words_out
        ).strip()

        start = words_out[0]["start"]
        end = words_out[-1]["end"]
        word_alignment_missing = False

    # Fallback: word_timestamps estava ligado, mas o alinhador
    # não conseguiu devolver palavras para este segmento.
    else:
        abs_start = offset_s + float(seg.start)
        abs_end = offset_s + float(seg.end)
        midpoint = (abs_start + abs_end) / 2.0

        if not (
            core_start <= midpoint < core_end
        ):
            return None

        text = (seg.text or "").strip()

        if not text:
            return None

        start = max(
            abs_start,
            core_start
        )

        end = min(
            abs_end,
            core_end
        )

        if end <= start:
            return None

        word_alignment_missing = bool(
            WORD_TIMESTAMPS
        )

        words_out = []

    return {
        "start": round(start, 3),
        "end": round(end, 3),
        "text": text,

        "words": (
            words_out
            if WORD_TIMESTAMPS
            else None
        ),

        "word_alignment_missing": (
            word_alignment_missing
        ),

        "avg_logprob": (
            round(float(seg.avg_logprob), 6)
            if seg.avg_logprob is not None
            else None
        ),

        "compression_ratio": (
            round(float(seg.compression_ratio), 6)
            if seg.compression_ratio is not None
            else None
        ),

        "no_speech_prob": (
            round(float(seg.no_speech_prob), 6)
            if seg.no_speech_prob is not None
            else None
        ),

        "temperature": (
            float(seg.temperature)
            if seg.temperature is not None
            else None
        ),
    }


def transcribe_chunk(
    chunk_path,
    chunk_file_start,
    core_start,
    core_end
):
    segments_iter, info = MODEL.transcribe(
        str(chunk_path),

        language=LANGUAGE,
        multilingual=MULTILINGUAL,

        task=TASK,

        beam_size=BEAM_SIZE,
        temperature=TEMPERATURE,

        condition_on_previous_text=(
            CONDITION_ON_PREVIOUS_TEXT
        ),

        word_timestamps=WORD_TIMESTAMPS,

        chunk_length=WHISPER_CHUNK_LENGTH,

        vad_filter=VAD_FILTER,

        vad_parameters=(
            VAD_PARAMETERS
            if VAD_FILTER
            else None
        ),

        no_speech_threshold=NO_SPEECH_THRESHOLD,

        hallucination_silence_threshold=(
            HALLUCINATION_SILENCE_THRESHOLD
        ),

        suppress_tokens=[-1],

        log_progress=False,
    )

    out = []
    missing_alignment_count = 0

    for seg in segments_iter:
        item = segment_to_dict(
            seg,
            chunk_file_start,
            core_start,
            core_end
        )

        if item and item["text"]:
            if item.get(
                "word_alignment_missing",
                False
            ):
                missing_alignment_count += 1

            out.append(item)

    if missing_alignment_count > 0:
        print(
            "  AVISO: "
            f"{missing_alignment_count} segmento(s) "
            "sem alinhamento por palavra; "
            "timestamps de segmento preservados."
        )

    return out, {
        "language": getattr(
            info,
            "language",
            LANGUAGE
        ),

        "language_probability": float(
            getattr(
                info,
                "language_probability",
                0
            ) or 0
        ),

        "duration": float(
            getattr(
                info,
                "duration",
                0
            ) or 0
        ),

        "duration_after_vad": float(
            getattr(
                info,
                "duration_after_vad",
                0
            ) or 0
        ),

        "missing_word_alignment_segments": (
            missing_alignment_count
        ),
    }


In [12]:
# ============================================================
# CÉLULA 10 — Processar uma live com retomada automática
#              + recuperação automática de GPU
#              + Passada 2 opcional
# ============================================================

def process_live(
    source_path,
    force=False,
    test_mode=TEST_MODE,
    repair=None,
):
    global MODEL, ACTIVE_RUNTIME

    if repair is None:
        repair = bool(
            ENABLE_REPAIR_PASS
        )

    source_path = Path(
        source_path
    )

    paths = job_paths(
        source_path,
        test_mode=test_mode
    )

    work_dir = Path(
        "/content/raru_revive"
    )

    work_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    repaired_json = (
        paths["final_json"]
        .with_name(
            paths["final_json"].stem
            + ".REPAIRED.json"
        )
    )

    repaired_txt = (
        paths["final_json"]
        .with_name(
            paths["final_json"].stem
            + ".REPAIRED.txt"
        )
    )

    repaired_srt = (
        paths["final_json"]
        .with_name(
            paths["final_json"].stem
            + ".REPAIRED.srt"
        )
    )

    raw_completed_status = (
        "test_completed"
        if test_mode
        else "completed"
    )

    repaired_completed_status = (
        "test_repaired"
        if test_mode
        else "completed_repaired"
    )

    fingerprint = source_fingerprint(
        source_path
    )

    def validate_output_source(
        data,
        label
    ):
        if not data:
            return False

        saved_fingerprint = (
            data
            .get("source", {})
            .get(
                "fingerprint_sha256_partial"
            )
        )

        if saved_fingerprint is None:
            return True

        if (
            saved_fingerprint
            != fingerprint
        ):
            raise RuntimeError(
                f"{label} pertence a uma versão "
                f"diferente do arquivo fonte.\n"
                f"Use force=True para retranscrever."
            )

        return True


    def persist_active_runtime():
        """
        Salva device/compute atuais no checkpoint
        após qualquer troca de runtime.
        """
        progress[
            "model"
        ] = ACTIVE_RUNTIME[
            "model"
        ]

        progress[
            "device"
        ] = ACTIVE_RUNTIME[
            "device"
        ]

        progress[
            "compute_type"
        ] = ACTIVE_RUNTIME[
            "compute_type"
        ]

        progress[
            "updated_at"
        ] = utc_now()

        atomic_json_dump(
            progress,
            paths["progress"]
        )


    def clear_models_before_gpu_retry():
        """
        Remove referências que podem estar prendendo VRAM.

        Isso é especialmente importante ao reexecutar o
        notebook na MESMA sessão de Colab depois de uma
        Passada 2, pois um large-v3 antigo pode ainda estar
        residente na GPU.
        """
        global MODEL

        MODEL = None

        if (
            "REPAIR_MODEL"
            in globals()
        ):
            globals()[
                "REPAIR_MODEL"
            ] = None

        release_gpu_memory()


    def switch_runtime_exact(
        device,
        compute_type,
        model_name
    ):
        """
        Troca para um runtime específico.
        """
        global MODEL, ACTIVE_RUNTIME

        clear_models_before_gpu_retry()

        gpu_memory_snapshot(
            "antes de carregar novo runtime"
        )

        MODEL, ACTIVE_RUNTIME = (
            load_model_exact({
                "device": device,
                "compute_type": compute_type,
                "model": model_name,
            })
        )

        gpu_memory_snapshot(
            "após carregar novo runtime"
        )

        persist_active_runtime()


    # ========================================================
    # FORCE
    # ========================================================

    if force:

        if paths["progress"].exists():
            paths["progress"].unlink()

        if paths["chunks_dir"].exists():
            shutil.rmtree(
                paths["chunks_dir"]
            )

        for output in (
            paths["final_json"],
            paths["txt"],
            paths["srt"],
            repaired_json,
            repaired_txt,
            repaired_srt,
        ):
            if output.exists():
                output.unlink()

        print(
            "FORCE: checkpoints e outputs anteriores "
            "deste job foram removidos."
        )


    # ========================================================
    # Reaproveitar RAW / REPAIRED existente
    # ========================================================

    if (
        not force
        and paths["final_json"].exists()
    ):

        existing_raw = (
            load_json(
                paths["final_json"],
                {}
            )
            or {}
        )

        if (
            existing_raw.get("status")
            == raw_completed_status
        ):

            validate_output_source(
                existing_raw,
                "Passada 1"
            )

            if not repair:

                print()
                print("=" * 80)
                print(
                    "PASSADA 1 JÁ CONCLUÍDA"
                )
                print("=" * 80)

                return existing_raw

            if repaired_json.exists():

                existing_repaired = (
                    load_json(
                        repaired_json,
                        {}
                    )
                    or {}
                )

                if (
                    existing_repaired.get("status")
                    == repaired_completed_status
                ):

                    validate_output_source(
                        existing_repaired,
                        "Passada 2"
                    )

                    print()
                    print("=" * 80)
                    print(
                        "PASSADA 2 JÁ CONCLUÍDA"
                    )
                    print("=" * 80)

                    return existing_repaired

            print()
            print("=" * 80)
            print(
                "PASSADA 1 JÁ CONCLUÍDA"
            )
            print("=" * 80)

            print(
                f"RAW encontrado: "
                f"{paths['final_json']}"
            )

            print(
                "Passada 1 não será executada novamente."
            )

            print()
            print(
                "Iniciando diretamente a Passada 2..."
            )

            return run_repair_pass(
                source_path=source_path,
                transcription=existing_raw,
                test_mode=test_mode,
            )


    # ========================================================
    # Passada 1 / retomada
    # ========================================================

    duration = media_duration(
        source_path
    )

    effective_duration = (
        min(
            duration,
            TEST_SECONDS
        )
        if test_mode
        else duration
    )

    progress = (
        load_json(
            paths["progress"],
            {}
        )
        or {}
    )

    if progress:

        progress_fingerprint = (
            progress.get(
                "source_fingerprint"
            )
        )

        if (
            progress_fingerprint
            not in (
                None,
                fingerprint
            )
        ):

            raise RuntimeError(
                "O arquivo fonte mudou desde "
                "o checkpoint.\n"
                "Use force=True para começar novamente."
            )


    # --------------------------------------------------------
    # Se uma Passada 2 de execução anterior estiver viva
    # no mesmo kernel, libera antes da Passada 1.
    # --------------------------------------------------------

    if (
        "REPAIR_MODEL"
        in globals()
        and
        globals().get(
            "REPAIR_MODEL"
        )
        is not None
    ):

        print(
            "Liberando modelo antigo da Passada 2 "
            "antes da Passada 1..."
        )

        globals()[
            "REPAIR_MODEL"
        ] = None

        release_gpu_memory()


    desired = choose_runtime_config(
        HW,
        QUALITY_MODE,
        progress if progress else None
    )

    active_runtime = (
        ACTIVE_RUNTIME
        if isinstance(
            ACTIVE_RUNTIME,
            dict
        )
        else {}
    )

    runtime_changed = any(
        active_runtime.get(key)
        != desired.get(key)
        for key in (
            "model",
            "device",
            "compute_type",
        )
    )

    if (
        MODEL is None
        or runtime_changed
    ):

        print()
        print(
            "Preparando runtime da Passada 1..."
        )

        # Remove referência antiga antes de carregar outra.
        MODEL = None
        release_gpu_memory()

        MODEL, ACTIVE_RUNTIME = (
            load_model_with_fallback(
                desired
            )
        )


    media_local, prep_mode = (
        prepare_source(
            source_path
        )
    )

    chunk_dir = (
        paths["chunks_dir"]
    )

    chunk_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    n_chunks = math.ceil(
        effective_duration
        / CHUNK_SECONDS
    )

    completed_chunks = set(
        progress.get(
            "completed_chunks",
            []
        )
    )

    progress.update({
        "schema_version": "1.0",
        "source": source_path.name,
        "source_path": str(source_path),
        "source_fingerprint": fingerprint,
        "status": "processing",
        "duration_seconds": duration,
        "effective_duration_seconds": effective_duration,
        "test_mode": bool(test_mode),
        "processed_until": progress.get(
            "processed_until",
            0.0
        ),
        "completed_chunks": sorted(
            completed_chunks
        ),
        "chunk_seconds": CHUNK_SECONDS,
        "overlap_seconds": OVERLAP_SECONDS,
        "model": ACTIVE_RUNTIME["model"],
        "device": ACTIVE_RUNTIME["device"],
        "compute_type": ACTIVE_RUNTIME["compute_type"],
        "language": LANGUAGE,
        "repair_requested": bool(repair),
        "updated_at": utc_now(),
    })

    atomic_json_dump(
        progress,
        paths["progress"]
    )


    print()
    print("=" * 80)
    print(
        "PASSADA 1 — TRANSCRIÇÃO"
    )
    print("=" * 80)

    print(
        f"Arquivo: "
        f"{source_path.name}"
    )

    print(
        f"Modelo: "
        f"{ACTIVE_RUNTIME['model']}"
    )

    print(
        f"Device: "
        f"{ACTIVE_RUNTIME['device']}"
    )

    print(
        f"Compute: "
        f"{ACTIVE_RUNTIME['compute_type']}"
    )

    print(
        f"Modo teste: "
        f"{test_mode}"
    )

    print(
        f"Repair após Passada 1: "
        f"{repair}"
    )

    gpu_memory_snapshot(
        "antes da Passada 1"
    )

    print()

    t_job = (
        time.perf_counter()
    )


    # ========================================================
    # Chunks externos
    # ========================================================

    for idx in range(
        n_chunks
    ):

        if idx in completed_chunks:

            print(
                f"[{idx + 1}/{n_chunks}] "
                f"já concluído; pulando."
            )

            continue


        core_start = (
            idx
            * CHUNK_SECONDS
        )

        core_end = min(
            (idx + 1)
            * CHUNK_SECONDS,
            effective_duration
        )

        read_start = max(
            0.0,
            core_start
            - OVERLAP_SECONDS
        )

        read_end = min(
            effective_duration,
            core_end
            + OVERLAP_SECONDS
        )

        read_duration = (
            read_end
            - read_start
        )

        chunk_audio = (
            work_dir
            /
            (
                f"{safe_stem(source_path)}"
                f".chunk_{idx:05d}.flac"
            )
        )


        try:

            print(
                f"[{idx + 1}/{n_chunks}] "
                f"{sec_to_clock(core_start)} "
                f"-> "
                f"{sec_to_clock(core_end)}"
            )

            extract_chunk(
                media_local,
                chunk_audio,
                read_start,
                read_duration
            )


            # =================================================
            # Retry automático do MESMO chunk
            # =================================================

            attempted_gpu_computes = set()

            while True:

                try:

                    if (
                        ACTIVE_RUNTIME[
                            "device"
                        ]
                        == "cuda"
                    ):
                        attempted_gpu_computes.add(
                            ACTIVE_RUNTIME[
                                "compute_type"
                            ]
                        )

                    t0 = (
                        time.perf_counter()
                    )

                    chunk_segments, info = (
                        transcribe_chunk(
                            chunk_audio,
                            chunk_file_start=read_start,
                            core_start=core_start,
                            core_end=core_end,
                        )
                    )

                    elapsed = (
                        time.perf_counter()
                        - t0
                    )

                    break


                except RuntimeError as e:

                    msg = str(e).lower()

                    gpu_error = (
                        "out of memory" in msg
                        or
                        "cuda" in msg
                        or
                        "cublas" in msg
                        or
                        "cudnn" in msg
                        or
                        "cufft" in msg
                        or
                        "allocator" in msg
                    )

                    if (
                        not gpu_error
                        or
                        ACTIVE_RUNTIME[
                            "device"
                        ]
                        != "cuda"
                    ):
                        raise

                    print()
                    print(
                        "=" * 80
                    )
                    print(
                        "ERRO DURANTE TRANSCRIÇÃO NA GPU"
                    )
                    print(
                        "=" * 80
                    )

                    print(
                        f"{type(e).__name__}: "
                        f"{e}"
                    )

                    gpu_memory_snapshot(
                        "no momento da falha"
                    )

                    model_name = (
                        ACTIVE_RUNTIME[
                            "model"
                        ]
                    )

                    # -----------------------------------------
                    # Tenta manter o trabalho na GPU.
                    # -----------------------------------------

                    gpu_candidates = [
                        compute
                        for compute in (
                            "int8_float16",
                            "int8",
                        )
                        if (
                            compute
                            not in attempted_gpu_computes
                            and
                            compute
                            in set(
                                HW[
                                    "ct2_gpu_compute_types"
                                ]
                            )
                        )
                    ]

                    switched = False

                    for compute in (
                        gpu_candidates
                    ):

                        print()
                        print(
                            "Tentando recuperar a T4 "
                            f"com compute={compute}..."
                        )

                        try:

                            switch_runtime_exact(
                                device="cuda",
                                compute_type=compute,
                                model_name=model_name,
                            )

                            switched = True

                            print(
                                "GPU recuperada. "
                                "Repetindo o MESMO bloco..."
                            )

                            break

                        except Exception as load_error:

                            print(
                                f"Falha ao recarregar GPU "
                                f"com {compute}: "
                                f"{type(load_error).__name__}: "
                                f"{load_error}"
                            )

                    if switched:
                        continue


                    # -----------------------------------------
                    # Último recurso: CPU.
                    # -----------------------------------------

                    print()
                    print(
                        "GPU não conseguiu completar "
                        "o bloco após os retries."
                    )

                    print(
                        "Usando CPU int8 como último recurso "
                        "e repetindo o MESMO bloco automaticamente."
                    )

                    switch_runtime_exact(
                        device="cpu",
                        compute_type="int8",
                        model_name=model_name,
                    )

                    # volta ao while e tenta o MESMO chunk em CPU
                    continue


            # =================================================
            # Chunk concluído
            # =================================================

            chunk_json = {
                "chunk_index": idx,
                "core_start": core_start,
                "core_end": core_end,
                "read_start": read_start,
                "read_end": read_end,
                "elapsed_seconds": elapsed,
                "rtf": (
                    elapsed
                    /
                    max(
                        0.001,
                        core_end
                        - core_start
                    )
                ),
                "info": info,
                "runtime": {
                    "model": (
                        ACTIVE_RUNTIME[
                            "model"
                        ]
                    ),
                    "device": (
                        ACTIVE_RUNTIME[
                            "device"
                        ]
                    ),
                    "compute_type": (
                        ACTIVE_RUNTIME[
                            "compute_type"
                        ]
                    ),
                },
                "segments": chunk_segments,
                "completed_at": utc_now(),
            }

            atomic_json_dump(
                chunk_json,
                (
                    chunk_dir
                    /
                    f"chunk_{idx:05d}.json"
                )
            )

            completed_chunks.add(
                idx
            )

            progress[
                "completed_chunks"
            ] = sorted(
                completed_chunks
            )

            progress[
                "processed_until"
            ] = core_end

            persist_active_runtime()

            print(
                f"  salvo | "
                f"{len(chunk_segments)} segmentos | "
                f"RTF={chunk_json['rtf']:.3f} | "
                f"{ACTIVE_RUNTIME['device']}/"
                f"{ACTIVE_RUNTIME['compute_type']}"
            )


        finally:

            try:
                chunk_audio.unlink(
                    missing_ok=True
                )

            except Exception:
                pass


    # ========================================================
    # Finalizar Passada 1
    # ========================================================

    total_elapsed = (
        time.perf_counter()
        - t_job
    )

    result = finalize_live(
        source_path,
        total_elapsed,
        prep_mode,
        test_mode
    )

    if result is None:
        return None


    # ========================================================
    # Passada 2
    # ========================================================

    if repair:

        print()
        print("=" * 80)
        print(
            "INICIANDO PASSADA 2"
        )
        print("=" * 80)

        result = run_repair_pass(
            source_path=source_path,
            transcription=result,
            test_mode=test_mode,
        )

    return result


In [13]:
# ============================================================
# CÉLULA 11 — Consolidar JSON oficial + TXT + SRT
# ============================================================

def finalize_live(
    source_path,
    session_elapsed,
    prep_mode,
    test_mode=False
):
    source_path = Path(
        source_path
    )

    paths = job_paths(
        source_path,
        test_mode=test_mode
    )

    progress = load_json(
        paths["progress"],
        {}
    )

    chunk_files = sorted(
        paths["chunks_dir"]
        .glob(
            "chunk_*.json"
        )
    )

    all_segments = []
    chunk_metrics = []

    for cf in chunk_files:
        cj = load_json(
            cf,
            {}
        )

        all_segments.extend(
            cj.get(
                "segments",
                []
            )
        )

        chunk_metrics.append({
            "chunk_index": (
                cj.get(
                    "chunk_index"
                )
            ),

            "elapsed_seconds": (
                cj.get(
                    "elapsed_seconds"
                )
            ),

            "rtf": (
                cj.get(
                    "rtf"
                )
            ),
        })

    all_segments.sort(
        key=lambda x: (
            x["start"],
            x["end"]
        )
    )

    for i, seg in enumerate(
        all_segments,
        start=1
    ):
        seg["id"] = i

    duration = float(
        progress[
            "duration_seconds"
        ]
    )

    effective_duration = float(
        progress[
            "effective_duration_seconds"
        ]
    )

    total_chunk_elapsed = sum(
        x["elapsed_seconds"] or 0
        for x in chunk_metrics
    )

    result = {
        "schema": "raru-revival-transcription",
        "schema_version": "1.0",

        "status": (
            "test_completed"
            if test_mode
            else "completed"
        ),

        "source": {
            "file_name": (
                source_path.name
            ),

            "relative_drive_path": (
                str(source_path)
                .split(
                    "/MyDrive/"
                )[-1]
                if "/MyDrive/" in str(source_path)
                else str(source_path)
            ),

            "fingerprint_sha256_partial": (
                progress[
                    "source_fingerprint"
                ]
            ),

            "duration_seconds": (
                duration
            ),

            "container": (
                ffprobe_json(
                    source_path
                )["format"]
                .get(
                    "format_name"
                )
            ),
        },

        "transcription": {
            "multilingual": MULTILINGUAL,
            "whisper_chunk_length_seconds": WHISPER_CHUNK_LENGTH,
            "no_speech_threshold": NO_SPEECH_THRESHOLD,
            "language_configured": LANGUAGE,
            "task": TASK,
            "model": progress["model"],
            "device": progress["device"],
            "compute_type": progress["compute_type"],
            "word_timestamps": WORD_TIMESTAMPS,
            "beam_size": BEAM_SIZE,
            "temperature": TEMPERATURE,
            "condition_on_previous_text": CONDITION_ON_PREVIOUS_TEXT,
            "vad_filter": VAD_FILTER,
            "vad_parameters": VAD_PARAMETERS,
            "hallucination_silence_threshold": HALLUCINATION_SILENCE_THRESHOLD,
        },

        "checkpointing": {
            "chunk_seconds": CHUNK_SECONDS,
            "overlap_seconds": OVERLAP_SECONDS,
            "chunks_completed": len(
                chunk_files
            ),
            "processed_until": (
                progress.get(
                    "processed_until"
                )
            ),
        },

        "performance": {
            "audio_processed_seconds": (
                effective_duration
            ),

            "transcription_elapsed_seconds_sum": round(
                total_chunk_elapsed,
                3
            ),

            "current_session_elapsed_seconds": round(
                session_elapsed,
                3
            ),

            "rtf": round(
                total_chunk_elapsed
                /
                max(
                    effective_duration,
                    0.001
                ),
                4
            ),

            "hardware": HW,
        },

        "segments": all_segments,
        "created_at": utc_now(),
    }

    atomic_json_dump(
        result,
        paths["final_json"]
    )

    with open(
        paths["txt"],
        "w",
        encoding="utf-8"
    ) as f:
        for seg in all_segments:
            f.write(
                f"[{sec_to_clock(seg['start'])} "
                f"--> {sec_to_clock(seg['end'])}]\n"
            )
            f.write(
                seg["text"].strip()
                + "\n"
            )

    with open(
        paths["srt"],
        "w",
        encoding="utf-8"
    ) as f:
        for i, seg in enumerate(
            all_segments,
            start=1
        ):
            f.write(
                f"{i}\n"
            )

            f.write(
                f"{sec_to_clock(seg['start'], True)} "
                f"--> "
                f"{sec_to_clock(seg['end'], True)}\n"
            )

            f.write(
                seg["text"].strip()
                + "\n\n"
            )

    progress["status"] = (
        "test_completed"
        if test_mode
        else "completed"
    )

    progress[
        "updated_at"
    ] = utc_now()

    atomic_json_dump(
        progress,
        paths["progress"]
    )

    print()
    print(
        "CONCLUÍDO"
    )

    print(
        "JSON:",
        paths["final_json"]
    )

    print(
        "TXT :",
        paths["txt"]
    )

    print(
        "SRT :",
        paths["srt"]
    )

    print(
        f"RTF : "
        f"{result['performance']['rtf']:.3f}"
    )

    return result


In [14]:
# ============================================================
# CÉLULA 11A — Quality Analyzer v3
# ============================================================

from collections import Counter


def _normalize_repair_word(word):
    word = (
        word
        or ""
    ).lower().strip()

    return re.sub(
        r"[^\wÀ-ÿ]+",
        "",
        word,
        flags=re.UNICODE,
    )


def _repetition_ratio(
    words,
    minimum_words=None
):
    if minimum_words is None:
        minimum_words = (
            REPAIR_REPETITION_MIN_WORDS
        )

    normalized = [
        _normalize_repair_word(
            word.get(
                "word",
                ""
            )
        )
        for word in (
            words
            or []
        )
    ]

    normalized = [
        word
        for word in normalized
        if word
    ]

    if (
        len(normalized)
        < minimum_words
    ):
        return 0.0

    counter = Counter(
        normalized
    )

    most_common_count = (
        counter
        .most_common(1)[0][1]
    )

    return (
        most_common_count
        / len(normalized)
    )


def _contains_cyrillic(text):
    return bool(
        re.search(
            r"[\u0400-\u04FF]",
            text or ""
        )
    )


def _contains_strong_unexpected_latin(
    text
):
    """
    Caracteres muito improváveis em PT/EN.
    Não inclui acentos normais do português.
    """
    return bool(
        re.search(
            (
                r"["
                r"őŐűŰ"
                r"łŁżŻźŹśŚćĆńŃ"
                r"řŘěĚšŠžŽďĎťŤůŮ"
                r"ğĞşŞıİ"
                r"]"
            ),
            text or ""
        )
    )


def _word_probabilities(
    segment
):
    return [
        float(
            word[
                "probability"
            ]
        )
        for word in (
            segment.get(
                "words"
            )
            or []
        )
        if (
            word.get(
                "probability"
            )
            is not None
        )
    ]


def _mean_word_probability(
    segment
):
    probabilities = (
        _word_probabilities(
            segment
        )
    )

    if not probabilities:
        return None

    return (
        sum(probabilities)
        / len(probabilities)
    )


def _min_word_probability(
    segment
):
    probabilities = (
        _word_probabilities(
            segment
        )
    )

    if not probabilities:
        return None

    return min(
        probabilities
    )


def _zero_duration_ratio(
    words
):
    words = (
        words
        or []
    )

    if not words:
        return 0.0

    zero_count = 0

    for word in words:
        start = float(
            word.get(
                "start",
                0
            )
        )

        end = float(
            word.get(
                "end",
                start
            )
        )

        if (
            abs(
                end - start
            )
            <= 0.001
        ):
            zero_count += 1

    return (
        zero_count
        / len(words)
    )


def _distance_to_whisper_boundary(
    timestamp
):
    chunk = float(
        WHISPER_CHUNK_LENGTH
    )

    nearest = (
        round(
            timestamp
            / chunk
        )
        * chunk
    )

    return abs(
        timestamp
        - nearest
    )


def analyze_segment_quality(
    segment
):
    reasons = []
    severity = "ok"

    words = (
        segment.get(
            "words"
        )
        or []
    )

    text = (
        segment.get(
            "text"
        )
        or ""
    ).strip()

    avg_logprob = float(
        segment.get(
            "avg_logprob",
            0
        )
        or 0
    )

    compression_ratio = float(
        segment.get(
            "compression_ratio",
            0
        )
        or 0
    )

    mean_probability = (
        _mean_word_probability(
            segment
        )
    )

    min_probability = (
        _min_word_probability(
            segment
        )
    )

    repetition_ratio = (
        _repetition_ratio(
            words
        )
    )

    zero_duration_ratio = (
        _zero_duration_ratio(
            words
        )
    )

    if (
        compression_ratio
        >= REPAIR_COMPRESSION_RATIO
    ):
        reasons.append(
            "high_compression_ratio"
        )
        severity = "critical"

    if (
        repetition_ratio
        >= REPAIR_REPETITION_RATIO
    ):
        reasons.append(
            "repetition_loop"
        )
        severity = "critical"

    if (
        avg_logprob
        <= REPAIR_AVG_LOGPROB
    ):
        reasons.append(
            "low_avg_logprob"
        )
        if (
            severity
            != "critical"
        ):
            severity = "suspect"

    if (
        len(words)
        <= REPAIR_SHORT_SEGMENT_MAX_WORDS
        and
        mean_probability is not None
        and
        mean_probability
        < REPAIR_SHORT_SEGMENT_MEAN_PROB
    ):
        reasons.append(
            "low_confidence_short_segment"
        )

        if (
            severity
            != "critical"
        ):
            severity = "suspect"

    if words:
        last_word = (
            words[-1]
        )

        last_end = float(
            last_word.get(
                "end",
                segment.get(
                    "end",
                    0
                )
            )
        )

        last_probability = float(
            last_word.get(
                "probability",
                1
            )
        )

        boundary_distance = (
            _distance_to_whisper_boundary(
                last_end
            )
        )

        if (
            boundary_distance
            <= REPAIR_BOUNDARY_MARGIN
            and
            last_probability
            < REPAIR_BOUNDARY_WORD_PROB
        ):
            reasons.append(
                "low_confidence_near_chunk_boundary"
            )

            if (
                severity
                != "critical"
            ):
                severity = "suspect"

    if text.endswith(
        "..."
    ):
        boundary_distance = (
            _distance_to_whisper_boundary(
                float(
                    segment.get(
                        "end",
                        0
                    )
                )
            )
        )

        if (
            boundary_distance
            <= REPAIR_BOUNDARY_MARGIN
        ):
            reasons.append(
                "unfinished_phrase_at_boundary"
            )

            if (
                severity
                != "critical"
            ):
                severity = "suspect"

    if (
        _contains_cyrillic(
            text
        )
    ):
        reasons.append(
            "unexpected_cyrillic_script"
        )

        if (
            severity
            != "critical"
        ):
            severity = "suspect"

    if (
        _contains_strong_unexpected_latin(
            text
        )
    ):
        reasons.append(
            "unexpected_latin_script"
        )

        if (
            severity
            != "critical"
        ):
            severity = "suspect"

    if (
        len(words) >= 3
        and
        zero_duration_ratio
        >= REPAIR_ZERO_DURATION_RATIO
    ):
        reasons.append(
            "excessive_zero_duration_words"
        )

        if (
            severity
            != "critical"
        ):
            severity = "suspect"

    # faster-whisper preservou o segmento, mas sem
    # alinhamento por palavra. A Passada 2 deve tentar
    # reconstruí-lo com contexto maior.
    if segment.get(
        "word_alignment_missing",
        False
    ):
        reasons.append(
            "missing_word_alignment"
        )

        if severity != "critical":
            severity = "suspect"

    return {
        "suspect": bool(
            reasons
        ),

        "severity": severity,

        "reasons": reasons,

        "metrics": {
            "word_count": len(words),

            "mean_word_probability": (
                round(
                    mean_probability,
                    4
                )
                if (
                    mean_probability
                    is not None
                )
                else None
            ),

            "min_word_probability": (
                round(
                    min_probability,
                    4
                )
                if (
                    min_probability
                    is not None
                )
                else None
            ),

            "avg_logprob": round(
                avg_logprob,
                4
            ),

            "compression_ratio": round(
                compression_ratio,
                4
            ),

            "repetition_ratio": round(
                repetition_ratio,
                4
            ),

            "zero_duration_ratio": round(
                zero_duration_ratio,
                4
            ),
        },
    }


def analyze_transcription_quality(
    transcription
):
    """
    Etapa 1:
      analisa cada segmento.

    Etapa 2:
      propaga suspeita para pequenos vizinhos
      de um segmento com script inesperado.
    """

    all_items = []

    segments = (
        transcription.get(
            "segments"
        )
        or []
    )

    for segment in segments:
        quality = (
            analyze_segment_quality(
                segment
            )
        )

        all_items.append({
            "segment_id": (
                segment.get(
                    "id"
                )
            ),

            "start": float(
                segment.get(
                    "start",
                    0
                )
            ),

            "end": float(
                segment.get(
                    "end",
                    0
                )
            ),

            "text": (
                segment.get(
                    "text",
                    ""
                )
            ),

            "words": (
                segment.get(
                    "words"
                )
                or []
            ),

            "quality": quality,
        })

    script_anchors = [
        item
        for item in all_items
        if (
            "unexpected_cyrillic_script"
            in item[
                "quality"
            ][
                "reasons"
            ]
            or
            "unexpected_latin_script"
            in item[
                "quality"
            ][
                "reasons"
            ]
        )
    ]

    for anchor in script_anchors:
        for item in all_items:
            if (
                item["segment_id"]
                == anchor[
                    "segment_id"
                ]
            ):
                continue

            if (
                item["end"]
                < anchor["start"]
            ):
                gap = (
                    anchor["start"]
                    - item["end"]
                )

            elif (
                item["start"]
                > anchor["end"]
            ):
                gap = (
                    item["start"]
                    - anchor["end"]
                )

            else:
                gap = 0.0

            if (
                gap
                > REPAIR_SCRIPT_NEIGHBOR_GAP_SECONDS
            ):
                continue

            if (
                len(
                    item[
                        "words"
                    ]
                )
                > REPAIR_SHORT_SEGMENT_MAX_WORDS
            ):
                continue

            reasons = (
                item[
                    "quality"
                ][
                    "reasons"
                ]
            )

            if (
                "neighbor_of_unexpected_script"
                not in reasons
            ):
                reasons.append(
                    "neighbor_of_unexpected_script"
                )

            item[
                "quality"
            ][
                "suspect"
            ] = True

            if (
                item[
                    "quality"
                ][
                    "severity"
                ]
                == "ok"
            ):
                item[
                    "quality"
                ][
                    "severity"
                ] = "suspect"

    return [
        item
        for item in all_items
        if (
            item[
                "quality"
            ][
                "suspect"
            ]
        )
    ]


In [15]:
# ============================================================
# CÉLULA 11B — Repair Groups v3
# ============================================================

def _merge_severity(
    current,
    new
):
    if (
        current == "critical"
        or new == "critical"
    ):
        return "critical"

    if (
        current == "suspect"
        or new == "suspect"
    ):
        return "suspect"

    return "ok"


def build_repair_regions(
    transcription
):
    """
    V3:
    suspeitos realmente adjacentes são agrupados.

    Exemplo:

      33.00 -> 33.98   suspect
      34.06 -> 34.84   suspect
      34.84 -> 35.80   suspect

    vira UM core:

      33.00 -> 35.80

    O large-v3 recebe padding antes/depois,
    mas somente o core agrupado pode substituir RAW.
    """

    suspects = (
        analyze_transcription_quality(
            transcription
        )
    )

    if not suspects:
        return [], []

    suspects = sorted(
        suspects,
        key=lambda item: (
            item["start"],
            item["end"]
        )
    )

    source_duration = float(
        transcription[
            "source"
        ][
            "duration_seconds"
        ]
    )

    processed_until = (
        transcription
        .get(
            "checkpointing",
            {}
        )
        .get(
            "processed_until"
        )
    )

    if (
        processed_until
        is not None
    ):
        effective_duration = min(
            source_duration,
            float(
                processed_until
            )
        )
    else:
        effective_duration = (
            source_duration
        )

    # --------------------------------------------------------
    # Agrupar cores adjacentes
    # --------------------------------------------------------

    groups = []

    for item in suspects:
        item_start = float(
            item["start"]
        )

        item_end = float(
            item["end"]
        )

        if not groups:
            groups.append({
                "core_start": item_start,
                "core_end": item_end,
                "items": [item],
                "severity": (
                    item[
                        "quality"
                    ][
                        "severity"
                    ]
                ),
            })
            continue

        previous = (
            groups[-1]
        )

        gap = (
            item_start
            - previous[
                "core_end"
            ]
        )

        combined_end = max(
            previous[
                "core_end"
            ],
            item_end
        )

        combined_duration = (
            combined_end
            - previous[
                "core_start"
            ]
        )

        can_group = (
            gap
            <= REPAIR_GROUP_MAX_GAP_SECONDS
            and
            combined_duration
            <= REPAIR_GROUP_MAX_CORE_SECONDS
        )

        if can_group:
            previous[
                "core_end"
            ] = combined_end

            previous[
                "items"
            ].append(
                item
            )

            previous[
                "severity"
            ] = (
                _merge_severity(
                    previous[
                        "severity"
                    ],
                    item[
                        "quality"
                    ][
                        "severity"
                    ]
                )
            )

        else:
            groups.append({
                "core_start": item_start,
                "core_end": item_end,
                "items": [item],
                "severity": (
                    item[
                        "quality"
                    ][
                        "severity"
                    ]
                ),
            })

    # --------------------------------------------------------
    # Construir regiões finais
    # --------------------------------------------------------

    regions = []

    for index, group in enumerate(
        groups,
        start=1
    ):
        core_start = float(
            group[
                "core_start"
            ]
        )

        core_end = float(
            group[
                "core_end"
            ]
        )

        context_start = max(
            0.0,
            core_start
            - REPAIR_PADDING_SECONDS
        )

        context_end = min(
            effective_duration,
            core_end
            + REPAIR_PADDING_SECONDS
        )

        reasons = sorted(
            set(
                reason
                for item in group[
                    "items"
                ]
                for reason in item[
                    "quality"
                ][
                    "reasons"
                ]
            )
        )

        segment_ids = [
            item[
                "segment_id"
            ]
            for item in group[
                "items"
            ]
        ]

        original_text = " ".join(
            item[
                "text"
            ].strip()
            for item in group[
                "items"
            ]
            if item[
                "text"
            ].strip()
        )

        original_word_count = sum(
            len(
                item[
                    "words"
                ]
            )
            for item in group[
                "items"
            ]
        )

        regions.append({
            "repair_id": index,

            "start": round(
                context_start,
                3
            ),

            "end": round(
                context_end,
                3
            ),

            "duration": round(
                context_end
                - context_start,
                3
            ),

            "core_start": round(
                core_start,
                3
            ),

            "core_end": round(
                core_end,
                3
            ),

            "core_duration": round(
                core_end
                - core_start,
                3
            ),

            "segment_ids": (
                segment_ids
            ),

            "group_size": (
                len(
                    segment_ids
                )
            ),

            "original_text": (
                original_text
            ),

            "original_word_count": (
                original_word_count
            ),

            "severity": (
                group[
                    "severity"
                ]
            ),

            "reasons": (
                reasons
            ),

            "original_items": [
                {
                    "segment_id": item[
                        "segment_id"
                    ],
                    "start": item[
                        "start"
                    ],
                    "end": item[
                        "end"
                    ],
                    "text": item[
                        "text"
                    ],
                    "quality": item[
                        "quality"
                    ],
                }
                for item in group[
                    "items"
                ]
            ],
        })

    return (
        suspects,
        regions
    )


In [16]:
# ============================================================
# CÉLULA 11C — Passada 2 / Repair Decision v3
# ============================================================

from faster_whisper import WhisperModel
import uuid
import copy


REPAIR_MODEL = None


# ============================================================
# Modelo
# ============================================================

def get_repair_model():
    global REPAIR_MODEL, MODEL

    if (
        REPAIR_MODEL
        is not None
    ):
        return REPAIR_MODEL

    print()
    print("=" * 80)
    print(
        "PASSADA 2 — CARREGANDO MODELO"
    )
    print("=" * 80)

    # A Passada 1 já terminou.
    # Não há motivo para manter large-v3-turbo residente
    # enquanto carregamos large-v3 na mesma T4.
    if (
        MODEL
        is not None
    ):
        print(
            "Liberando modelo da Passada 1 "
            "antes de carregar large-v3..."
        )

        MODEL = None

        release_gpu_memory()

        gpu_memory_snapshot(
            "após liberar Passada 1"
        )

    print(
        f"Modelo: "
        f"{REPAIR_MODEL_NAME}"
    )

    print(
        f"Device: "
        f"{ACTIVE_RUNTIME['device']}"
    )

    print(
        f"Compute: "
        f"{ACTIVE_RUNTIME['compute_type']}"
    )

    REPAIR_MODEL = WhisperModel(
        REPAIR_MODEL_NAME,

        device=ACTIVE_RUNTIME[
            "device"
        ],

        compute_type=ACTIVE_RUNTIME[
            "compute_type"
        ],
    )

    print(
        "Modelo de repair carregado "
        "com sucesso."
    )

    gpu_memory_snapshot(
        "após carregar Passada 2"
    )

    return REPAIR_MODEL


# ============================================================
# Áudio temporário
# ============================================================

def extract_repair_audio(
    source_path,
    start,
    end
):
    source_path = Path(
        source_path
    )

    start = float(start)
    end = float(end)

    duration = (
        end - start
    )

    if duration <= 0:
        raise ValueError(
            f"Região inválida: "
            f"{start:.3f}s -> {end:.3f}s"
        )

    temp_dir = Path(
        "/content/raru_revive"
    )

    temp_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    output = (
        temp_dir
        /
        (
            "repair_"
            + uuid.uuid4().hex
            + ".flac"
        )
    )

    cmd = [
        "ffmpeg",
        "-hide_banner",
        "-loglevel",
        "error",
        "-y",

        "-ss",
        f"{start:.3f}",

        "-i",
        str(source_path),

        "-t",
        f"{duration:.3f}",

        "-vn",
        "-ac",
        "1",
        "-ar",
        "16000",
        "-c:a",
        "flac",

        str(output),
    ]

    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )

    if (
        result.returncode
        != 0
    ):
        raise RuntimeError(
            "FFmpeg falhou durante "
            "a extração do repair.\n\n"
            + result.stderr
        )

    if (
        not output.exists()
        or
        output.stat().st_size
        <= 0
    ):
        raise RuntimeError(
            "Arquivo temporário do repair "
            "não foi criado corretamente."
        )

    return output


# ============================================================
# Timestamp local -> absoluto, recortado no CORE
# ============================================================

def convert_candidate_to_absolute(
    candidate_segments,
    region
):
    absolute_segments = []

    region_start = float(
        region[
            "start"
        ]
    )

    core_start = float(
        region[
            "core_start"
        ]
    )

    core_end = float(
        region[
            "core_end"
        ]
    )

    for segment in (
        candidate_segments
        or []
    ):
        retained_words = []

        for word in (
            segment.get(
                "words"
            )
            or []
        ):
            absolute_start = (
                region_start
                +
                float(
                    word[
                        "start"
                    ]
                )
            )

            absolute_end = (
                region_start
                +
                float(
                    word[
                        "end"
                    ]
                )
            )

            midpoint = (
                absolute_start
                + absolute_end
            ) / 2.0

            if not (
                core_start
                <= midpoint
                <= core_end
            ):
                continue

            clipped_start = max(
                absolute_start,
                core_start
            )

            clipped_end = min(
                absolute_end,
                core_end
            )

            if (
                clipped_end
                <= clipped_start
            ):
                continue

            retained_words.append({
                "start": round(
                    clipped_start,
                    3
                ),

                "end": round(
                    clipped_end,
                    3
                ),

                "word": (
                    word[
                        "word"
                    ]
                ),

                "probability": round(
                    float(
                        word[
                            "probability"
                        ]
                    ),
                    5
                ),
            })

        if not retained_words:
            continue

        text = "".join(
            word["word"]
            for word in retained_words
        ).strip()

        if not text:
            continue

        absolute_segments.append({
            "start": (
                retained_words[0][
                    "start"
                ]
            ),

            "end": (
                retained_words[-1][
                    "end"
                ]
            ),

            "text": text,

            "words": (
                retained_words
            ),

            "avg_logprob": (
                segment.get(
                    "avg_logprob"
                )
            ),

            "compression_ratio": (
                segment.get(
                    "compression_ratio"
                )
            ),

            "no_speech_prob": (
                segment.get(
                    "no_speech_prob",
                    0
                )
            ),

            "temperature": 0.0,

            "origin": "repair",
            "repair_model": REPAIR_MODEL_NAME,
        })

    return absolute_segments


# ============================================================
# Transcrever candidato
# ============================================================

def transcribe_repair_candidate(
    audio_path,
    region,
    mode
):
    model = (
        get_repair_model()
    )

    if mode == "pt":
        language = "pt"
        multilingual = False

    elif mode == "en":
        language = "en"
        multilingual = False

    elif mode == "auto":
        language = None
        multilingual = True

    else:
        raise ValueError(
            f"Modo inválido: {mode}"
        )

    segments_iter, info = (
        model.transcribe(
            str(audio_path),

            language=language,
            multilingual=multilingual,

            task="transcribe",

            beam_size=5,
            temperature=0.0,

            condition_on_previous_text=False,

            word_timestamps=True,

            # Sem chunk_length=3:
            # a Passada 2 precisa de contexto.
            vad_filter=False,

            no_speech_threshold=None,
            hallucination_silence_threshold=None,

            suppress_tokens=[-1],
            log_progress=False,
        )
    )

    raw_segments = []

    for segment in segments_iter:
        words = []

        if segment.words:
            for word in segment.words:
                words.append({
                    "start": round(
                        float(
                            word.start
                        ),
                        3
                    ),

                    "end": round(
                        float(
                            word.end
                        ),
                        3
                    ),

                    "word": (
                        word.word
                    ),

                    "probability": round(
                        float(
                            word.probability
                        ),
                        5
                    ),
                })

        raw_segments.append({
            "start": round(
                float(
                    segment.start
                ),
                3
            ),

            "end": round(
                float(
                    segment.end
                ),
                3
            ),

            "text": (
                segment.text.strip()
            ),

            "words": words,

            "avg_logprob": round(
                float(
                    segment.avg_logprob
                ),
                6
            ),

            "compression_ratio": round(
                float(
                    segment.compression_ratio
                ),
                6
            ),

            "no_speech_prob": round(
                float(
                    segment.no_speech_prob
                ),
                6
            ),
        })

    absolute_segments = (
        convert_candidate_to_absolute(
            raw_segments,
            region
        )
    )

    return {
        "mode": mode,

        "language_requested": (
            language
        ),

        "multilingual": (
            multilingual
        ),

        "language_detected": (
            info.language
        ),

        "language_probability": (
            float(
                info.language_probability
            )
            if (
                info.language_probability
                is not None
            )
            else None
        ),

        "segments": (
            absolute_segments
        ),
    }


# ============================================================
# Métricas
# ============================================================

def _repair_segment_metrics(
    segments
):
    words = []
    probabilities = []
    avg_logprobs = []

    max_compression = 0.0

    texts = []
    starts = []
    ends = []

    for segment in (
        segments
        or []
    ):
        text = (
            segment.get(
                "text"
            )
            or ""
        ).strip()

        if text:
            texts.append(
                text
            )

        if (
            segment.get(
                "start"
            )
            is not None
        ):
            starts.append(
                float(
                    segment[
                        "start"
                    ]
                )
            )

        if (
            segment.get(
                "end"
            )
            is not None
        ):
            ends.append(
                float(
                    segment[
                        "end"
                    ]
                )
            )

        logprob = segment.get(
            "avg_logprob"
        )

        if logprob is not None:
            avg_logprobs.append(
                float(logprob)
            )

        compression = segment.get(
            "compression_ratio"
        )

        if compression is not None:
            max_compression = max(
                max_compression,
                float(
                    compression
                )
            )

        for word in (
            segment.get(
                "words"
            )
            or []
        ):
            words.append(
                word
            )

            probability = (
                word.get(
                    "probability"
                )
            )

            if (
                probability
                is not None
            ):
                probabilities.append(
                    float(
                        probability
                    )
                )

    text = " ".join(
        texts
    ).strip()

    mean_probability = (
        sum(probabilities)
        / len(probabilities)
        if probabilities
        else 0.0
    )

    mean_logprob = (
        sum(avg_logprobs)
        / len(avg_logprobs)
        if avg_logprobs
        else -2.0
    )

    repetition_ratio = (
        _repetition_ratio(
            words,
            minimum_words=(
                REPAIR_CANDIDATE_REPETITION_MIN_WORDS
            )
        )
    )

    temporal_span = (
        max(
            0.0,
            max(ends)
            - min(starts)
        )
        if (
            starts
            and ends
        )
        else 0.0
    )

    return {
        "text": text,
        "words": words,
        "word_count": len(words),
        "mean_word_probability": mean_probability,
        "mean_logprob": mean_logprob,
        "max_compression_ratio": max_compression,
        "repetition_ratio": repetition_ratio,
        "temporal_span": temporal_span,
    }


def repair_quality_score(
    segments
):
    metrics = (
        _repair_segment_metrics(
            segments
        )
    )

    if (
        metrics[
            "word_count"
        ]
        == 0
    ):
        return 0.0

    mean_probability = (
        metrics[
            "mean_word_probability"
        ]
    )

    mean_logprob = (
        metrics[
            "mean_logprob"
        ]
    )

    log_score = (
        mean_logprob
        + 1.5
    ) / 1.5

    log_score = max(
        0.0,
        min(
            1.0,
            log_score
        )
    )

    score = (
        mean_probability
        * 0.75
        +
        log_score
        * 0.25
    )

    return round(
        max(
            0.0,
            min(
                1.0,
                score
            )
        ),
        4
    )


# ============================================================
# Validação dos candidatos
# ============================================================

def validate_repair_candidate(
    candidate,
    region
):
    reasons = []

    metrics = (
        _repair_segment_metrics(
            candidate[
                "segments"
            ]
        )
    )

    text = (
        metrics[
            "text"
        ]
    )

    if (
        not text
        or
        metrics[
            "word_count"
        ]
        == 0
    ):
        reasons.append(
            "empty_candidate"
        )

    if (
        metrics[
            "repetition_ratio"
        ]
        >= REPAIR_CANDIDATE_REPETITION_RATIO
    ):
        reasons.append(
            "candidate_repetition_loop"
        )

    if (
        metrics[
            "max_compression_ratio"
        ]
        >= REPAIR_CANDIDATE_MAX_COMPRESSION_RATIO
    ):
        reasons.append(
            "candidate_high_compression_ratio"
        )

    if (
        metrics[
            "word_count"
        ]
        > 0
        and
        metrics[
            "mean_word_probability"
        ]
        < REPAIR_CANDIDATE_MIN_MEAN_WORD_PROB
    ):
        reasons.append(
            "candidate_low_word_probability"
        )

    if (
        _contains_cyrillic(
            text
        )
    ):
        reasons.append(
            "candidate_unexpected_cyrillic"
        )

    if (
        _contains_strong_unexpected_latin(
            text
        )
    ):
        reasons.append(
            "candidate_unexpected_latin_script"
        )

    if (
        candidate[
            "mode"
        ]
        == "auto"
    ):
        detected = (
            candidate.get(
                "language_detected"
            )
        )

        probability = (
            candidate.get(
                "language_probability"
            )
        )

        if (
            detected
            and
            detected
            not in REPAIR_ALLOWED_LANGUAGES
            and
            probability is not None
            and
            probability
            >= REPAIR_OTHER_LANGUAGE_REJECT_PROB
        ):
            reasons.append(
                "candidate_unexpected_language"
            )

    core_duration = max(
        0.001,
        float(
            region[
                "core_duration"
            ]
        )
    )

    coverage = (
        metrics[
            "temporal_span"
        ]
        / core_duration
    )

    if (
        region[
            "original_word_count"
        ]
        >= 4
        and
        coverage
        < REPAIR_CANDIDATE_MIN_COVERAGE
    ):
        reasons.append(
            "candidate_insufficient_coverage"
        )

    if (
        region[
            "severity"
        ]
        != "critical"
        and
        region[
            "original_word_count"
        ]
        >= 4
    ):
        minimum_words = max(
            1,
            math.ceil(
                region[
                    "original_word_count"
                ]
                * 0.25
            )
        )

        if (
            metrics[
                "word_count"
            ]
            < minimum_words
            and
            coverage < 0.50
        ):
            reasons.append(
                "candidate_severe_content_collapse"
            )

    return {
        "valid": (
            len(reasons)
            == 0
        ),

        "reasons": reasons,

        "metrics": {
            "word_count": (
                metrics[
                    "word_count"
                ]
            ),

            "mean_word_probability": round(
                metrics[
                    "mean_word_probability"
                ],
                4
            ),

            "mean_logprob": round(
                metrics[
                    "mean_logprob"
                ],
                4
            ),

            "max_compression_ratio": round(
                metrics[
                    "max_compression_ratio"
                ],
                4
            ),

            "repetition_ratio": round(
                metrics[
                    "repetition_ratio"
                ],
                4
            ),

            "coverage": round(
                coverage,
                4
            ),
        },
    }


# ============================================================
# Escolher vencedor
# ============================================================

def choose_repair_winner(
    candidates,
    region
):
    for candidate in candidates:
        validation = (
            validate_repair_candidate(
                candidate,
                region
            )
        )

        candidate[
            "validation"
        ] = validation

        if (
            validation[
                "valid"
            ]
        ):
            candidate[
                "score"
            ] = (
                repair_quality_score(
                    candidate[
                        "segments"
                    ]
                )
            )
        else:
            candidate[
                "score"
            ] = 0.0

        candidate[
            "text"
        ] = (
            _repair_segment_metrics(
                candidate[
                    "segments"
                ]
            )[
                "text"
            ]
        )

    valid_candidates = [
        candidate
        for candidate in candidates
        if (
            candidate[
                "validation"
            ][
                "valid"
            ]
        )
    ]

    if not valid_candidates:
        return None

    auto_candidate = next(
        (
            candidate
            for candidate
            in valid_candidates
            if (
                candidate[
                    "mode"
                ]
                == "auto"
            )
        ),
        None
    )

    forced_candidates = [
        candidate
        for candidate
        in valid_candidates
        if (
            candidate[
                "mode"
            ]
            in (
                "pt",
                "en",
            )
        )
    ]

    best_forced = (
        max(
            forced_candidates,
            key=lambda item:
            item[
                "score"
            ]
        )
        if forced_candidates
        else None
    )

    if (
        auto_candidate
        is not None
    ):
        if (
            best_forced
            is not None
            and
            best_forced[
                "score"
            ]
            >= (
                auto_candidate[
                    "score"
                ]
                +
                REPAIR_FORCED_LANGUAGE_MIN_GAIN
            )
        ):
            return best_forced

        return auto_candidate

    return max(
        valid_candidates,
        key=lambda item:
        item[
            "score"
        ]
    )


# ============================================================
# RAW original do CORE
# ============================================================

def original_segments_for_region(
    transcription,
    region
):
    result = []

    core_start = float(
        region[
            "core_start"
        ]
    )

    core_end = float(
        region[
            "core_end"
        ]
    )

    for segment in (
        transcription[
            "segments"
        ]
    ):
        start = float(
            segment[
                "start"
            ]
        )

        end = float(
            segment[
                "end"
            ]
        )

        if (
            end > core_start
            and
            start < core_end
        ):
            result.append(
                segment
            )

    return result


# ============================================================
# Confiança do repair
# ============================================================

def _repair_confidence_class(
    winner
):
    if winner is None:
        return None

    score = float(
        winner.get(
            "score",
            0
        )
    )

    if (
        score
        >= REPAIR_CONFIDENCE_HIGH
    ):
        return "high"

    if (
        score
        >= REPAIR_CONFIDENCE_MEDIUM
    ):
        return "medium"

    return "low"


# ============================================================
# Decisão final
# ============================================================

def _decide_repair_action(
    region,
    original_score,
    winner
):
    # CRITICAL:
    # original já foi considerado patológico.
    if (
        region[
            "severity"
        ]
        == "critical"
    ):
        if (
            winner is not None
            and
            winner[
                "score"
            ]
            >= REPAIR_CRITICAL_MIN_SCORE
        ):
            return "replace"

        return "suppress"

    if winner is None:
        return "keep"

    winner_score = float(
        winner[
            "score"
        ]
    )

    # Script inesperado / vizinho:
    # evidência forte de que o RAW está errado.
    script_problem = any(
        reason
        in region[
            "reasons"
        ]
        for reason in (
            "unexpected_cyrillic_script",
            "unexpected_latin_script",
            "neighbor_of_unexpected_script",
        )
    )

    if (
        script_problem
        and
        winner_score
        >= REPAIR_SCRIPT_MIN_SCORE
    ):
        return "replace"

    gain = (
        winner_score
        - original_score
    )

    if (
        gain
        >= REPAIR_MIN_SCORE_GAIN
    ):
        return "replace"

    boundary_problem = any(
        reason
        in region[
            "reasons"
        ]
        for reason in (
            "low_confidence_near_chunk_boundary",
            "unfinished_phrase_at_boundary",
        )
    )

    if (
        boundary_problem
        and
        winner_score
        >= REPAIR_BOUNDARY_TRUST_MIN_SCORE
        and
        winner_score
        >= (
            original_score
            -
            REPAIR_BOUNDARY_MAX_SCORE_DROP
        )
    ):
        return "replace"

    return "keep"


def _candidate_summary(
    candidate
):
    return {
        "mode": (
            candidate[
                "mode"
            ]
        ),

        "language_requested": (
            candidate.get(
                "language_requested"
            )
        ),

        "language_detected": (
            candidate.get(
                "language_detected"
            )
        ),

        "language_probability": (
            candidate.get(
                "language_probability"
            )
        ),

        "score": (
            candidate.get(
                "score",
                0.0
            )
        ),

        "text": (
            candidate.get(
                "text",
                ""
            )
        ),

        "valid": (
            candidate[
                "validation"
            ][
                "valid"
            ]
        ),

        "validation_reasons": (
            candidate[
                "validation"
            ][
                "reasons"
            ]
        ),

        "validation_metrics": (
            candidate[
                "validation"
            ][
                "metrics"
            ]
        ),
    }


# ============================================================
# Processar um Repair Group
# ============================================================

def repair_one_region(
    source_path,
    transcription,
    region
):
    print()
    print("=" * 80)

    print(
        f'REPAIR GROUP '
        f'{region["repair_id"]:03d}'
    )

    print(
        f'Contexto: '
        f'{region["start"]:.2f}s '
        f'→ '
        f'{region["end"]:.2f}s'
    )

    print(
        f'CORE: '
        f'{region["core_start"]:.2f}s '
        f'→ '
        f'{region["core_end"]:.2f}s'
    )

    print(
        f'Segmentos agrupados: '
        f'{region["segment_ids"]}'
    )

    print(
        "Motivos:",
        ", ".join(
            region[
                "reasons"
            ]
        )
    )

    original_segments = (
        original_segments_for_region(
            transcription,
            region
        )
    )

    original_score = (
        repair_quality_score(
            original_segments
        )
    )

    print(
        f"Score original: "
        f"{original_score:.4f}"
    )

    print(
        "Original:",
        region[
            "original_text"
        ]
    )

    repair_audio = (
        extract_repair_audio(
            source_path,
            region[
                "start"
            ],
            region[
                "end"
            ]
        )
    )

    candidates = []

    try:
        for mode in (
            "auto",
            "pt",
            "en",
        ):
            candidate = (
                transcribe_repair_candidate(
                    repair_audio,
                    region,
                    mode
                )
            )

            candidates.append(
                candidate
            )

    finally:
        try:
            repair_audio.unlink(
                missing_ok=True
            )
        except Exception:
            pass

    winner = (
        choose_repair_winner(
            candidates,
            region
        )
    )

    for candidate in candidates:
        validation = (
            candidate[
                "validation"
            ]
        )

        state = (
            "VALID"
            if (
                validation[
                    "valid"
                ]
            )
            else "REJECTED"
        )

        print()
        print(
            f'[{candidate["mode"].upper()}] '
            f'{state} '
            f'score='
            f'{candidate["score"]:.4f}'
        )

        print(
            candidate[
                "text"
            ]
        )

        if (
            not validation[
                "valid"
            ]
        ):
            print(
                "  Rejeitado por:",
                ", ".join(
                    validation[
                        "reasons"
                    ]
                )
            )

    action = (
        _decide_repair_action(
            region,
            original_score,
            winner
        )
    )

    confidence_class = (
        _repair_confidence_class(
            winner
        )
    )

    print()

    if (
        action
        == "replace"
    ):
        print(
            "✓ REPAIR ACEITO"
        )

        print(
            f"Vencedor: "
            f"{winner['mode']}"
        )

        print(
            f"Score: "
            f"{winner['score']:.4f}"
        )

        print(
            f"Confiança: "
            f"{confidence_class}"
        )

    elif (
        action
        == "suppress"
    ):
        print(
            "⚠ CRITICAL NÃO RESOLVIDO"
        )

        print(
            "Original será removido do "
            "texto final e preservado "
            "somente na auditoria."
        )

    else:
        print(
            "→ Original preservado"
        )

    winner_for_json = None

    if (
        winner
        is not None
    ):
        winner_for_json = {
            "mode": (
                winner[
                    "mode"
                ]
            ),

            "score": (
                winner[
                    "score"
                ]
            ),

            "confidence_class": (
                confidence_class
            ),

            "text": (
                winner[
                    "text"
                ]
            ),

            "language_detected": (
                winner.get(
                    "language_detected"
                )
            ),

            "language_probability": (
                winner.get(
                    "language_probability"
                )
            ),

            "segments": copy.deepcopy(
                winner[
                    "segments"
                ]
            ),
        }

    return {
        "repair_id": (
            region[
                "repair_id"
            ]
        ),

        "segment_ids": (
            region[
                "segment_ids"
            ]
        ),

        "group_size": (
            region[
                "group_size"
            ]
        ),

        "start": (
            region[
                "start"
            ]
        ),

        "end": (
            region[
                "end"
            ]
        ),

        "core_start": (
            region[
                "core_start"
            ]
        ),

        "core_end": (
            region[
                "core_end"
            ]
        ),

        "severity": (
            region[
                "severity"
            ]
        ),

        "reasons": (
            region[
                "reasons"
            ]
        ),

        "original_text": (
            region[
                "original_text"
            ]
        ),

        "original_score": (
            original_score
        ),

        "action": (
            action
        ),

        "confidence_class": (
            confidence_class
        ),

        "winner": (
            winner_for_json
        ),

        "candidates": [
            _candidate_summary(
                candidate
            )
            for candidate
            in candidates
        ],
    }


# ============================================================
# Aplicar repairs
# ============================================================

def apply_repairs_to_segments(
    original_segments,
    repair_results
):
    segments = copy.deepcopy(
        original_segments
    )

    for repair in sorted(
        repair_results,
        key=lambda item:
        item[
            "core_start"
        ]
    ):
        action = (
            repair[
                "action"
            ]
        )

        if (
            action
            == "keep"
        ):
            continue

        core_start = float(
            repair[
                "core_start"
            ]
        )

        core_end = float(
            repair[
                "core_end"
            ]
        )

        # Replace e suppress removem RAW dentro do CORE.
        segments = [
            segment
            for segment in segments
            if not (
                float(
                    segment[
                        "end"
                    ]
                )
                > core_start
                and
                float(
                    segment[
                        "start"
                    ]
                )
                < core_end
            )
        ]

        if (
            action
            == "replace"
        ):
            winner = (
                repair.get(
                    "winner"
                )
            )

            if (
                winner
                and
                winner.get(
                    "segments"
                )
            ):
                new_segments = (
                    copy.deepcopy(
                        winner[
                            "segments"
                        ]
                    )
                )

                for segment in new_segments:
                    segment[
                        "repair_id"
                    ] = (
                        repair[
                            "repair_id"
                        ]
                    )

                    segment[
                        "repair_confidence_class"
                    ] = (
                        repair[
                            "confidence_class"
                        ]
                    )

                segments.extend(
                    new_segments
                )

    segments.sort(
        key=lambda segment: (
            float(
                segment[
                    "start"
                ]
            ),
            float(
                segment[
                    "end"
                ]
            ),
        )
    )

    return segments


# ============================================================
# Overlaps residuais
# ============================================================

def normalize_small_overlaps(
    segments
):
    if (
        len(segments)
        < 2
    ):
        return (
            segments,
            0,
            0
        )

    segments = copy.deepcopy(
        segments
    )

    fixed = 0
    remaining = 0

    for index in range(
        1,
        len(segments)
    ):
        previous = (
            segments[
                index - 1
            ]
        )

        current = (
            segments[
                index
            ]
        )

        previous_end = float(
            previous[
                "end"
            ]
        )

        current_start = float(
            current[
                "start"
            ]
        )

        overlap = (
            previous_end
            - current_start
        )

        if overlap <= 0:
            continue

        if (
            overlap
            <= REPAIR_MAX_AUTO_OVERLAP_FIX_SECONDS
        ):
            new_end = (
                current_start
            )

            previous[
                "end"
            ] = round(
                new_end,
                3
            )

            words = (
                previous.get(
                    "words"
                )
                or []
            )

            if words:
                words[-1][
                    "end"
                ] = round(
                    min(
                        float(
                            words[-1][
                                "end"
                            ]
                        ),
                        new_end
                    ),
                    3
                )

            fixed += 1

        else:
            remaining += 1

    cleaned = [
        segment
        for segment in segments
        if (
            float(
                segment[
                    "end"
                ]
            )
            >
            float(
                segment[
                    "start"
                ]
            )
        )
    ]

    for index, segment in enumerate(
        cleaned,
        start=1
    ):
        segment[
            "id"
        ] = index

    return (
        cleaned,
        fixed,
        remaining
    )


# ============================================================
# Escrita dos outputs REPAIRED
# ============================================================

def _atomic_repair_json(
    path,
    data
):
    atomic_json_dump(
        data,
        path
    )


def _repair_srt_timestamp(
    seconds
):
    return sec_to_clock(
        seconds,
        True
    )


def save_repaired_text_outputs(
    repaired,
    json_path
):
    json_path = Path(
        json_path
    )

    txt_path = (
        json_path.with_suffix(
            ".txt"
        )
    )

    srt_path = (
        json_path.with_suffix(
            ".srt"
        )
    )

    with open(
        txt_path,
        "w",
        encoding="utf-8"
    ) as f:
        for segment in (
            repaired[
                "segments"
            ]
        ):
            f.write(
                f'['
                f'{segment["start"]:.3f}'
                f' --> '
                f'{segment["end"]:.3f}'
                f']\n'
            )

            f.write(
                segment[
                    "text"
                ].strip()
                + "\n"
            )

    with open(
        srt_path,
        "w",
        encoding="utf-8"
    ) as f:
        for index, segment in enumerate(
            repaired[
                "segments"
            ],
            start=1
        ):
            f.write(
                f"{index}\n"
            )

            f.write(
                _repair_srt_timestamp(
                    segment[
                        "start"
                    ]
                )
                + " --> "
                + _repair_srt_timestamp(
                    segment[
                        "end"
                    ]
                )
                + "\n"
            )

            f.write(
                segment[
                    "text"
                ].strip()
                + "\n\n"
            )

    return (
        txt_path,
        srt_path
    )


# ============================================================
# Orquestrador da Passada 2
# ============================================================

def run_repair_pass(
    source_path,
    transcription,
    test_mode=False
):
    t_repair = (
        time.perf_counter()
    )

    print()
    print("=" * 80)
    print(
        "PASSADA 2 — QUALITY ANALYZER / REPAIR GROUPS V3"
    )
    print("=" * 80)

    suspects, regions = (
        build_repair_regions(
            transcription
        )
    )

    print(
        f"Segmentos suspeitos: "
        f"{len(suspects)}"
    )

    print(
        f"Repair Groups: "
        f"{len(regions)}"
    )

    grouped_segments = sum(
        max(
            0,
            region[
                "group_size"
            ]
            - 1
        )
        for region in regions
    )

    print(
        f"Segmentos absorvidos por agrupamento: "
        f"{grouped_segments}"
    )

    if not regions:
        print(
            "Nenhuma região precisa "
            "de repair."
        )

        return transcription

    repair_results = []

    for region in regions:
        result = (
            repair_one_region(
                source_path,
                transcription,
                region
            )
        )

        repair_results.append(
            result
        )

    repaired = copy.deepcopy(
        transcription
    )

    final_segments = (
        apply_repairs_to_segments(
            transcription[
                "segments"
            ],
            repair_results
        )
    )

    (
        final_segments,
        overlaps_fixed,
        overlaps_remaining,
    ) = (
        normalize_small_overlaps(
            final_segments
        )
    )

    repaired[
        "segments"
    ] = (
        final_segments
    )

    repaired[
        "status"
    ] = (
        "test_repaired"
        if test_mode
        else "completed_repaired"
    )

    replaced_count = sum(
        1
        for result in repair_results
        if (
            result[
                "action"
            ]
            == "replace"
        )
    )

    kept_count = sum(
        1
        for result in repair_results
        if (
            result[
                "action"
            ]
            == "keep"
        )
    )

    suppressed_count = sum(
        1
        for result in repair_results
        if (
            result[
                "action"
            ]
            == "suppress"
        )
    )

    rejected_candidates = sum(
        1
        for result in repair_results
        for candidate in result[
            "candidates"
        ]
        if (
            not candidate[
                "valid"
            ]
        )
    )

    confidence_counts = {
        "high": 0,
        "medium": 0,
        "low": 0,
    }

    for result in repair_results:
        if (
            result[
                "action"
            ]
            != "replace"
        ):
            continue

        confidence = (
            result.get(
                "confidence_class"
            )
        )

        if (
            confidence
            in confidence_counts
        ):
            confidence_counts[
                confidence
            ] += 1

    elapsed = (
        time.perf_counter()
        - t_repair
    )

    repaired[
        "repair_pass"
    ] = {
        "version": "3.0",
        "strategy": "adjacent_repair_groups",
        "enabled": True,
        "model": REPAIR_MODEL_NAME,

        "elapsed_seconds": round(
            elapsed,
            3
        ),

        "suspect_segments": (
            len(
                suspects
            )
        ),

        "repair_groups": (
            len(
                regions
            )
        ),

        "segments_absorbed_by_grouping": (
            grouped_segments
        ),

        "regions_replaced": (
            replaced_count
        ),

        "regions_kept": (
            kept_count
        ),

        "regions_suppressed": (
            suppressed_count
        ),

        "candidates_rejected": (
            rejected_candidates
        ),

        "confidence_counts": (
            confidence_counts
        ),

        "small_overlaps_fixed": (
            overlaps_fixed
        ),

        "overlaps_remaining": (
            overlaps_remaining
        ),

        "results": (
            repair_results
        ),
    }

    paths = job_paths(
        Path(
            source_path
        ),
        test_mode=test_mode
    )

    raw_json = (
        paths[
            "final_json"
        ]
    )

    repaired_json = (
        raw_json.with_name(
            raw_json.stem
            + ".REPAIRED.json"
        )
    )

    _atomic_repair_json(
        repaired_json,
        repaired
    )

    (
        txt_path,
        srt_path,
    ) = (
        save_repaired_text_outputs(
            repaired,
            repaired_json
        )
    )

    print()
    print("=" * 80)
    print(
        "PASSADA 2 CONCLUÍDA — V3"
    )
    print("=" * 80)

    print(
        f"Repair Groups : "
        f"{len(regions)}"
    )

    print(
        f"Reparados     : "
        f"{replaced_count}"
    )

    print(
        f"Preservados   : "
        f"{kept_count}"
    )

    print(
        f"Suprimidos    : "
        f"{suppressed_count}"
    )

    print(
        f"Candidatos rejeitados: "
        f"{rejected_candidates}"
    )

    print(
        f"Confiança HIGH   : "
        f"{confidence_counts['high']}"
    )

    print(
        f"Confiança MEDIUM : "
        f"{confidence_counts['medium']}"
    )

    print(
        f"Confiança LOW    : "
        f"{confidence_counts['low']}"
    )

    print(
        f"Overlaps corrigidos: "
        f"{overlaps_fixed}"
    )

    print(
        f"Overlaps restantes: "
        f"{overlaps_remaining}"
    )

    print()

    print(
        f"JSON: "
        f"{repaired_json}"
    )

    print(
        f"TXT : "
        f"{txt_path}"
    )

    print(
        f"SRT : "
        f"{srt_path}"
    )

    return repaired


In [17]:
# ============================================================
# CÉLULA 11D — Language Guard v3.1
# Coloque DEPOIS da 11C e ANTES da célula de execução.
# ============================================================

import re
import unicodedata

# ---------- configuração ----------
LANG_V31_LOW_WORD_PROB = 0.25
LANG_V31_LOW_WORD_MIN_COUNT = 2
LANG_V31_LOW_WORD_RATIO = 0.25

LANG_V31_ENGLISH_MAX_WORDS = 8
LANG_V31_CONTEXT_GAP_SECONDS = 4.0
LANG_V31_CONTEXT_SCAN_SEGMENTS = 4

LANG_V31_PT_MIN_SCORE = 0.55
LANG_V31_EN_MIN_SCORE = 0.78
LANG_V31_EN_AUTO_PROB = 0.85

LANG_V31_NORMAL_MIN_SCORE = 0.70
LANG_V31_SCRIPT_MIN_SCORE = 0.65
LANG_V31_CRITICAL_MIN_SCORE = 0.60
LANG_V31_ISOLATED_EN_MIN_SCORE = 0.75
LANG_V31_ISOLATED_EN_MAX_SCORE_DROP = 0.05

_PT_HINTS_V31 = {
    "a","agora","aí","aqui","as","assim","até","beleza","bora","cara","com","como",
    "da","de","do","e","é","ele","ela","em","essa","esse","eu","foi","gente","isso",
    "já","mais","mano","me","meu","minha","na","não","né","no","nos","nossa","o",
    "olha","os","para","peraí","por","pra","que","rapaz","se","sim","só","tá","tô",
    "um","uma","vai","vamos","velho","você","vocês",
}

_EN_HINTS_V31 = {
    "alive","and","are","break","bye","can","can't","cant","come","do","don't","dont",
    "for","go","good","happy","he","hello","hey","how","i","i'm","im","in","is","it",
    "my","of","on","out","pull","run","she","stop","stress","thank","thanks","that",
    "the","they","this","to","wait","was","we","were","what","why","with","you","your",
}


# ============================================================
# HELPERS
# ============================================================

def _v31_tokens(text):
    return [
        t.lower()
        for t in re.findall(
            r"[A-Za-zÀ-ÿ]+(?:'[A-Za-zÀ-ÿ]+)?",
            text or ""
        )
    ]


def _language_hint_v31(text):
    tokens = _v31_tokens(text)

    pt_score = sum(
        t in _PT_HINTS_V31
        for t in tokens
    )

    en_score = sum(
        t in _EN_HINTS_V31
        for t in tokens
    )

    # Acentuação típica de português ajuda como evidência.
    if re.search(
        r"[ãõáéíóúâêôàç]",
        (text or "").lower()
    ):
        pt_score += 1

    if pt_score > en_score and pt_score:
        language = "pt"

    elif en_score > pt_score and en_score:
        language = "en"

    else:
        language = "unknown"

    return {
        "language": language,
        "pt_score": int(pt_score),
        "en_score": int(en_score),
        "tokens": tokens,
    }


def _contains_non_latin_v31(text):
    """
    Permite:
      português
      inglês
      espanhol
      francês
      nomes próprios em alfabeto latino

    Ignora:
      números
      pontuação
      símbolos
      emoji

    Detecta como suspeito:
      cirílico
      japonês
      coreano
      chinês
      grego
      árabe
      hebraico
      etc.
    """

    for char in (
        text
        or ""
    ):
        if char.isalpha():

            name = unicodedata.name(
                char,
                ""
            )

            if (
                "LATIN"
                not in name
            ):
                return True

    return False


def _very_low_words_v31(
    segment
):
    probabilities = [
        float(
            word[
                "probability"
            ]
        )
        for word in (
            segment.get(
                "words"
            )
            or []
        )
        if (
            word.get(
                "probability"
            )
            is not None
        )
    ]

    if not probabilities:
        return (
            0,
            0.0
        )

    count = sum(
        probability
        < LANG_V31_LOW_WORD_PROB
        for probability
        in probabilities
    )

    return (
        count,
        count
        /
        len(
            probabilities
        )
    )


def _make_item_v31(
    segment
):
    return {
        "segment_id": (
            segment.get(
                "id"
            )
        ),

        "start": float(
            segment.get(
                "start",
                0
            )
        ),

        "end": float(
            segment.get(
                "end",
                0
            )
        ),

        "text": (
            segment.get(
                "text",
                ""
            )
        ),

        "words": (
            segment.get(
                "words"
            )
            or []
        ),

        "quality": (
            analyze_segment_quality(
                segment
            )
        ),
    }


def _add_reason_v31(
    item,
    reason,
    severity="suspect",
):
    quality = (
        item[
            "quality"
        ]
    )

    reasons = (
        quality.setdefault(
            "reasons",
            []
        )
    )

    if (
        reason
        not in reasons
    ):
        reasons.append(
            reason
        )

    quality[
        "suspect"
    ] = True

    if (
        severity
        == "critical"
    ):
        quality[
            "severity"
        ] = "critical"

    elif (
        quality.get(
            "severity",
            "ok"
        )
        == "ok"
    ):
        quality[
            "severity"
        ] = "suspect"


# ============================================================
# GUARDA AS FUNÇÕES ORIGINAIS DA V3
#
# Isso impede recursão se você executar esta célula novamente.
# ============================================================

if (
    "_RARU_V31_BASE_ANALYZE_TRANSCRIPTION"
    not in globals()
):
    _RARU_V31_BASE_ANALYZE_TRANSCRIPTION = (
        analyze_transcription_quality
    )


if (
    "_RARU_V31_BASE_VALIDATE_CANDIDATE"
    not in globals()
):
    _RARU_V31_BASE_VALIDATE_CANDIDATE = (
        validate_repair_candidate
    )


# ============================================================
# OVERRIDE 1
# QUALITY ANALYZER COM PORTUGUÊS DOMINANTE
# ============================================================

def analyze_transcription_quality(
    transcription
):
    segments = (
        transcription.get(
            "segments"
        )
        or []
    )

    # Primeiro preserva TODOS os suspects que a v3
    # já identificava normalmente.
    base_suspects = (
        _RARU_V31_BASE_ANALYZE_TRANSCRIPTION(
            transcription
        )
    )

    suspects_by_id = {
        item[
            "segment_id"
        ]:
        item

        for item
        in base_suspects
    }

    # Cache simples de idioma.
    language_hints = [
        _language_hint_v31(
            segment.get(
                "text",
                ""
            )
        )

        for segment
        in segments
    ]


    def get_item(
        index
    ):
        segment = (
            segments[
                index
            ]
        )

        segment_id = (
            segment.get(
                "id"
            )
        )

        if (
            segment_id
            not in suspects_by_id
        ):
            suspects_by_id[
                segment_id
            ] = (
                _make_item_v31(
                    segment
                )
            )

        return (
            suspects_by_id[
                segment_id
            ]
        )


    for (
        index,
        segment
    ) in enumerate(
        segments
    ):
        text = (
            segment.get(
                "text"
            )
            or ""
        ).strip()


        # ====================================================
        # REGRA A
        # QUALQUER ESCRITA NÃO LATINA
        # ====================================================

        if (
            _contains_non_latin_v31(
                text
            )
        ):
            _add_reason_v31(
                get_item(
                    index
                ),

                "unexpected_non_latin_script",

                severity="critical",
            )


        # ====================================================
        # REGRA B
        # VÁRIAS PALAVRAS INDIVIDUAIS COM CONFIANÇA
        # QUASE ZERO
        #
        # Captura, por exemplo:
        #
        # Pull it, pull it, pull it!
        #
        # Mesmo quando avg_logprob não parece tão ruim.
        # ====================================================

        (
            low_count,
            low_ratio,
        ) = (
            _very_low_words_v31(
                segment
            )
        )

        if (
            low_count
            >= LANG_V31_LOW_WORD_MIN_COUNT

            and

            low_ratio
            >= LANG_V31_LOW_WORD_RATIO
        ):
            item = (
                get_item(
                    index
                )
            )

            _add_reason_v31(
                item,

                "multiple_very_low_confidence_words",
            )

            metrics = (
                item[
                    "quality"
                ].setdefault(
                    "metrics",
                    {}
                )
            )

            metrics[
                "very_low_word_count"
            ] = (
                low_count
            )

            metrics[
                "very_low_word_ratio"
            ] = round(
                low_ratio,
                4
            )


        # ====================================================
        # REGRA C
        # INGLÊS CURTO ISOLADO EM CONTEXTO PORTUGUÊS
        #
        # IMPORTANTE:
        # NÃO eliminamos inglês.
        #
        # Apenas obrigamos aquele trecho a passar pelo
        # large-v3 contextual.
        # ====================================================

        hint = (
            language_hints[
                index
            ]
        )

        if (
            hint[
                "language"
            ]
            != "en"

            or

            len(
                hint[
                    "tokens"
                ]
            )
            >
            LANG_V31_ENGLISH_MAX_WORDS
        ):
            continue


        pt_context = False


        # ----------------------------------------------------
        # procurar português ANTES
        # ----------------------------------------------------

        for offset in range(
            1,
            LANG_V31_CONTEXT_SCAN_SEGMENTS
            + 1
        ):
            neighbor_index = (
                index
                - offset
            )

            if (
                neighbor_index
                < 0
            ):
                break

            gap = (
                float(
                    segment.get(
                        "start",
                        0
                    )
                )
                -
                float(
                    segments[
                        neighbor_index
                    ].get(
                        "end",
                        0
                    )
                )
            )

            if (
                gap
                >
                LANG_V31_CONTEXT_GAP_SECONDS
            ):
                break

            if (
                language_hints[
                    neighbor_index
                ][
                    "language"
                ]
                == "pt"
            ):
                pt_context = True

                break


        # ----------------------------------------------------
        # procurar português DEPOIS
        # ----------------------------------------------------

        if not pt_context:

            for offset in range(
                1,
                LANG_V31_CONTEXT_SCAN_SEGMENTS
                + 1
            ):
                neighbor_index = (
                    index
                    + offset
                )

                if (
                    neighbor_index
                    >= len(
                        segments
                    )
                ):
                    break

                gap = (
                    float(
                        segments[
                            neighbor_index
                        ].get(
                            "start",
                            0
                        )
                    )
                    -
                    float(
                        segment.get(
                            "end",
                            0
                        )
                    )
                )

                if (
                    gap
                    >
                    LANG_V31_CONTEXT_GAP_SECONDS
                ):
                    break

                if (
                    language_hints[
                        neighbor_index
                    ][
                        "language"
                    ]
                    == "pt"
                ):
                    pt_context = True

                    break


        if (
            pt_context
        ):
            _add_reason_v31(
                get_item(
                    index
                ),

                "isolated_english_in_pt_context",
            )


    return sorted(
        suspects_by_id.values(),

        key=lambda item: (
            item[
                "start"
            ],

            item[
                "end"
            ],
        )
    )


# ============================================================
# OVERRIDE 2
# CANDIDATO NÃO PODE INTRODUZIR SCRIPT NÃO LATINO
# ============================================================

def validate_repair_candidate(
    candidate,
    region
):
    result = (
        _RARU_V31_BASE_VALIDATE_CANDIDATE(
            candidate,
            region
        )
    )

    reasons = list(
        result.get(
            "reasons",
            []
        )
    )

    text = (
        _repair_segment_metrics(
            candidate.get(
                "segments",
                []
            )
        )[
            "text"
        ]
    )

    if (
        text

        and

        _contains_non_latin_v31(
            text
        )

        and

        "candidate_unexpected_non_latin_script"
        not in reasons
    ):
        reasons.append(
            "candidate_unexpected_non_latin_script"
        )


    result[
        "reasons"
    ] = (
        reasons
    )

    result[
        "valid"
    ] = (
        len(
            reasons
        )
        == 0
    )

    return result


# ============================================================
# OVERRIDE 3
# ESCOLHA DO CANDIDATO VENCEDOR
# ============================================================

def choose_repair_winner(
    candidates,
    region
):
    for candidate in (
        candidates
    ):
        validation = (
            validate_repair_candidate(
                candidate,
                region
            )
        )

        candidate[
            "validation"
        ] = (
            validation
        )

        candidate[
            "score"
        ] = (
            repair_quality_score(
                candidate[
                    "segments"
                ]
            )

            if (
                validation[
                    "valid"
                ]
            )

            else 0.0
        )

        candidate[
            "text"
        ] = (
            _repair_segment_metrics(
                candidate[
                    "segments"
                ]
            )[
                "text"
            ]
        )


    valid = [
        candidate

        for candidate
        in candidates

        if (
            candidate[
                "validation"
            ][
                "valid"
            ]
        )
    ]


    if not valid:
        return None


    auto = next(
        (
            candidate

            for candidate
            in valid

            if (
                candidate[
                    "mode"
                ]
                == "auto"
            )
        ),

        None
    )


    pt = next(
        (
            candidate

            for candidate
            in valid

            if (
                candidate[
                    "mode"
                ]
                == "pt"
            )
        ),

        None
    )


    en = next(
        (
            candidate

            for candidate
            in valid

            if (
                candidate[
                    "mode"
                ]
                == "en"
            )
        ),

        None
    )


    # ========================================================
    # PRIORIDADE 1 — AUTO
    #
    # Mudança importante:
    # EN NÃO rouba mais a decisão apenas porque ficou
    # 0.15 acima do AUTO.
    # ========================================================

    if (
        auto
        is not None
    ):
        auto_hint = (
            _language_hint_v31(
                auto.get(
                    "text",
                    ""
                )
            )
        )


        # Se AUTO parece inglês mas PT encontrou praticamente
        # a mesma coisa com score próximo, privilegiamos PT.
        if (
            auto_hint[
                "language"
            ]
            == "en"

            and

            pt
            is not None

            and

            pt[
                "score"
            ]
            >= LANG_V31_PT_MIN_SCORE

            and

            pt[
                "score"
            ]
            >= (
                auto[
                    "score"
                ]
                - 0.10
            )
        ):
            return pt


        return auto


    # ========================================================
    # PRIORIDADE 2 — PT
    # ========================================================

    if (
        pt
        is not None

        and

        pt[
            "score"
        ]
        >= LANG_V31_PT_MIN_SCORE
    ):
        return pt


    # ========================================================
    # PRIORIDADE 3 — EN
    #
    # Só pode vencer se houver evidência real.
    # ========================================================

    if (
        en
        is None

        or

        en[
            "score"
        ]
        < LANG_V31_EN_MIN_SCORE
    ):
        return None


    original_text = (
        region.get(
            "original_text",
            ""
        )
        or ""
    )


    # Não usamos inglês como chute para japonês/coreano/etc.
    if (
        _contains_non_latin_v31(
            original_text
        )
    ):
        return None


    original_hint = (
        _language_hint_v31(
            original_text
        )
    )


    original_is_english = (
        original_hint[
            "language"
        ]
        == "en"

        and

        original_hint[
            "en_score"
        ]
        >= 1
    )


    # Mesmo um AUTO inválido pode ter dado uma detecção
    # linguística útil.
    auto_any = next(
        (
            candidate

            for candidate
            in candidates

            if (
                candidate[
                    "mode"
                ]
                == "auto"
            )
        ),

        None
    )


    auto_detected_english = (
        auto_any
        is not None

        and

        auto_any.get(
            "language_detected"
        )
        == "en"

        and

        float(
            auto_any.get(
                "language_probability"
            )
            or 0
        )
        >= LANG_V31_EN_AUTO_PROB
    )


    if (
        original_is_english

        or

        auto_detected_english
    ):
        return en


    return None


# ============================================================
# OVERRIDE 4
# DECISÃO FINAL: REPLACE / KEEP / SUPPRESS
# ============================================================

def _decide_repair_action(
    region,
    original_score,
    winner
):
    reasons = set(
        region.get(
            "reasons",
            []
        )
    )


    script_problem = bool(
        reasons.intersection({
            "unexpected_non_latin_script",
            "unexpected_cyrillic_script",
            "unexpected_latin_script",
            "neighbor_of_unexpected_script",
        })
    )


    # ========================================================
    # SCRIPT IMPOSSÍVEL PARA O NOSSO DOMÍNIO
    #
    # Ex:
    #   ミュウアミク
    #   아아아아
    #   Продолжение следует
    #
    # Candidate decente -> replace.
    # Caso contrário -> suppress.
    # ========================================================

    if (
        script_problem
    ):
        if (
            winner
            is not None

            and

            float(
                winner.get(
                    "score",
                    0
                )
            )
            >= LANG_V31_SCRIPT_MIN_SCORE
        ):
            return "replace"

        return "suppress"


    # ========================================================
    # CRITICAL ESTRUTURAL
    #
    # Loops / compression absurda.
    #
    # Melhor apagar lixo do índice do que inventar fala.
    # ========================================================

    if (
        region.get(
            "severity"
        )
        == "critical"
    ):
        if (
            winner
            is not None

            and

            float(
                winner.get(
                    "score",
                    0
                )
            )
            >= LANG_V31_CRITICAL_MIN_SCORE
        ):
            return "replace"

        return "suppress"


    if (
        winner
        is None
    ):
        return "keep"


    winner_score = float(
        winner.get(
            "score",
            0
        )
    )


    # ========================================================
    # REGIÃO NORMAL:
    # candidate LOW não substitui mais o original.
    # ========================================================

    if (
        winner_score
        < LANG_V31_NORMAL_MIN_SCORE
    ):
        return "keep"


    gain = (
        winner_score
        - original_score
    )


    if (
        gain
        >= REPAIR_MIN_SCORE_GAIN
    ):
        return "replace"


    # ========================================================
    # INGLÊS ISOLADO
    #
    # Uma alternativa claramente PT pode vencer mesmo sem
    # ganho numérico de 0.07.
    # ========================================================

    if (
        "isolated_english_in_pt_context"
        in reasons

        and

        winner.get(
            "mode"
        )
        in {
            "auto",
            "pt",
        }
    ):
        winner_hint = (
            _language_hint_v31(
                winner.get(
                    "text",
                    ""
                )
            )
        )

        if (
            winner_hint[
                "language"
            ]
            == "pt"

            and

            winner_score
            >= LANG_V31_ISOLATED_EN_MIN_SCORE

            and

            winner_score
            >= (
                original_score
                -
                LANG_V31_ISOLATED_EN_MAX_SCORE_DROP
            )
        ):
            return "replace"


    # ========================================================
    # MANTER REGRA DE BOUNDARY JÁ VALIDADA
    # ========================================================

    boundary_problem = bool(
        reasons.intersection({
            "low_confidence_near_chunk_boundary",
            "unfinished_phrase_at_boundary",
        })
    )


    if (
        boundary_problem

        and

        winner_score
        >= REPAIR_BOUNDARY_TRUST_MIN_SCORE

        and

        winner_score
        >= (
            original_score
            -
            REPAIR_BOUNDARY_MAX_SCORE_DROP
        )
    ):
        return "replace"


    return "keep"


print("=" * 80)
print("LANGUAGE GUARD v3.1 ATIVO")
print("=" * 80)

print(
    "✓ português = idioma dominante, "
    "não obrigatório"
)

print(
    "✓ inglês curto/isolado = suspect"
)

print(
    "✓ EN não vence mais AUTO apenas por score"
)

print(
    "✓ japonês/coreano/cirílico/etc. = critical"
)

print(
    "✓ LOW não substitui região normal"
)

LANGUAGE GUARD v3.1 ATIVO
✓ português = idioma dominante, não obrigatório
✓ inglês curto/isolado = suspect
✓ EN não vence mais AUTO apenas por score
✓ japonês/coreano/cirílico/etc. = critical
✓ LOW não substitui região normal


In [18]:
# ============================================================
# CÉLULA 11E — CONTRATO FINAL DA FASE 1
# Raru Revival — Schema 1.1
# ============================================================
#
# Objetivos:
#
# 1. Formalizar timestamps absolutos.
# 2. Padronizar segmentos INITIAL / REPAIR.
# 3. Deduplicar SOMENTE duplicatas inequívocas de overlap.
# 4. Validar integridade temporal.
# 5. Garantir JSON / TXT / SRT derivados dos mesmos segments.
# 6. Preservar integralmente repair_pass / auditoria.
# 7. Produzir relatório de prontidão para a Fase 2.
#
# NÃO altera:
#
# - faster-whisper
# - large-v3-turbo
# - large-v3 repair
# - word timestamps
# - beam size
# - temperature
# - VAD
# - heurísticas do repair
# ============================================================

from pathlib import Path
from difflib import SequenceMatcher
import copy
import hashlib
import json
import re


# ============================================================
# CONTRATO PÚBLICO
# ============================================================

RARU_SCHEMA_NAME = "raru-revival-transcription"
RARU_SCHEMA_VERSION = "1.1"

TIMELINE_TOLERANCE_SECONDS = 0.080

# Deduplicação propositalmente conservadora.
OVERLAP_DUPLICATE_TIME_TOLERANCE = 0.120
OVERLAP_DUPLICATE_TEXT_SIMILARITY = 0.92
OVERLAP_DUPLICATE_MIN_IOU = 0.80


# ============================================================
# TEXTO / SIMILARIDADE
# ============================================================

def _contract_normalize_text(text):
    text = (
        text
        or ""
    ).lower()

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    # Para comparação estrutural.
    text = re.sub(
        r"[^\wÀ-ÿ]+",
        " ",
        text,
        flags=re.UNICODE,
    )

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


def _contract_text_similarity(
    a,
    b,
):
    a = _contract_normalize_text(a)
    b = _contract_normalize_text(b)

    if not a or not b:
        return 0.0

    return SequenceMatcher(
        None,
        a,
        b,
    ).ratio()


# ============================================================
# PADRONIZAÇÃO DE WORD
# ============================================================

def normalize_final_word(
    word
):
    return {
        "start": (
            float(
                word["start"]
            )
            if (
                word.get("start")
                is not None
            )
            else None
        ),

        "end": (
            float(
                word["end"]
            )
            if (
                word.get("end")
                is not None
            )
            else None
        ),

        "word": str(
            word.get(
                "word",
                ""
            )
        ),

        "probability": (
            float(
                word["probability"]
            )
            if (
                word.get("probability")
                is not None
            )
            else None
        ),
    }


# ============================================================
# PADRONIZAÇÃO DE SEGMENTO
#
# IMPORTANTE:
# preservamos quaisquer campos adicionais já existentes.
# ============================================================

def normalize_final_segment(
    segment
):
    out = copy.deepcopy(
        segment
    )

    words = [
        normalize_final_word(
            word
        )
        for word
        in (
            out.get("words")
            or []
        )
    ]

    out["start"] = float(
        out.get(
            "start",
            0.0
        )
    )

    out["end"] = float(
        out.get(
            "end",
            out["start"]
        )
    )

    out["text"] = str(
        out.get(
            "text",
            ""
        )
    )

    out["words"] = words

    # --------------------------------------------------------
    # Campo obrigatório em TODOS os segmentos.
    # --------------------------------------------------------

    out["word_alignment_missing"] = bool(
        out.get(
            "word_alignment_missing",
            len(words) == 0
        )
    )

    # --------------------------------------------------------
    # Métricas obrigatórias estruturalmente.
    #
    # None significa "não disponível",
    # não "campo inexistente".
    # --------------------------------------------------------

    for metric in (
        "avg_logprob",
        "compression_ratio",
        "no_speech_prob",
        "temperature",
    ):
        value = out.get(
            metric
        )

        out[metric] = (
            float(value)
            if value is not None
            else None
        )

    # --------------------------------------------------------
    # ORIGIN obrigatório.
    # --------------------------------------------------------

    origin = out.get(
        "origin"
    )

    if origin not in (
        "initial",
        "repair",
    ):
        origin = (
            "repair"
            if (
                out.get("repair_model")
                is not None
                or
                out.get("repair_id")
                is not None
            )
            else "initial"
        )

    out["origin"] = origin

    # --------------------------------------------------------
    # Campos de repair existem SEMPRE.
    #
    # Segmento initial:
    #
    # repair_id = None
    # repair_model = None
    # repair_confidence_class = None
    #
    # O histórico detalhado continua em repair_pass.
    # --------------------------------------------------------

    out["repair_id"] = out.get(
        "repair_id"
    )

    out["repair_model"] = out.get(
        "repair_model"
    )

    out["repair_confidence_class"] = out.get(
        "repair_confidence_class"
    )

    return out


# ============================================================
# IOU TEMPORAL
# ============================================================

def _segment_temporal_iou(
    a,
    b,
):
    intersection = max(
        0.0,

        min(
            a["end"],
            b["end"]
        )
        -
        max(
            a["start"],
            b["start"]
        )
    )

    union = max(
        a["end"],
        b["end"]
    ) - min(
        a["start"],
        b["start"]
    )

    if union <= 0:
        return 0.0

    return (
        intersection
        /
        union
    )


# ============================================================
# QUAL SEGMENTO MANTER NUMA DUPLICATA?
# ============================================================

def _duplicate_preference_score(
    segment
):
    words = (
        segment.get("words")
        or []
    )

    probabilities = [
        word["probability"]

        for word
        in words

        if (
            word.get("probability")
            is not None
        )
    ]

    mean_probability = (
        sum(probabilities)
        /
        len(probabilities)

        if probabilities

        else 0.0
    )

    return (
        1
        if (
            segment.get("origin")
            == "repair"
        )
        else 0,

        mean_probability,

        len(words),

        len(
            segment.get(
                "text",
                ""
            )
        ),
    )


# ============================================================
# DEDUPLICAÇÃO CONSERVADORA DE OVERLAP
#
# NÃO tenta juntar frases.
# NÃO faz edição.
#
# Só remove duplicatas quase inequívocas.
# ============================================================

def deduplicate_final_segments(
    segments
):
    if not segments:
        return [], []

    ordered = sorted(
        segments,
        key=lambda s: (
            float(
                s["start"]
            ),
            float(
                s["end"]
            ),
        )
    )

    result = []
    removed = []

    for current in ordered:

        if not result:
            result.append(
                current
            )
            continue

        previous = result[-1]

        similarity = (
            _contract_text_similarity(
                previous.get(
                    "text",
                    ""
                ),
                current.get(
                    "text",
                    ""
                ),
            )
        )

        iou = _segment_temporal_iou(
            previous,
            current,
        )

        nearly_same_times = (
            abs(
                previous["start"]
                -
                current["start"]
            )
            <=
            OVERLAP_DUPLICATE_TIME_TOLERANCE

            and

            abs(
                previous["end"]
                -
                current["end"]
            )
            <=
            OVERLAP_DUPLICATE_TIME_TOLERANCE
        )

        duplicate = (
            similarity
            >=
            OVERLAP_DUPLICATE_TEXT_SIMILARITY

            and

            (
                nearly_same_times

                or

                iou
                >=
                OVERLAP_DUPLICATE_MIN_IOU
            )
        )

        if not duplicate:

            result.append(
                current
            )

            continue

        # ----------------------------------------------------
        # Duplicata inequívoca.
        # Mantemos a versão considerada melhor.
        # ----------------------------------------------------

        previous_score = (
            _duplicate_preference_score(
                previous
            )
        )

        current_score = (
            _duplicate_preference_score(
                current
            )
        )

        if (
            current_score
            >
            previous_score
        ):
            kept = current
            discarded = previous

            result[-1] = (
                current
            )

        else:
            kept = previous
            discarded = current

        removed.append({
            "kept_id": (
                kept.get("id")
            ),

            "discarded_id": (
                discarded.get("id")
            ),

            "similarity": round(
                similarity,
                4
            ),

            "temporal_iou": round(
                iou,
                4
            ),

            "start": min(
                previous["start"],
                current["start"],
            ),

            "end": max(
                previous["end"],
                current["end"],
            ),
        })

    return (
        result,
        removed,
    )


# ============================================================
# VALIDAÇÃO TEMPORAL
# ============================================================

def validate_timeline(
    transcription
):
    errors = []
    warnings = []

    segments = (
        transcription.get(
            "segments"
        )
        or []
    )

    duration = float(
        transcription.get(
            "source",
            {}
        ).get(
            "duration_seconds",
            0.0
        )
        or 0.0
    )

    seen_ids = set()

    previous_segment_start = None
    previous_global_word_start = None

    duplicate_segments = 0
    duplicate_words = 0

    repair_segments = 0

    total_words = 0

    missing_word_alignments = 0


    # ========================================================
    # SEGMENTOS
    # ========================================================

    for index, segment in enumerate(
        segments
    ):
        segment_id = (
            segment.get("id")
        )

        start = float(
            segment.get(
                "start",
                0.0
            )
        )

        end = float(
            segment.get(
                "end",
                0.0
            )
        )

        text = str(
            segment.get(
                "text",
                ""
            )
        )

        words = (
            segment.get("words")
            or []
        )

        origin = segment.get(
            "origin"
        )


        # ----------------------------------------------------
        # ID
        # ----------------------------------------------------

        if segment_id is None:

            errors.append(
                f"segment[{index}]: id ausente"
            )

        elif segment_id in seen_ids:

            errors.append(
                f"segment id duplicado: "
                f"{segment_id}"
            )

        else:
            seen_ids.add(
                segment_id
            )


        # ----------------------------------------------------
        # Contrato estrutural
        # ----------------------------------------------------

        required_fields = (
            "id",
            "start",
            "end",
            "text",
            "words",
            "word_alignment_missing",
            "avg_logprob",
            "compression_ratio",
            "no_speech_prob",
            "temperature",
            "origin",
            "repair_id",
            "repair_model",
            "repair_confidence_class",
        )

        for field in (
            required_fields
        ):
            if field not in segment:

                errors.append(
                    f"segment {segment_id}: "
                    f"campo obrigatório ausente: "
                    f"{field}"
                )


        # ----------------------------------------------------
        # Range
        # ----------------------------------------------------

        if start < 0:

            errors.append(
                f"segment {segment_id}: "
                f"start negativo ({start})"
            )

        if end < start:

            errors.append(
                f"segment {segment_id}: "
                f"end < start "
                f"({start} -> {end})"
            )

        if (
            duration > 0

            and

            end
            >
            duration
            +
            TIMELINE_TOLERANCE_SECONDS
        ):

            errors.append(
                f"segment {segment_id}: "
                f"end {end} > duração "
                f"{duration}"
            )


        # ----------------------------------------------------
        # Ordem temporal
        # ----------------------------------------------------

        if (
            previous_segment_start
            is not None

            and

            start
            <
            previous_segment_start
            -
            TIMELINE_TOLERANCE_SECONDS
        ):

            errors.append(
                f"segment {segment_id}: "
                "regressão temporal de segmento"
            )

        previous_segment_start = (
            start
        )


        # ----------------------------------------------------
        # Segmento vazio
        # ----------------------------------------------------

        if (
            not text.strip()

            and

            not words
        ):

            errors.append(
                f"segment {segment_id}: "
                "sem texto e sem palavras"
            )


        # ----------------------------------------------------
        # Alignment
        # ----------------------------------------------------

        if (
            segment.get(
                "word_alignment_missing",
                False
            )
        ):

            missing_word_alignments += 1


        # ----------------------------------------------------
        # Repair
        # ----------------------------------------------------

        if (
            origin
            == "repair"
        ):

            repair_segments += 1

            # Para o contrato final:
            # segmento produzido pelo repair deve possuir
            # alinhamento por palavra.

            if not words:

                errors.append(
                    f"segment {segment_id}: "
                    "repair sem words"
                )

            if (
                segment.get(
                    "word_alignment_missing"
                )
            ):

                errors.append(
                    f"segment {segment_id}: "
                    "repair com "
                    "word_alignment_missing=True"
                )


        # ====================================================
        # WORDS
        # ====================================================

        previous_local_word_start = None

        for word_index, word in enumerate(
            words
        ):
            total_words += 1

            w_start = (
                word.get("start")
            )

            w_end = (
                word.get("end")
            )

            if (
                w_start
                is None

                or

                w_end
                is None
            ):

                errors.append(
                    f"segment {segment_id} "
                    f"word[{word_index}]: "
                    "timestamp ausente"
                )

                continue

            w_start = float(
                w_start
            )

            w_end = float(
                w_end
            )


            if w_start < 0:

                errors.append(
                    f"segment {segment_id}: "
                    f"word start negativo"
                )


            if w_end < w_start:

                errors.append(
                    f"segment {segment_id}: "
                    f"word end < start"
                )


            if (
                duration > 0

                and

                w_end
                >
                duration
                +
                TIMELINE_TOLERANCE_SECONDS
            ):

                errors.append(
                    f"segment {segment_id}: "
                    "word fora da duração da mídia"
                )


            # ------------------------------------------------
            # Palavra x segmento
            # ------------------------------------------------

            if (
                w_start
                <
                start
                -
                TIMELINE_TOLERANCE_SECONDS
            ):

                errors.append(
                    f"segment {segment_id}: "
                    "word começa antes do segmento"
                )


            if (
                w_end
                >
                end
                +
                TIMELINE_TOLERANCE_SECONDS
            ):

                errors.append(
                    f"segment {segment_id}: "
                    "word termina depois do segmento"
                )


            # ------------------------------------------------
            # Ordem interna
            # ------------------------------------------------

            if (
                previous_local_word_start
                is not None

                and

                w_start
                <
                previous_local_word_start
                -
                TIMELINE_TOLERANCE_SECONDS
            ):

                errors.append(
                    f"segment {segment_id}: "
                    "ordem das words regrediu"
                )


            previous_local_word_start = (
                w_start
            )


            # ------------------------------------------------
            # Ordem global
            #
            # Pequeno overlap é warning.
            # Regressão grande é erro.
            # ------------------------------------------------

            if (
                previous_global_word_start
                is not None

                and

                w_start
                <
                previous_global_word_start
            ):

                regression = (
                    previous_global_word_start
                    -
                    w_start
                )

                if (
                    regression
                    >
                    0.250
                ):

                    errors.append(
                        f"segment {segment_id}: "
                        "regressão global grave "
                        f"de word ({regression:.3f}s)"
                    )

                else:

                    warnings.append(
                        f"segment {segment_id}: "
                        "pequeno overlap de words "
                        f"({regression:.3f}s)"
                    )


            previous_global_word_start = max(
                previous_global_word_start
                if (
                    previous_global_word_start
                    is not None
                )
                else w_start,

                w_start,
            )


        # ----------------------------------------------------
        # Coerência TEXT x WORDS
        # ----------------------------------------------------

        if (
            text.strip()

            and

            words
        ):
            words_text = "".join(
                str(
                    word.get(
                        "word",
                        ""
                    )
                )
                for word
                in words
            ).strip()

            similarity = (
                _contract_text_similarity(
                    text,
                    words_text,
                )
            )

            if similarity < 0.35:

                errors.append(
                    f"segment {segment_id}: "
                    "texto e words divergem fortemente "
                    f"(similaridade={similarity:.3f})"
                )

            elif similarity < 0.70:

                warnings.append(
                    f"segment {segment_id}: "
                    "texto e words divergem "
                    f"(similaridade={similarity:.3f})"
                )


    # ========================================================
    # DUPLICATAS ENTRE SEGMENTOS
    # ========================================================

    for index in range(
        1,
        len(segments)
    ):
        previous = (
            segments[
                index - 1
            ]
        )

        current = (
            segments[
                index
            ]
        )

        similarity = (
            _contract_text_similarity(
                previous.get(
                    "text",
                    ""
                ),
                current.get(
                    "text",
                    ""
                ),
            )
        )

        iou = (
            _segment_temporal_iou(
                previous,
                current,
            )
        )

        if (
            similarity
            >=
            OVERLAP_DUPLICATE_TEXT_SIMILARITY

            and

            iou
            >=
            OVERLAP_DUPLICATE_MIN_IOU
        ):
            duplicate_segments += 1

            errors.append(
                "possível segmento duplicado por overlap: "
                f"{previous.get('id')} / "
                f"{current.get('id')}"
            )


        # ----------------------------------------------------
        # Word duplicada praticamente no mesmo instante.
        # ----------------------------------------------------

        previous_words = (
            previous.get(
                "words"
            )
            or []
        )

        current_words = (
            current.get(
                "words"
            )
            or []
        )

        if (
            previous_words

            and

            current_words
        ):
            tail = previous_words[
                -4:
            ]

            head = current_words[
                :4
            ]

            for a in tail:

                for b in head:

                    if (
                        _contract_normalize_text(
                            a.get(
                                "word",
                                ""
                            )
                        )
                        !=
                        _contract_normalize_text(
                            b.get(
                                "word",
                                ""
                            )
                        )
                    ):
                        continue

                    if (
                        abs(
                            float(
                                a["start"]
                            )
                            -
                            float(
                                b["start"]
                            )
                        )
                        <= 0.050

                        and

                        abs(
                            float(
                                a["end"]
                            )
                            -
                            float(
                                b["end"]
                            )
                        )
                        <= 0.050
                    ):
                        duplicate_words += 1

                        errors.append(
                            "possível word duplicada por overlap "
                            f"entre segments "
                            f"{previous.get('id')} e "
                            f"{current.get('id')}: "
                            f"{a.get('word')!r}"
                        )


    return {
        "valid": (
            len(errors)
            == 0
        ),

        "errors": (
            errors
        ),

        "warnings": (
            warnings
        ),

        "metrics": {
            "segments": len(
                segments
            ),

            "words": (
                total_words
            ),

            "repair_segments": (
                repair_segments
            ),

            "missing_word_alignments": (
                missing_word_alignments
            ),

            "duplicate_segments": (
                duplicate_segments
            ),

            "duplicate_words": (
                duplicate_words
            ),
        },
    }


# ============================================================
# OUTPUT TXT / SRT
#
# A PARTIR DE AGORA:
#
# FINAL SEGMENTS
#       |
#       +--> JSON
#       +--> TXT
#       +--> SRT
#
# Não existem mais caminhos separados.
# ============================================================

def _phase1_srt_timestamp(
    seconds
):
    total_ms = int(
        round(
            float(seconds)
            * 1000
        )
    )

    hours = (
        total_ms
        //
        3_600_000
    )

    total_ms %= (
        3_600_000
    )

    minutes = (
        total_ms
        //
        60_000
    )

    total_ms %= (
        60_000
    )

    secs = (
        total_ms
        //
        1000
    )

    millis = (
        total_ms
        %
        1000
    )

    return (
        f"{hours:02d}:"
        f"{minutes:02d}:"
        f"{secs:02d},"
        f"{millis:03d}"
    )


def render_final_txt(
    segments
):
    lines = []

    for segment in (
        segments
    ):
        lines.append(
            "["
            f"{float(segment['start']):.3f}"
            " --> "
            f"{float(segment['end']):.3f}"
            "]"
        )

        lines.append(
            str(
                segment.get(
                    "text",
                    ""
                )
            ).strip()
        )

    return (
        "\n".join(
            lines
        )
        +
        (
            "\n"
            if lines
            else ""
        )
    )


def render_final_srt(
    segments
):
    blocks = []

    for index, segment in enumerate(
        segments,
        start=1,
    ):
        blocks.append(
            "\n".join([
                str(index),

                (
                    _phase1_srt_timestamp(
                        segment["start"]
                    )
                    +
                    " --> "
                    +
                    _phase1_srt_timestamp(
                        segment["end"]
                    )
                ),

                str(
                    segment.get(
                        "text",
                        ""
                    )
                ).strip(),
            ])
        )

    return (
        "\n\n".join(
            blocks
        )
        +
        (
            "\n"
            if blocks
            else ""
        )
    )


# ============================================================
# HASH DOS SEGMENTOS
# ============================================================

def final_segments_sha256(
    segments
):
    canonical = json.dumps(
        segments,
        ensure_ascii=False,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    )

    return hashlib.sha256(
        canonical.encode(
            "utf-8"
        )
    ).hexdigest()


# ============================================================
# CAMINHOS RAW / REPAIRED
# ============================================================

def phase1_output_paths(
    source_path,
    test_mode,
    repaired=False,
):
    paths = job_paths(
        source_path,
        test_mode=test_mode,
    )

    if not repaired:

        return {
            "json": (
                paths[
                    "final_json"
                ]
            ),

            "txt": (
                paths[
                    "txt"
                ]
            ),

            "srt": (
                paths[
                    "srt"
                ]
            ),
        }

    return {
        "json": (
            paths[
                "final_json"
            ].with_name(
                paths[
                    "final_json"
                ].stem
                +
                ".REPAIRED.json"
            )
        ),

        "txt": (
            paths[
                "final_json"
            ].with_name(
                paths[
                    "final_json"
                ].stem
                +
                ".REPAIRED.txt"
            )
        ),

        "srt": (
            paths[
                "final_json"
            ].with_name(
                paths[
                    "final_json"
                ].stem
                +
                ".REPAIRED.srt"
            )
        ),
    }


# ============================================================
# APLICAR CONTRATO 1.1
# ============================================================

def apply_phase1_contract(
    transcription,
    source_path,
    test_mode=False,
    repaired=False,
    initial_segment_count=None,
):
    if transcription is None:
        return None

    data = copy.deepcopy(
        transcription
    )

    # ========================================================
    # SCHEMA
    # ========================================================

    data["schema"] = (
        RARU_SCHEMA_NAME
    )

    data["schema_version"] = (
        RARU_SCHEMA_VERSION
    )

    # ========================================================
    # CONTRATO TEMPORAL EXPLÍCITO
    # ========================================================

    data["timeline"] = {
        "reference": (
            "source_media"
        ),

        "timestamp_type": (
            "absolute"
        ),

        "unit": (
            "seconds"
        ),

        "zero_point": (
            "start_of_source_media"
        ),

        "chunk_offsets_applied": True,

        "silence_removed_from_timeline": False,

        "editorial_padding_applied": False,
    }

    # ========================================================
    # NORMALIZAR SEGMENTOS
    # ========================================================

    original_segments = (
        data.get(
            "segments"
        )
        or []
    )

    normalized = [
        normalize_final_segment(
            segment
        )
        for segment
        in original_segments
    ]

    # --------------------------------------------------------
    # IDs:
    #
    # só criamos IDs quando estiverem ausentes.
    #
    # IDs existentes NÃO são renumerados porque podem ser
    # referenciados pela auditoria.
    # --------------------------------------------------------

    existing_ids = {
        segment["id"]

        for segment
        in normalized

        if (
            segment.get("id")
            is not None
        )
    }

    next_id = (
        max(
            existing_ids
        )
        + 1

        if existing_ids

        else 1
    )

    for segment in normalized:

        if (
            segment.get("id")
            is None
        ):
            while (
                next_id
                in existing_ids
            ):
                next_id += 1

            segment["id"] = (
                next_id
            )

            existing_ids.add(
                next_id
            )

            next_id += 1

    # ========================================================
    # DEDUP DE OVERLAP
    # ========================================================

    normalized, removed_duplicates = (
        deduplicate_final_segments(
            normalized
        )
    )

    data["segments"] = (
        normalized
    )

    data["overlap_deduplication"] = {
        "strategy": (
            "conservative_temporal_and_text"
        ),

        "duplicates_removed": len(
            removed_duplicates
        ),

        "removed": (
            removed_duplicates
        ),
    }

    # ========================================================
    # TIMELINE VALIDATION
    # ========================================================

    timeline_validation = (
        validate_timeline(
            data
        )
    )

    data[
        "timeline_validation"
    ] = (
        timeline_validation
    )

    # ========================================================
    # HASH DOS FINAL SEGMENTS
    # ========================================================

    segments_hash = (
        final_segments_sha256(
            data[
                "segments"
            ]
        )
    )

    data[
        "final_segments_sha256"
    ] = (
        segments_hash
    )

    # ========================================================
    # OUTPUTS DERIVADOS EXCLUSIVAMENTE DOS FINAL SEGMENTS
    # ========================================================

    outputs = (
        phase1_output_paths(
            source_path,
            test_mode,
            repaired=repaired,
        )
    )

    txt_content = (
        render_final_txt(
            data[
                "segments"
            ]
        )
    )

    srt_content = (
        render_final_srt(
            data[
                "segments"
            ]
        )
    )

    outputs[
        "txt"
    ].write_text(
        txt_content,
        encoding="utf-8",
    )

    outputs[
        "srt"
    ].write_text(
        srt_content,
        encoding="utf-8",
    )

    # ========================================================
    # CONSISTÊNCIA
    # ========================================================

    txt_valid = (
        outputs[
            "txt"
        ].read_text(
            encoding="utf-8"
        )
        ==
        txt_content
    )

    srt_valid = (
        outputs[
            "srt"
        ].read_text(
            encoding="utf-8"
        )
        ==
        srt_content
    )

    data[
        "output_validation"
    ] = {
        "source_of_truth": (
            "segments"
        ),

        "segments_sha256": (
            segments_hash
        ),

        "txt_derived_from_final_segments": (
            txt_valid
        ),

        "srt_derived_from_final_segments": (
            srt_valid
        ),

        "valid": (
            txt_valid
            and
            srt_valid
        ),
    }

    # ========================================================
    # SUMMARY
    # ========================================================

    repair_pass = (
        data.get(
            "repair_pass"
        )
        or {}
    )

    data[
        "phase1_summary"
    ] = {
        "media_duration_seconds": (
            data.get(
                "source",
                {}
            ).get(
                "duration_seconds"
            )
        ),

        "chunks_processed": (
            data.get(
                "checkpointing",
                {}
            ).get(
                "chunks_completed"
            )
        ),

        "initial_segments": (
            initial_segment_count
            if (
                initial_segment_count
                is not None
            )
            else (
                len(
                    data[
                        "segments"
                    ]
                )
                if not repaired
                else None
            )
        ),

        "suspect_segments": (
            repair_pass.get(
                "suspect_segments",
                0
            )
        ),

        "regions_repaired": (
            repair_pass.get(
                "regions_replaced",
                0
            )
        ),

        "segments_final": len(
            data[
                "segments"
            ]
        ),

        "duplicates_removed": len(
            removed_duplicates
        ),

        "timeline_valid": (
            timeline_validation[
                "valid"
            ]
        ),

        "outputs_valid": (
            data[
                "output_validation"
            ][
                "valid"
            ]
        ),

        "absolute_timestamps": True,

        "errors": len(
            timeline_validation[
                "errors"
            ]
        ),

        "warnings": len(
            timeline_validation[
                "warnings"
            ]
        ),
    }

    # ========================================================
    # NÃO LIBERAR JSON QUEBRADO PARA FASE 2
    # ========================================================

    contract_valid = (
        timeline_validation[
            "valid"
        ]

        and

        data[
            "output_validation"
        ][
            "valid"
        ]
    )

    data[
        "phase1_ready"
    ] = bool(
        contract_valid
    )

    # --------------------------------------------------------
    # Gravamos o estado para diagnóstico.
    # --------------------------------------------------------

    if not contract_valid:

        data[
            "status"
        ] = (
            "validation_failed"
        )

    atomic_json_dump(
        data,
        outputs[
            "json"
        ]
    )

    if not contract_valid:

        raise RuntimeError(
            "FASE 1 REPROVADA NA VALIDAÇÃO.\n\n"
            +
            "\n".join(
                timeline_validation[
                    "errors"
                ][
                    :20
                ]
            )
        )

    return data


# ============================================================
# RELATÓRIO
# ============================================================

def print_phase1_report(
    data
):
    if data is None:
        return

    summary = (
        data.get(
            "phase1_summary"
        )
        or {}
    )

    validation = (
        data.get(
            "timeline_validation"
        )
        or {}
    )

    print()
    print("=" * 80)
    print("RELATÓRIO FINAL — FASE 1")
    print("=" * 80)

    print(
        "Schema:",
        data.get(
            "schema_version"
        )
    )

    print(
        "Duração da mídia:",
        summary.get(
            "media_duration_seconds"
        )
    )

    print(
        "Chunks processados:",
        summary.get(
            "chunks_processed"
        )
    )

    print(
        "Segmentos iniciais:",
        summary.get(
            "initial_segments"
        )
    )

    print(
        "Segmentos suspeitos:",
        summary.get(
            "suspect_segments"
        )
    )

    print(
        "Regiões reparadas:",
        summary.get(
            "regions_repaired"
        )
    )

    print(
        "Segmentos finais:",
        summary.get(
            "segments_final"
        )
    )

    print(
        "Duplicatas removidas:",
        summary.get(
            "duplicates_removed"
        )
    )

    print()

    print(
        "Timeline:",
        (
            "VALID"
            if (
                summary.get(
                    "timeline_valid"
                )
            )
            else "INVALID"
        )
    )

    print(
        "JSON ↔ SRT:",
        (
            "VALID"
            if (
                data.get(
                    "output_validation",
                    {}
                ).get(
                    "srt_derived_from_final_segments"
                )
            )
            else "INVALID"
        )
    )

    print(
        "JSON ↔ TXT:",
        (
            "VALID"
            if (
                data.get(
                    "output_validation",
                    {}
                ).get(
                    "txt_derived_from_final_segments"
                )
            )
            else "INVALID"
        )
    )

    print(
        "Timestamps absolutos:",
        (
            "VALID"
            if (
                data.get(
                    "timeline",
                    {}
                ).get(
                    "timestamp_type"
                )
                ==
                "absolute"
            )
            else "INVALID"
        )
    )

    metrics = (
        validation.get(
            "metrics",
            {}
        )
    )

    print(
        "Duplicatas restantes:",
        (
            metrics.get(
                "duplicate_segments",
                0
            )
            +
            metrics.get(
                "duplicate_words",
                0
            )
        )
    )

    print(
        "Erros:",
        len(
            validation.get(
                "errors",
                []
            )
        )
    )

    print(
        "Warnings:",
        len(
            validation.get(
                "warnings",
                []
            )
        )
    )

    print()

    print(
        "FASE 1:",
        (
            "READY FOR PHASE 2"
            if (
                data.get(
                    "phase1_ready"
                )
            )
            else "NOT READY"
        )
    )


# ============================================================
# WRAPPERS
#
# Não reescrevemos finalize_live / run_repair_pass /
# process_live.
#
# Encapsulamos somente a saída.
# ============================================================


# ------------------------------------------------------------
# Guardar implementações originais uma única vez.
# ------------------------------------------------------------

if (
    "_RARU_PHASE1_BASE_FINALIZE_LIVE"
    not in globals()
):
    _RARU_PHASE1_BASE_FINALIZE_LIVE = (
        finalize_live
    )


if (
    "_RARU_PHASE1_BASE_RUN_REPAIR_PASS"
    not in globals()
):
    _RARU_PHASE1_BASE_RUN_REPAIR_PASS = (
        run_repair_pass
    )


if (
    "_RARU_PHASE1_BASE_PROCESS_LIVE"
    not in globals()
):
    _RARU_PHASE1_BASE_PROCESS_LIVE = (
        process_live
    )


# ============================================================
# FINALIZE RAW
# ============================================================

def finalize_live(
    source_path,
    total_elapsed,
    prep_mode,
    test_mode,
):
    result = (
        _RARU_PHASE1_BASE_FINALIZE_LIVE(
            source_path,
            total_elapsed,
            prep_mode,
            test_mode,
        )
    )

    if result is None:
        return None

    result = (
        apply_phase1_contract(
            result,
            source_path=source_path,
            test_mode=test_mode,
            repaired=False,
            initial_segment_count=len(
                result.get(
                    "segments",
                    []
                )
            ),
        )
    )

    return result


# ============================================================
# FINALIZE REPAIR
# ============================================================

def run_repair_pass(
    source_path,
    transcription,
    test_mode=False,
):
    initial_count = len(
        transcription.get(
            "segments",
            []
        )
    )

    result = (
        _RARU_PHASE1_BASE_RUN_REPAIR_PASS(
            source_path=source_path,
            transcription=transcription,
            test_mode=test_mode,
        )
    )

    if result is None:
        return None

    result = (
        apply_phase1_contract(
            result,
            source_path=source_path,
            test_mode=test_mode,
            repaired=True,
            initial_segment_count=(
                initial_count
            ),
        )
    )

    print_phase1_report(
        result
    )

    return result


# ============================================================
# PROCESS LIVE
#
# Serve também para outputs antigos que forem carregados
# por SKIP e ainda estiverem em schema 1.0.
# ============================================================

def process_live(
    source_path,
    force=False,
    test_mode=TEST_MODE,
    repair=ENABLE_REPAIR_PASS,
):
    result = (
        _RARU_PHASE1_BASE_PROCESS_LIVE(
            source_path,
            force=force,
            test_mode=test_mode,
            repair=repair,
        )
    )

    if result is None:
        return None

    # Já passou pelo contrato durante esta execução.
    if (
        result.get(
            "schema_version"
        )
        ==
        RARU_SCHEMA_VERSION

        and

        "timeline_validation"
        in result

        and

        "output_validation"
        in result
    ):
        return result

    repaired = bool(
        result.get(
            "repair_pass"
        )
    ) or str(
        result.get(
            "status",
            ""
        )
    ).endswith(
        "repaired"
    )

    initial_count = None

    if repaired:
        raw_paths = (
            phase1_output_paths(
                source_path,
                test_mode,
                repaired=False,
            )
        )

        if (
            raw_paths[
                "json"
            ].exists()
        ):
            raw = load_json(
                raw_paths[
                    "json"
                ],
                {},
            )

            initial_count = len(
                raw.get(
                    "segments",
                    []
                )
            )

    result = (
        apply_phase1_contract(
            result,
            source_path=source_path,
            test_mode=test_mode,
            repaired=repaired,
            initial_segment_count=(
                initial_count
            ),
        )
    )

    print_phase1_report(
        result
    )

    return result


print("=" * 80)
print("RARU REVIVAL — PHASE 1 CONTRACT")
print("=" * 80)
print("Schema: 1.1")
print("✓ timeline absoluta formalizada")
print("✓ schema de segmentos padronizado")
print("✓ origin obrigatório")
print("✓ repair audit preservado")
print("✓ deduplicação conservadora de overlap")
print("✓ validação temporal ativa")
print("✓ JSON/TXT/SRT usam os mesmos final segments")

RARU REVIVAL — PHASE 1 CONTRACT
Schema: 1.1
✓ timeline absoluta formalizada
✓ schema de segmentos padronizado
✓ origin obrigatório
✓ repair audit preservado
✓ deduplicação conservadora de overlap
✓ validação temporal ativa
✓ JSON/TXT/SRT usam os mesmos final segments


In [24]:
# ============================================================
# CÉLULA 12 — EXECUÇÃO + TESTE FINAL DE ACEITAÇÃO DA FASE 1
#
# Raru Revival
#
# Esta célula substitui a antiga 12 + 12A.
#
# RUN_PHASE1_ACCEPTANCE = True
#
#   Executa:
#
#   1. baseline RAW single-chunk;
#   2. transcrição multi-chunk;
#   3. overlap;
#   4. repair pass;
#   5. validação absoluta do repair;
#   6. comparação single-chunk x multi-chunk nas bordas;
#   7. validação JSON/SRT/TXT;
#   8. relatório A-H.
#
#
# RUN_PHASE1_ACCEPTANCE = False
#
#   Executa somente o fluxo normal:
#
#       process_live(...)
#
# Depois que a Fase 1 estiver definitivamente aprovada,
# pode deixar False para uso cotidiano.
# ============================================================

from pathlib import Path
from collections import defaultdict
import copy
import math


# ============================================================
# CONFIGURAÇÃO
# ============================================================

TEST_MODE = True

RUN_PHASE1_ACCEPTANCE = True


# ------------------------------------------------------------
# Teste multi-chunk
# ------------------------------------------------------------

MULTICHUNK_TEST_CHUNK_SECONDS = 60

MULTICHUNK_TEST_OVERLAP_SECONDS = 2.0

MULTICHUNK_BOUNDARY_WINDOW = 3.0


# ------------------------------------------------------------
# Tolerâncias exclusivas dos testes
# ------------------------------------------------------------

ACCEPTANCE_WORD_TIME_TOLERANCE = 0.100

ACCEPTANCE_BASELINE_MIDPOINT_TOLERANCE = 0.300

ACCEPTANCE_BASELINE_TEXT_TOLERANCE = 0.750

ACCEPTANCE_REPAIR_CORE_TOLERANCE = 0.150

# ------------------------------------------------------------
# Detecção REAL de buraco temporal
#
# Diferença de palavra/tokenização entre single e multi
# NÃO é suficiente para reprovar.
#
# Só consideramos buraco quando a região de fala do baseline:
#
# - possui múltiplas palavras consecutivas;
# - tem duração relevante;
# - praticamente não possui cobertura por segmentos multi;
# - não possui words multi próximas.
# ------------------------------------------------------------

ACCEPTANCE_SPEECH_COVERAGE_SLACK = 0.450

ACCEPTANCE_HOLE_MAX_MULTI_COVERAGE = 0.20

ACCEPTANCE_HOLE_NEARBY_WORD_TOLERANCE = 0.550

ACCEPTANCE_SIGNIFICANT_HOLE_MIN_WORDS = 2

ACCEPTANCE_SIGNIFICANT_HOLE_MIN_DURATION = 0.350

ACCEPTANCE_UNMATCHED_GROUP_GAP = 0.450


# ============================================================
# ENCONTRAR LIVES
# ============================================================

LIVES = sorted([
    p

    for p
    in DIRS["lives"].iterdir()

    if (
        p.suffix.lower()
        in VIDEO_EXTS
    )
])


print(
    "Lives encontradas:"
)

for i, p in enumerate(
    LIVES
):
    print(
        f"[{i}] {p.name}"
    )


if not LIVES:

    raise RuntimeError(
        "Nenhuma live encontrada."
    )


TEST_SOURCE = (
    LIVES[0]
)


# ============================================================
# MODO NORMAL
# ============================================================

if not RUN_PHASE1_ACCEPTANCE:

    result = process_live(
        TEST_SOURCE,

        force=True,

        test_mode=TEST_MODE,

        repair=True,
    )


# ============================================================
# TESTE FINAL DA FASE 1
# ============================================================

else:

    # ========================================================
    # GARANTIR QUE A 11E FOI EXECUTADA
    # ========================================================

    required_phase1_symbols = (
        "_contract_normalize_text",
        "render_final_txt",
        "render_final_srt",
        "final_segments_sha256",
        "validate_timeline",
        "RARU_SCHEMA_VERSION",
    )

    missing_symbols = [
        name

        for name
        in required_phase1_symbols

        if (
            name
            not in globals()
        )
    ]

    if missing_symbols:

        raise RuntimeError(
            "Execute a CÉLULA 11E antes da CÉLULA 12.\n"
            "Símbolos ausentes: "
            +
            ", ".join(
                missing_symbols
            )
        )


    # ========================================================
    # HELPERS
    # ========================================================

    def _acceptance_words(
        segments
    ):
        result = []

        for segment in (
            segments
            or []
        ):

            segment_id = (
                segment.get("id")
            )

            repair_id = (
                segment.get(
                    "repair_id"
                )
            )

            origin = (
                segment.get(
                    "origin",
                    "initial"
                )
            )


            for word in (
                segment.get(
                    "words"
                )
                or []
            ):

                if (
                    word.get("start")
                    is None

                    or

                    word.get("end")
                    is None
                ):
                    continue


                start = float(
                    word["start"]
                )

                end = float(
                    word["end"]
                )


                result.append({
                    "word": (
                        _contract_normalize_text(
                            word.get(
                                "word",
                                ""
                            )
                        )
                    ),

                    "raw_word": str(
                        word.get(
                            "word",
                            ""
                        )
                    ),

                    "start": (
                        start
                    ),

                    "end": (
                        end
                    ),

                    "midpoint": (
                        (
                            start
                            +
                            end
                        )
                        /
                        2.0
                    ),

                    "segment_id": (
                        segment_id
                    ),

                    "origin": (
                        origin
                    ),

                    "repair_id": (
                        repair_id
                    ),
                })

        return result


    # ========================================================
    # PALAVRA EXISTE NO RESULTADO?
    # ========================================================

    def _acceptance_word_exists(
        target,
        final_words,
        tolerance=(
            ACCEPTANCE_WORD_TIME_TOLERANCE
        ),
    ):

        for candidate in (
            final_words
        ):

            if (
                candidate[
                    "word"
                ]
                !=
                target[
                    "word"
                ]
            ):
                continue


            if (
                abs(
                    candidate[
                        "start"
                    ]
                    -
                    target[
                        "start"
                    ]
                )
                <=
                tolerance

                and

                abs(
                    candidate[
                        "end"
                    ]
                    -
                    target[
                        "end"
                    ]
                )
                <=
                tolerance
            ):

                return True


        return False


    # ========================================================
    # EXISTE FALA NO MESMO PONTO TEMPORAL?
    #
    # Usado na comparação:
    #
    # single-chunk RAW
    #        x
    # multi-chunk RAW
    #
    # O texto pode mudar levemente porque o contexto do
    # Whisper muda.
    #
    # Portanto aceitamos:
    #
    # 1. mesma palavra próxima;
    # OU
    # 2. alguma palavra ocupando praticamente o mesmo tempo.
    # ========================================================

    def _baseline_word_has_support(
        baseline_word,
        multi_words,
    ):

        baseline_midpoint = (
            baseline_word[
                "midpoint"
            ]
        )


        for candidate in (
            multi_words
        ):

            candidate_midpoint = (
                candidate[
                    "midpoint"
                ]
            )


            # ------------------------------------------------
            # Mesma palavra com tolerância temporal maior.
            # ------------------------------------------------

            if (
                baseline_word[
                    "word"
                ]

                and

                baseline_word[
                    "word"
                ]
                ==
                candidate[
                    "word"
                ]

                and

                abs(
                    candidate_midpoint
                    -
                    baseline_midpoint
                )
                <=
                ACCEPTANCE_BASELINE_TEXT_TOLERANCE
            ):

                return True


            # ------------------------------------------------
            # Evidência temporal independente do texto.
            #
            # Útil quando:
            #
            # baseline: "porta"
            # multi   : "a porta"
            #
            # ou Whisper segmentou as palavras diferente.
            # ------------------------------------------------

            if (
                abs(
                    candidate_midpoint
                    -
                    baseline_midpoint
                )
                <=
                ACCEPTANCE_BASELINE_MIDPOINT_TOLERANCE
            ):

                return True


            # ------------------------------------------------
            # Overlap real entre intervalos.
            # ------------------------------------------------

            intersection = max(
                0.0,

                min(
                    baseline_word[
                        "end"
                    ],
                    candidate[
                        "end"
                    ],
                )
                -
                max(
                    baseline_word[
                        "start"
                    ],
                    candidate[
                        "start"
                    ],
                )
            )


            if (
                intersection
                >
                0.0
            ):

                return True


        return False


    # ========================================================
    # AGRUPAR PALAVRAS BASELINE NÃO ENCONTRADAS
    #
    # Uma única palavra diferente não significa necessariamente
    # que houve buraco.
    #
    # Duas ou mais palavras consecutivas sem nenhuma evidência
    # temporal podem indicar uma fala perdida na fronteira.
    # ========================================================

    def _group_unmatched_words(
        words
    ):

        if not words:

            return []


        ordered = sorted(
            words,
            key=lambda item: (
                item[
                    "start"
                ]
            ),
        )


        groups = [
            [
                ordered[0]
            ]
        ]


        for word in (
            ordered[1:]
        ):

            previous = (
                groups[
                    -1
                ][
                    -1
                ]
            )


            gap = (
                word[
                    "start"
                ]
                -
                previous[
                    "end"
                ]
            )


            if (
                gap
                <=
                ACCEPTANCE_UNMATCHED_GROUP_GAP
            ):

                groups[
                    -1
                ].append(
                    word
                )

            else:

                groups.append(
                    [
                        word
                    ]
                )


        return groups


    # ========================================================
    # VALIDAÇÃO DOS CHECKPOINTS MULTI-CHUNK
    # ========================================================

    def validate_multichunk_checkpoints(
        source_path,
        test_mode=True,
    ):

        paths = job_paths(
            source_path,
            test_mode=test_mode,
        )


        chunk_dir = (
            paths[
                "chunks_dir"
            ]
        )


        chunk_files = sorted(
            chunk_dir.glob(
                "chunk_*.json"
            )
        )


        errors = []
        warnings = []


        if (
            len(
                chunk_files
            )
            <
            2
        ):

            errors.append(
                "Teste não produziu múltiplos chunks."
            )

            return {
                "valid": False,

                "errors": (
                    errors
                ),

                "warnings": (
                    warnings
                ),

                "chunks": len(
                    chunk_files
                ),

                "boundaries": [],

                "checkpoint_words": 0,

                "boundary_words_checked": 0,

                "missing_after_merge": 0,

                "boundary_words_missing": 0,
            }


        # ====================================================
        # RAW CONSOLIDADO
        # ====================================================

        raw = load_json(
            paths[
                "final_json"
            ],
            {},
        )


        final_words = (
            _acceptance_words(
                raw.get(
                    "segments",
                    []
                )
            )
        )


        checkpoint_words = []

        boundaries = []

        previous_core_end = None


        # ====================================================
        # ANALISAR TODOS OS CHUNKS
        # ====================================================

        for chunk_index, chunk_file in enumerate(
            chunk_files
        ):

            chunk = load_json(
                chunk_file,
                {},
            )


            core_start = float(
                chunk[
                    "core_start"
                ]
            )

            core_end = float(
                chunk[
                    "core_end"
                ]
            )


            # ------------------------------------------------
            # Continuidade dos cores
            # ------------------------------------------------

            if (
                previous_core_end
                is not None

                and

                abs(
                    core_start
                    -
                    previous_core_end
                )
                >
                0.001
            ):

                errors.append(
                    f"Gap entre cores: "
                    f"{previous_core_end:.3f} -> "
                    f"{core_start:.3f}"
                )


            if (
                chunk_index
                >
                0
            ):

                boundaries.append(
                    core_start
                )


            previous_core_end = (
                core_end
            )


            # =================================================
            # SEGMENTOS DO CHUNK
            # =================================================

            for segment in (
                chunk.get(
                    "segments",
                    []
                )
            ):

                start = float(
                    segment[
                        "start"
                    ]
                )

                end = float(
                    segment[
                        "end"
                    ]
                )


                # --------------------------------------------
                # Novo chunk NÃO pode voltar para zero.
                # --------------------------------------------

                if (
                    chunk_index > 0

                    and

                    start
                    <
                    core_start
                    -
                    0.250
                ):

                    errors.append(
                        f"Chunk {chunk_index}: "
                        "timestamp aparentemente relativo "
                        "ao chunk. "
                        f"segment.start={start:.3f}, "
                        f"core_start={core_start:.3f}"
                    )


                if (
                    end
                    >
                    core_end
                    +
                    0.250
                ):

                    warnings.append(
                        f"Chunk {chunk_index}: "
                        "segmento ultrapassa "
                        "ligeiramente o core."
                    )


                # ============================================
                # WORDS
                # ============================================

                for word in (
                    segment.get(
                        "words"
                    )
                    or []
                ):

                    if (
                        word.get("start")
                        is None

                        or

                        word.get("end")
                        is None
                    ):
                        continue


                    w_start = float(
                        word[
                            "start"
                        ]
                    )

                    w_end = float(
                        word[
                            "end"
                        ]
                    )


                    midpoint = (
                        (
                            w_start
                            +
                            w_end
                        )
                        /
                        2.0
                    )


                    # ----------------------------------------
                    # A palavra oficial precisa pertencer
                    # ao core do chunk.
                    # ----------------------------------------

                    if not (
                        core_start
                        -
                        0.100

                        <=

                        midpoint

                        <=

                        core_end
                        +
                        0.100
                    ):

                        errors.append(
                            f"Chunk {chunk_index}: "
                            "word fora do core oficial: "
                            f"{word.get('word')!r} "
                            f"@ {w_start:.3f}"
                        )


                    checkpoint_words.append({
                        "word": (
                            _contract_normalize_text(
                                word.get(
                                    "word",
                                    ""
                                )
                            )
                        ),

                        "raw_word": str(
                            word.get(
                                "word",
                                ""
                            )
                        ),

                        "start": (
                            w_start
                        ),

                        "end": (
                            w_end
                        ),

                        "midpoint": (
                            midpoint
                        ),
                    })


        # ====================================================
        # CHECKPOINT -> RAW FINAL
        #
        # Nenhuma palavra oficial do chunk pode sumir
        # durante o merge.
        # ====================================================

        missing_after_merge = []


        for target in (
            checkpoint_words
        ):

            if not (
                target[
                    "word"
                ]
            ):
                continue


            if not (
                _acceptance_word_exists(
                    target,
                    final_words,
                )
            ):

                missing_after_merge.append(
                    target
                )


        if (
            missing_after_merge
        ):

            errors.append(
                f"{len(missing_after_merge)} word(s) "
                "presentes nos chunks desapareceram "
                "durante o merge."
            )


        # ====================================================
        # VALIDAÇÃO ESPECÍFICA DAS BORDAS
        # ====================================================

        boundary_word_count = 0

        boundary_missing = []


        for boundary in (
            boundaries
        ):

            near_boundary = [
                word

                for word
                in checkpoint_words

                if (
                    abs(
                        word[
                            "midpoint"
                        ]
                        -
                        boundary
                    )
                    <=
                    MULTICHUNK_BOUNDARY_WINDOW
                )
            ]


            boundary_word_count += len(
                near_boundary
            )


            for word in (
                near_boundary
            ):

                if not (
                    _acceptance_word_exists(
                        word,
                        final_words,
                    )
                ):

                    boundary_missing.append({
                        "boundary": (
                            boundary
                        ),

                        "word": (
                            word
                        ),
                    })


        if (
            boundary_missing
        ):

            errors.append(
                f"{len(boundary_missing)} word(s) "
                "próximas às fronteiras desapareceram."
            )


        return {
            "valid": (
                len(
                    errors
                )
                ==
                0
            ),

            "errors": (
                errors
            ),

            "warnings": (
                warnings
            ),

            "chunks": len(
                chunk_files
            ),

            "boundaries": (
                boundaries
            ),

            "checkpoint_words": len(
                checkpoint_words
            ),

            "boundary_words_checked": (
                boundary_word_count
            ),

            "missing_after_merge": len(
                missing_after_merge
            ),

            "boundary_words_missing": len(
                boundary_missing
            ),
        }


    # ========================================================
    # NOVO TESTE:
    #
    # SINGLE-CHUNK RAW
    #       x
    # MULTI-CHUNK RAW
    #
    # Procura BURACOS NOVOS próximos às fronteiras.
    # ========================================================

    # ========================================================
    # COBERTURA DE INTERVALOS
    #
    # Retorna a fração de [start, end] coberta pela união dos
    # intervalos fornecidos.
    #
    # Serve para distinguir:
    #
    # "Whisper escreveu/segmentou diferente"
    #
    # de:
    #
    # "a fala realmente desapareceu no multi-chunk".
    # ========================================================

    def _interval_union_coverage(
        intervals,
        start,
        end,
    ):

        start = float(
            start
        )

        end = float(
            end
        )


        if (
            end
            <=
            start
        ):

            return 1.0


        clipped = []


        for interval_start, interval_end in (
            intervals
        ):

            interval_start = max(
                float(
                    interval_start
                ),
                start,
            )

            interval_end = min(
                float(
                    interval_end
                ),
                end,
            )


            if (
                interval_end
                >
                interval_start
            ):

                clipped.append(
                    (
                        interval_start,
                        interval_end,
                    )
                )


        if not clipped:

            return 0.0


        clipped.sort(
            key=lambda item: (
                item[0],
                item[1],
            )
        )


        merged = [
            [
                clipped[0][0],
                clipped[0][1],
            ]
        ]


        for interval_start, interval_end in (
            clipped[
                1:
            ]
        ):

            previous = (
                merged[
                    -1
                ]
            )


            if (
                interval_start
                <=
                previous[1]
            ):

                previous[1] = max(
                    previous[1],
                    interval_end,
                )


            else:

                merged.append(
                    [
                        interval_start,
                        interval_end,
                    ]
                )


        covered = sum(
            interval_end
            -
            interval_start

            for interval_start, interval_end
            in merged
        )


        return min(
            1.0,

            covered
            /
            (
                end
                -
                start
            ),
        )


    # ========================================================
    # TESTE:
    #
    # SINGLE-CHUNK RAW
    #       x
    # MULTI-CHUNK RAW
    #
    # O objetivo NÃO é exigir transcrição idêntica.
    #
    # Alterar o tamanho do contexto pode mudar:
    #
    # - tokenização;
    # - palavras reconhecidas;
    # - limites de palavras;
    # - limites de segmentos.
    #
    # O que precisa permanecer é a COBERTURA TEMPORAL DA FALA.
    #
    # Portanto um "buraco" só existe quando:
    #
    # 1. baseline tinha uma região de fala relevante;
    # 2. palavras baseline não encontram suporte individual;
    # 3. quase não existe segmento multi cobrindo a região;
    # 4. não existem words multi próximas.
    # ========================================================

    def validate_boundaries_against_single_chunk(
        baseline_raw,
        multichunk_raw,
        chunk_seconds,
    ):

        errors = []

        warnings = []


        baseline_words = (
            _acceptance_words(
                baseline_raw.get(
                    "segments",
                    []
                )
            )
        )


        multi_words = (
            _acceptance_words(
                multichunk_raw.get(
                    "segments",
                    []
                )
            )
        )


        baseline_segments = [
            {
                "start": float(
                    segment[
                        "start"
                    ]
                ),

                "end": float(
                    segment[
                        "end"
                    ]
                ),

                "text": str(
                    segment.get(
                        "text",
                        ""
                    )
                ),
            }

            for segment
            in (
                baseline_raw.get(
                    "segments",
                    []
                )
                or []
            )

            if (
                segment.get(
                    "start"
                )
                is not None

                and

                segment.get(
                    "end"
                )
                is not None
            )
        ]


        multi_segments = [
            {
                "start": float(
                    segment[
                        "start"
                    ]
                ),

                "end": float(
                    segment[
                        "end"
                    ]
                ),

                "text": str(
                    segment.get(
                        "text",
                        ""
                    )
                ),
            }

            for segment
            in (
                multichunk_raw.get(
                    "segments",
                    []
                )
                or []
            )

            if (
                segment.get(
                    "start"
                )
                is not None

                and

                segment.get(
                    "end"
                )
                is not None
            )
        ]


        duration = min(

            float(
                baseline_raw.get(
                    "source",
                    {}
                ).get(
                    "duration_seconds",
                    0.0
                )
                or 0.0
            ),

            float(
                multichunk_raw.get(
                    "source",
                    {}
                ).get(
                    "duration_seconds",
                    0.0
                )
                or 0.0
            ),
        )


        boundaries = []

        boundary = float(
            chunk_seconds
        )


        while (
            boundary
            <
            duration
        ):

            boundaries.append(
                boundary
            )

            boundary += float(
                chunk_seconds
            )


        checked_words = 0

        unmatched_words = []

        significant_holes = []

        coverage_checks = []


        for boundary in (
            boundaries
        ):

            window_start = (
                boundary
                -
                MULTICHUNK_BOUNDARY_WINDOW
            )

            window_end = (
                boundary
                +
                MULTICHUNK_BOUNDARY_WINDOW
            )


            baseline_window = [
                word

                for word
                in baseline_words

                if (
                    window_start
                    <=
                    word[
                        "midpoint"
                    ]
                    <=
                    window_end
                )
            ]


            multi_window = [
                word

                for word
                in multi_words

                if (
                    window_start
                    -
                    1.0

                    <=

                    word[
                        "midpoint"
                    ]

                    <=

                    window_end
                    +
                    1.0
                )
            ]


            checked_words += len(
                baseline_window
            )


            # ====================================================
            # Caso extremo:
            #
            # baseline tem fala, mas nenhuma word multi aparece
            # na janela.
            #
            # Ainda assim verificamos cobertura por SEGMENTOS
            # antes de declarar um buraco.
            # ====================================================

            if (
                baseline_window

                and

                not multi_window
            ):

                group_start = min(
                    word[
                        "start"
                    ]

                    for word
                    in baseline_window
                )


                group_end = max(
                    word[
                        "end"
                    ]

                    for word
                    in baseline_window
                )


                nearby_multi_segments = [
                    segment

                    for segment
                    in multi_segments

                    if (
                        segment[
                            "end"
                        ]
                        >=
                        group_start
                        -
                        ACCEPTANCE_SPEECH_COVERAGE_SLACK

                        and

                        segment[
                            "start"
                        ]
                        <=
                        group_end
                        +
                        ACCEPTANCE_SPEECH_COVERAGE_SLACK
                    )
                ]


                coverage = (
                    _interval_union_coverage(

                        [
                            (
                                segment[
                                    "start"
                                ],

                                segment[
                                    "end"
                                ],
                            )

                            for segment
                            in nearby_multi_segments
                        ],

                        group_start,

                        group_end,
                    )
                )


                coverage_checks.append({
                    "boundary": (
                        boundary
                    ),

                    "start": (
                        group_start
                    ),

                    "end": (
                        group_end
                    ),

                    "duration": (
                        group_end
                        -
                        group_start
                    ),

                    "baseline_words": len(
                        baseline_window
                    ),

                    "multi_segment_coverage": round(
                        coverage,
                        4
                    ),

                    "nearby_multi_words": 0,
                })


                if (
                    coverage
                    <=
                    ACCEPTANCE_HOLE_MAX_MULTI_COVERAGE
                ):

                    significant_holes.append({
                        "boundary": (
                            boundary
                        ),

                        "start": (
                            group_start
                        ),

                        "end": (
                            group_end
                        ),

                        "duration": (
                            group_end
                            -
                            group_start
                        ),

                        "words": [
                            word[
                                "raw_word"
                            ]

                            for word
                            in baseline_window
                        ],

                        "multi_segment_coverage": round(
                            coverage,
                            4
                        ),

                        "nearby_multi_words": 0,

                        "reason": (
                            "baseline_has_speech_but_"
                            "multi_has_no_words_or_coverage"
                        ),
                    })


                else:

                    warnings.append(
                        f"Boundary {boundary:.3f}s: "
                        "baseline possui words sem "
                        "correspondência individual, "
                        "mas a região continua coberta "
                        "por segmentos multi "
                        f"(coverage={coverage:.3f})."
                    )


                continue


            # ====================================================
            # PALAVRAS BASELINE SEM SUPORTE INDIVIDUAL
            # ====================================================

            local_unmatched = []


            for baseline_word in (
                baseline_window
            ):

                if not (
                    _baseline_word_has_support(
                        baseline_word,
                        multi_window,
                    )
                ):

                    local_unmatched.append(
                        baseline_word
                    )


                    unmatched_words.append({
                        "boundary": (
                            boundary
                        ),

                        **baseline_word,
                    })


            # ====================================================
            # Agrupar ausências consecutivas.
            #
            # Palavra isolada divergente é warning.
            #
            # Um grupo só será buraco se, além disso,
            # não houver cobertura temporal multi.
            # ====================================================

            unmatched_groups = (
                _group_unmatched_words(
                    local_unmatched
                )
            )


            for group in (
                unmatched_groups
            ):

                group_start = (
                    group[
                        0
                    ][
                        "start"
                    ]
                )


                group_end = (
                    group[
                        -1
                    ][
                        "end"
                    ]
                )


                group_duration = (
                    group_end
                    -
                    group_start
                )


                nearby_multi_segments = [
                    segment

                    for segment
                    in multi_segments

                    if (
                        segment[
                            "end"
                        ]
                        >=
                        group_start
                        -
                        ACCEPTANCE_SPEECH_COVERAGE_SLACK

                        and

                        segment[
                            "start"
                        ]
                        <=
                        group_end
                        +
                        ACCEPTANCE_SPEECH_COVERAGE_SLACK
                    )
                ]


                coverage = (
                    _interval_union_coverage(

                        [
                            (
                                segment[
                                    "start"
                                ],

                                segment[
                                    "end"
                                ],
                            )

                            for segment
                            in nearby_multi_segments
                        ],

                        group_start,

                        group_end,
                    )
                )


                nearby_multi_words = [
                    word

                    for word
                    in multi_words

                    if (
                        group_start
                        -
                        ACCEPTANCE_HOLE_NEARBY_WORD_TOLERANCE

                        <=

                        word[
                            "midpoint"
                        ]

                        <=

                        group_end
                        +
                        ACCEPTANCE_HOLE_NEARBY_WORD_TOLERANCE
                    )
                ]


                coverage_checks.append({
                    "boundary": (
                        boundary
                    ),

                    "start": (
                        group_start
                    ),

                    "end": (
                        group_end
                    ),

                    "duration": (
                        group_duration
                    ),

                    "baseline_words": len(
                        group
                    ),

                    "multi_segment_coverage": round(
                        coverage,
                        4
                    ),

                    "nearby_multi_words": len(
                        nearby_multi_words
                    ),
                })


                significant = (

                    len(
                        group
                    )
                    >=
                    ACCEPTANCE_SIGNIFICANT_HOLE_MIN_WORDS

                    and

                    group_duration
                    >=
                    ACCEPTANCE_SIGNIFICANT_HOLE_MIN_DURATION

                    and

                    coverage
                    <=
                    ACCEPTANCE_HOLE_MAX_MULTI_COVERAGE

                    and

                    len(
                        nearby_multi_words
                    )
                    ==
                    0
                )


                if (
                    significant
                ):

                    significant_holes.append({
                        "boundary": (
                            boundary
                        ),

                        "start": (
                            group_start
                        ),

                        "end": (
                            group_end
                        ),

                        "duration": (
                            group_duration
                        ),

                        "words": [
                            word[
                                "raw_word"
                            ]

                            for word
                            in group
                        ],

                        "multi_segment_coverage": round(
                            coverage,
                            4
                        ),

                        "nearby_multi_words": len(
                            nearby_multi_words
                        ),

                        "reason": (
                            "baseline_speech_region_"
                            "missing_in_multi"
                        ),
                    })


                else:

                    warnings.append(
                        f"Boundary {boundary:.3f}s: "
                        f"{len(group)} palavra(s) baseline "
                        "sem correspondência forte "
                        f"@ {group_start:.3f}-"
                        f"{group_end:.3f}s, "
                        "mas há cobertura temporal multi "
                        f"(coverage={coverage:.3f}, "
                        "nearby_words="
                        f"{len(nearby_multi_words)})."
                    )


        if (
            significant_holes
        ):

            errors.append(
                f"{len(significant_holes)} "
                "buraco(s) temporal(is) real(is) "
                "detectado(s) nas fronteiras "
                "quando comparado ao baseline "
                "single-chunk."
            )


        return {
            "valid": (
                len(
                    errors
                )
                ==
                0
            ),

            "errors": (
                errors
            ),

            "warnings": (
                warnings
            ),

            "boundaries_checked": len(
                boundaries
            ),

            "baseline_words_checked": (
                checked_words
            ),

            "unmatched_baseline_words": len(
                unmatched_words
            ),

            "significant_holes": (
                significant_holes
            ),

            "coverage_checks": (
                coverage_checks
            ),
        }


    # ========================================================
    # NOVO TESTE FORTE DE REPAIR
    #
    # repair_id
    #    ↓
    # repair_pass.results
    #    ↓
    # core_start / core_end
    #    ↓
    # final segments origin="repair"
    #    ↓
    # words absolutas
    #
    # Isto prova diretamente que o contexto usado pelo repair
    # NÃO vazou como timestamp relativo.
    # ========================================================

    def validate_repair_absolute_timestamps(
        repaired,
        chunk_seconds,
    ):

        errors = []
        warnings = []


        duration = float(
            repaired.get(
                "source",
                {}
            ).get(
                "duration_seconds",
                0.0
            )
            or 0.0
        )


        repair_results = (
            repaired.get(
                "repair_pass",
                {}
            ).get(
                "results",
                []
            )
            or []
        )


        audit_by_id = {}

        for repair in (
            repair_results
        ):

            repair_id = (
                repair.get(
                    "repair_id"
                )
            )

            if (
                repair_id
                is None
            ):

                continue


            if (
                repair.get(
                    "action"
                )
                ==
                "replace"
            ):

                audit_by_id[
                    repair_id
                ] = (
                    repair
                )


        # ====================================================
        # FINAL SEGMENTS POR REPAIR ID
        # ====================================================

        final_by_repair_id = defaultdict(
            list
        )


        for segment in (
            repaired.get(
                "segments",
                []
            )
        ):

            if (
                segment.get(
                    "origin"
                )
                !=
                "repair"
            ):

                continue


            repair_id = (
                segment.get(
                    "repair_id"
                )
            )


            if (
                repair_id
                is None
            ):

                errors.append(
                    f"Segmento repair "
                    f"{segment.get('id')} "
                    "sem repair_id."
                )

                continue


            final_by_repair_id[
                repair_id
            ].append(
                segment
            )


        # ====================================================
        # REPAIR FINAL TEM AUDITORIA?
        # ====================================================

        for repair_id in (
            final_by_repair_id
        ):

            if (
                repair_id
                not in audit_by_id
            ):

                errors.append(
                    f"repair_id {repair_id}: "
                    "segmento final existe, "
                    "mas auditoria replace correspondente "
                    "não foi encontrada."
                )


        # ====================================================
        # AUDITORIA REPLACE TEM SEGMENTO FINAL?
        # ====================================================

        near_boundary_repairs = []

        repair_segments_checked = 0

        repair_words_checked = 0


        for repair_id, audit in (
            audit_by_id.items()
        ):

            core_start = float(
                audit.get(
                    "core_start",
                    0.0
                )
            )

            core_end = float(
                audit.get(
                    "core_end",
                    core_start
                )
            )


            final_segments = (
                final_by_repair_id.get(
                    repair_id,
                    []
                )
            )


            if not final_segments:

                errors.append(
                    f"repair_id {repair_id}: "
                    "action=replace mas nenhum "
                    "segmento final origin=repair."
                )

                continue


            # ------------------------------------------------
            # Repair próximo de alguma fronteira?
            # ------------------------------------------------

            boundary = float(
                chunk_seconds
            )


            while (
                boundary
                <
                duration
            ):

                distance = min(
                    abs(
                        core_start
                        -
                        boundary
                    ),

                    abs(
                        core_end
                        -
                        boundary
                    ),

                    (
                        0.0
                        if (
                            core_start
                            <=
                            boundary
                            <=
                            core_end
                        )
                        else float(
                            "inf"
                        )
                    ),
                )


                if (
                    distance
                    <=
                    MULTICHUNK_BOUNDARY_WINDOW
                ):

                    near_boundary_repairs.append({
                        "repair_id": (
                            repair_id
                        ),

                        "boundary": (
                            boundary
                        ),

                        "core_start": (
                            core_start
                        ),

                        "core_end": (
                            core_end
                        ),

                        "distance": (
                            distance
                        ),
                    })

                    break


                boundary += float(
                    chunk_seconds
                )


            # =================================================
            # VALIDAR CADA SEGMENTO PRODUZIDO PELO REPAIR
            # =================================================

            for segment in (
                final_segments
            ):

                repair_segments_checked += 1


                start = float(
                    segment[
                        "start"
                    ]
                )

                end = float(
                    segment[
                        "end"
                    ]
                )


                # --------------------------------------------
                # ESSENCIAL:
                #
                # segmento repair não pode estar fora do core.
                #
                # Se um repair de ~600s voltasse como 2s,
                # falharia aqui imediatamente.
                # --------------------------------------------

                if (
                    start
                    <
                    core_start
                    -
                    ACCEPTANCE_REPAIR_CORE_TOLERANCE
                ):

                    errors.append(
                        f"repair_id {repair_id}, "
                        f"segment {segment.get('id')}: "
                        f"start={start:.3f} "
                        f"antes do core_start="
                        f"{core_start:.3f}"
                    )


                if (
                    end
                    >
                    core_end
                    +
                    ACCEPTANCE_REPAIR_CORE_TOLERANCE
                ):

                    errors.append(
                        f"repair_id {repair_id}, "
                        f"segment {segment.get('id')}: "
                        f"end={end:.3f} "
                        f"depois do core_end="
                        f"{core_end:.3f}"
                    )


                if (
                    end
                    <
                    core_start
                    -
                    ACCEPTANCE_REPAIR_CORE_TOLERANCE

                    or

                    start
                    >
                    core_end
                    +
                    ACCEPTANCE_REPAIR_CORE_TOLERANCE
                ):

                    errors.append(
                        f"repair_id {repair_id}, "
                        f"segment {segment.get('id')}: "
                        "não intersecta o core original."
                    )


                if (
                    start
                    <
                    -ACCEPTANCE_REPAIR_CORE_TOLERANCE

                    or

                    end
                    >
                    duration
                    +
                    ACCEPTANCE_REPAIR_CORE_TOLERANCE
                ):

                    errors.append(
                        f"repair_id {repair_id}, "
                        f"segment {segment.get('id')}: "
                        "timestamp fora da mídia."
                    )


                words = (
                    segment.get(
                        "words"
                    )
                    or []
                )


                if not words:

                    errors.append(
                        f"repair_id {repair_id}, "
                        f"segment {segment.get('id')}: "
                        "repair final sem words."
                    )

                    continue


                # ============================================
                # WORDS DO REPAIR
                # ============================================

                for word in (
                    words
                ):

                    repair_words_checked += 1


                    if (
                        word.get("start")
                        is None

                        or

                        word.get("end")
                        is None
                    ):

                        errors.append(
                            f"repair_id {repair_id}: "
                            "word sem timestamps."
                        )

                        continue


                    w_start = float(
                        word[
                            "start"
                        ]
                    )

                    w_end = float(
                        word[
                            "end"
                        ]
                    )


                    midpoint = (
                        (
                            w_start
                            +
                            w_end
                        )
                        /
                        2.0
                    )


                    # ----------------------------------------
                    # Palavra precisa estar na mídia.
                    # ----------------------------------------

                    if (
                        w_start
                        <
                        0

                        or

                        w_end
                        >
                        duration
                        +
                        ACCEPTANCE_REPAIR_CORE_TOLERANCE
                    ):

                        errors.append(
                            f"repair_id {repair_id}: "
                            f"word {word.get('word')!r} "
                            "fora da timeline da mídia."
                        )


                    # ----------------------------------------
                    # Palavra precisa pertencer ao segmento.
                    # ----------------------------------------

                    if (
                        w_start
                        <
                        start
                        -
                        ACCEPTANCE_REPAIR_CORE_TOLERANCE

                        or

                        w_end
                        >
                        end
                        +
                        ACCEPTANCE_REPAIR_CORE_TOLERANCE
                    ):

                        errors.append(
                            f"repair_id {repair_id}: "
                            f"word {word.get('word')!r} "
                            "fora do segmento repair."
                        )


                    # ----------------------------------------
                    # Principal teste contra timestamp relativo.
                    # ----------------------------------------

                    if not (
                        core_start
                        -
                        ACCEPTANCE_REPAIR_CORE_TOLERANCE

                        <=

                        midpoint

                        <=

                        core_end
                        +
                        ACCEPTANCE_REPAIR_CORE_TOLERANCE
                    ):

                        errors.append(
                            f"repair_id {repair_id}: "
                            f"word {word.get('word')!r} "
                            f"@ {w_start:.3f}s "
                            "fora do core absoluto "
                            f"{core_start:.3f}-"
                            f"{core_end:.3f}s."
                        )


        return {
            "valid": (
                len(
                    errors
                )
                ==
                0
            ),

            "errors": (
                errors
            ),

            "warnings": (
                warnings
            ),

            "replacements_in_audit": len(
                audit_by_id
            ),

            "repair_segments_checked": (
                repair_segments_checked
            ),

            "repair_words_checked": (
                repair_words_checked
            ),

            "near_boundary_repairs": (
                near_boundary_repairs
            ),

            "near_boundary_repairs_count": len(
                near_boundary_repairs
            ),
        }


    # ========================================================
    # VALIDAR SCHEMA UNIFORME INDEPENDENTEMENTE
    # ========================================================

    def validate_uniform_final_schema(
        transcription
    ):

        errors = []


        required_segment_fields = {
            "id",
            "start",
            "end",
            "text",
            "words",
            "word_alignment_missing",
            "avg_logprob",
            "compression_ratio",
            "no_speech_prob",
            "temperature",
            "origin",
            "repair_id",
            "repair_model",
            "repair_confidence_class",
        }


        required_word_fields = {
            "word",
            "start",
            "end",
            "probability",
        }


        segment_ids = set()


        for index, segment in enumerate(
            transcription.get(
                "segments",
                []
            )
        ):

            missing = (
                required_segment_fields
                -
                set(
                    segment.keys()
                )
            )


            if (
                missing
            ):

                errors.append(
                    f"segment[{index}]: "
                    "campos ausentes: "
                    +
                    ", ".join(
                        sorted(
                            missing
                        )
                    )
                )


            segment_id = (
                segment.get(
                    "id"
                )
            )


            if (
                segment_id
                in segment_ids
            ):

                errors.append(
                    f"id duplicado: "
                    f"{segment_id}"
                )


            segment_ids.add(
                segment_id
            )


            if (
                segment.get(
                    "origin"
                )
                not in (
                    "initial",
                    "repair",
                )
            ):

                errors.append(
                    f"segment {segment_id}: "
                    f"origin inválido "
                    f"{segment.get('origin')!r}"
                )


            for word_index, word in enumerate(
                segment.get(
                    "words",
                    []
                )
            ):

                missing_word_fields = (
                    required_word_fields
                    -
                    set(
                        word.keys()
                    )
                )


                if (
                    missing_word_fields
                ):

                    errors.append(
                        f"segment {segment_id} "
                        f"word[{word_index}]: "
                        "campos ausentes: "
                        +
                        ", ".join(
                            sorted(
                                missing_word_fields
                            )
                        )
                    )


        return {
            "valid": (
                len(
                    errors
                )
                ==
                0
            ),

            "errors": (
                errors
            ),

            "segments_checked": len(
                transcription.get(
                    "segments",
                    []
                )
            ),

            "unique_ids": len(
                segment_ids
            ),
        }


    # ========================================================
    # VALIDAÇÃO REAL DOS ARQUIVOS SALVOS
    #
    # Reabre JSON/TXT/SRT DO DISCO.
    #
    # Não confia apenas no estado em memória.
    # ========================================================

    def validate_saved_outputs(
        source_path,
        test_mode=True,
        repaired=True,
    ):

        paths = job_paths(
            source_path,
            test_mode=test_mode,
        )


        if (
            repaired
        ):

            json_path = (
                paths[
                    "final_json"
                ].with_name(
                    paths[
                        "final_json"
                    ].stem
                    +
                    ".REPAIRED.json"
                )
            )

            txt_path = (
                paths[
                    "final_json"
                ].with_name(
                    paths[
                        "final_json"
                    ].stem
                    +
                    ".REPAIRED.txt"
                )
            )

            srt_path = (
                paths[
                    "final_json"
                ].with_name(
                    paths[
                        "final_json"
                    ].stem
                    +
                    ".REPAIRED.srt"
                )
            )

        else:

            json_path = (
                paths[
                    "final_json"
                ]
            )

            txt_path = (
                paths[
                    "txt"
                ]
            )

            srt_path = (
                paths[
                    "srt"
                ]
            )


        errors = []


        if not (
            json_path.exists()
        ):

            errors.append(
                f"JSON ausente: "
                f"{json_path}"
            )


        if not (
            txt_path.exists()
        ):

            errors.append(
                f"TXT ausente: "
                f"{txt_path}"
            )


        if not (
            srt_path.exists()
        ):

            errors.append(
                f"SRT ausente: "
                f"{srt_path}"
            )


        if errors:

            return {
                "valid": False,

                "errors": (
                    errors
                ),

                "json_matches_segments": False,

                "txt_matches_json": False,

                "srt_matches_json": False,
            }


        saved = load_json(
            json_path,
            {},
        )


        segments = (
            saved.get(
                "segments",
                []
            )
        )


        expected_txt = (
            render_final_txt(
                segments
            )
        )


        expected_srt = (
            render_final_srt(
                segments
            )
        )


        actual_txt = (
            txt_path.read_text(
                encoding="utf-8"
            )
        )


        actual_srt = (
            srt_path.read_text(
                encoding="utf-8"
            )
        )


        calculated_hash = (
            final_segments_sha256(
                segments
            )
        )


        stored_hash = (
            saved.get(
                "final_segments_sha256"
            )
        )


        hash_valid = (
            stored_hash
            ==
            calculated_hash
        )


        txt_valid = (
            actual_txt
            ==
            expected_txt
        )


        srt_valid = (
            actual_srt
            ==
            expected_srt
        )


        if not hash_valid:

            errors.append(
                "Hash dos final_segments "
                "não corresponde ao JSON salvo."
            )


        if not txt_valid:

            errors.append(
                "TXT não corresponde aos "
                "segments do JSON salvo."
            )


        if not srt_valid:

            errors.append(
                "SRT não corresponde aos "
                "segments do JSON salvo."
            )


        return {
            "valid": (
                len(
                    errors
                )
                ==
                0
            ),

            "errors": (
                errors
            ),

            "json_matches_segments": (
                hash_valid
            ),

            "txt_matches_json": (
                txt_valid
            ),

            "srt_matches_json": (
                srt_valid
            ),
        }


    # ========================================================
    # SALVAR CONFIGURAÇÃO NORMAL
    # ========================================================

    _original_chunk_seconds = (
        CHUNK_SECONDS
    )

    _original_overlap_seconds = (
        OVERLAP_SECONDS
    )

    _original_test_seconds = (
        TEST_SECONDS
    )


    try:

        source_duration = (
            media_duration(
                TEST_SOURCE
            )
        )


        # ====================================================
        # ETAPA 1
        # BASELINE SINGLE-CHUNK
        #
        # Somente RAW.
        #
        # Não executamos repair aqui porque o objetivo desta
        # referência é validar exclusivamente:
        #
        # chunking + overlap + merge.
        #
        # Isso evita gastar uma segunda Passada 2 completa.
        # ====================================================

        print()
        print("=" * 80)
        print("FASE 1 — BASELINE SINGLE-CHUNK RAW")
        print("=" * 80)


        CHUNK_SECONDS = max(
            float(
                _original_chunk_seconds
            ),

            math.ceil(
                source_duration
            )
            +
            1,
        )


        OVERLAP_SECONDS = (
            _original_overlap_seconds
        )


        TEST_SECONDS = (
            source_duration
        )


        print(
            "Arquivo:",
            Path(
                TEST_SOURCE
            ).name
        )

        print(
            "Duração:",
            source_duration
        )

        print(
            "CHUNK_SECONDS baseline:",
            CHUNK_SECONDS
        )

        print(
            "Chunks esperados: 1"
        )


        baseline_raw = (
            process_live(
                TEST_SOURCE,

                force=True,

                test_mode=True,

                repair=False,
            )
        )


        if (
            baseline_raw
            is None
        ):

            raise RuntimeError(
                "Baseline single-chunk retornou None."
            )


        baseline_raw = copy.deepcopy(
            baseline_raw
        )


        baseline_chunks = (
            baseline_raw.get(
                "checkpointing",
                {}
            ).get(
                "chunks_completed",
                0
            )
        )


        if (
            baseline_chunks
            !=
            1
        ):

            raise RuntimeError(
                "Baseline deveria possuir exatamente "
                "1 chunk, mas possui "
                f"{baseline_chunks}."
            )


        print()
        print(
            "✓ Baseline single-chunk preservado em memória."
        )


        # ====================================================
        # ETAPA 2
        # MULTI-CHUNK + OVERLAP + REPAIR
        # ====================================================

        CHUNK_SECONDS = (
            MULTICHUNK_TEST_CHUNK_SECONDS
        )


        OVERLAP_SECONDS = (
            MULTICHUNK_TEST_OVERLAP_SECONDS
        )


        TEST_SECONDS = (
            source_duration
        )


        print()
        print("=" * 80)
        print(
            "FASE 1 — TESTE DE ACEITAÇÃO MULTI-CHUNK"
        )
        print("=" * 80)


        print(
            "Arquivo:",
            Path(
                TEST_SOURCE
            ).name
        )

        print(
            "Duração:",
            TEST_SECONDS
        )

        print(
            "CHUNK_SECONDS:",
            CHUNK_SECONDS
        )

        print(
            "OVERLAP_SECONDS:",
            OVERLAP_SECONDS
        )

        print(
            "Chunks esperados:",
            math.ceil(
                TEST_SECONDS
                /
                CHUNK_SECONDS
            )
        )

        print()


        acceptance_result = (
            process_live(
                TEST_SOURCE,

                force=True,

                test_mode=True,

                repair=True,
            )
        )


        if (
            acceptance_result
            is None
        ):

            raise RuntimeError(
                "process_live multi-chunk retornou None."
            )


        # Mantém o comportamento habitual da célula.
        result = (
            acceptance_result
        )


        # ====================================================
        # CARREGAR RAW MULTI-CHUNK
        #
        # process_live retornou REPAIRED,
        # mas o RAW continua salvo separadamente.
        # ====================================================

        current_paths = (
            job_paths(
                TEST_SOURCE,
                test_mode=True,
            )
        )


        multichunk_raw = (
            load_json(
                current_paths[
                    "final_json"
                ],
                {},
            )
        )


        if not (
            multichunk_raw
        ):

            raise RuntimeError(
                "RAW multi-chunk não pôde ser carregado."
            )


        # ====================================================
        # TESTE 1
        # CHUNKS / OFFSETS / MERGE
        # ====================================================

        multichunk_validation = (
            validate_multichunk_checkpoints(
                TEST_SOURCE,
                test_mode=True,
            )
        )


        # ====================================================
        # TESTE 2
        # SINGLE-CHUNK x MULTI-CHUNK
        #
        # Detectar buracos novos nas bordas.
        # ====================================================

        baseline_boundary_validation = (
            validate_boundaries_against_single_chunk(
                baseline_raw,
                multichunk_raw,
                CHUNK_SECONDS,
            )
        )


        # ====================================================
        # TESTE 3
        # REPAIR ABSOLUTO
        # ====================================================

        repair_absolute_validation = (
            validate_repair_absolute_timestamps(
                acceptance_result,
                CHUNK_SECONDS,
            )
        )


        # ====================================================
        # TESTE 4
        # SCHEMA UNIFORME
        # ====================================================

        schema_validation = (
            validate_uniform_final_schema(
                acceptance_result
            )
        )


        # ====================================================
        # TESTE 5
        # ARQUIVOS SALVOS
        # ====================================================

        saved_output_validation = (
            validate_saved_outputs(
                TEST_SOURCE,
                test_mode=True,
                repaired=True,
            )
        )


        # ====================================================
        # VALIDAÇÕES EXISTENTES DA 11E
        # ====================================================

        timeline_validation = (
            acceptance_result.get(
                "timeline_validation",
                {}
            )
        )


        timeline_valid = (
            timeline_validation.get(
                "valid",
                False
            )
        )


        timeline_contract = (
            acceptance_result.get(
                "timeline",
                {}
            )
        )


        absolute_contract_valid = (
            timeline_contract.get(
                "reference"
            )
            ==
            "source_media"

            and

            timeline_contract.get(
                "timestamp_type"
            )
            ==
            "absolute"

            and

            timeline_contract.get(
                "unit"
            )
            ==
            "seconds"

            and

            timeline_contract.get(
                "zero_point"
            )
            ==
            "start_of_source_media"

            and

            timeline_contract.get(
                "chunk_offsets_applied"
            )
            is True

            and

            timeline_contract.get(
                "silence_removed_from_timeline"
            )
            is False

            and

            timeline_contract.get(
                "editorial_padding_applied"
            )
            is False
        )


        raw_duplicate_metrics = (
            multichunk_raw.get(
                "timeline_validation",
                {}
            ).get(
                "metrics",
                {}
            )
        )


        repaired_duplicate_metrics = (
            timeline_validation.get(
                "metrics",
                {}
            )
        )


        no_raw_duplicates = (
            raw_duplicate_metrics.get(
                "duplicate_segments",
                0
            )
            ==
            0

            and

            raw_duplicate_metrics.get(
                "duplicate_words",
                0
            )
            ==
            0
        )


        no_repaired_duplicates = (
            repaired_duplicate_metrics.get(
                "duplicate_segments",
                0
            )
            ==
            0

            and

            repaired_duplicate_metrics.get(
                "duplicate_words",
                0
            )
            ==
            0
        )


        # ====================================================
        # CRITÉRIOS A-H
        # ====================================================

        acceptance = {

            # ------------------------------------------------
            # A
            # Timestamp aponta para a mídia original.
            # ------------------------------------------------

            "A_timestamp_points_to_source_media": (
                absolute_contract_valid
                and
                timeline_valid
            ),


            # ------------------------------------------------
            # B
            # Isso continua verdadeiro em múltiplos chunks.
            # ------------------------------------------------

            "B_absolute_across_multiple_chunks": (
                multichunk_validation[
                    "valid"
                ]
            ),


            # ------------------------------------------------
            # C
            # Overlap não duplica conteúdo.
            #
            # Validamos RAW e REPAIRED.
            # ------------------------------------------------

            "C_overlap_has_no_duplicates": (
                no_raw_duplicates
                and
                no_repaired_duplicates
            ),


            # ------------------------------------------------
            # D
            # Overlap não remove fala.
            #
            # Agora possui DUAS provas:
            #
            # 1. checkpoint -> merged RAW
            # 2. single-chunk RAW -> multi-chunk RAW
            #
            # O segundo teste detecta buracos que poderiam
            # ocorrer ANTES mesmo do checkpoint oficial.
            # ------------------------------------------------

            "D_overlap_did_not_drop_boundary_speech": (
                multichunk_validation[
                    "missing_after_merge"
                ]
                ==
                0

                and

                multichunk_validation[
                    "boundary_words_missing"
                ]
                ==
                0

                and

                baseline_boundary_validation[
                    "valid"
                ]

                and

                len(
                    baseline_boundary_validation[
                        "significant_holes"
                    ]
                )
                ==
                0
            ),


            # ------------------------------------------------
            # E
            # Repair mantém timestamps absolutos.
            #
            # AGORA NÃO É MAIS:
            #
            #     E = timeline_valid
            #
            # Verificamos diretamente:
            #
            # repair_id
            # -> audit
            # -> core
            # -> final segment
            # -> final words
            #
            # E exigimos que pelo menos um repair aconteça
            # próximo de uma fronteira.
            # ------------------------------------------------

            "E_repair_preserves_absolute_timestamps": (
                repair_absolute_validation[
                    "valid"
                ]

                and

                repair_absolute_validation[
                    "near_boundary_repairs_count"
                ]
                >
                0
            ),


            # ------------------------------------------------
            # F
            # Initial / repair seguem mesmo schema.
            # ------------------------------------------------

            "F_uniform_segment_schema": (
                schema_validation[
                    "valid"
                ]
            ),


            # ------------------------------------------------
            # G
            # JSON, SRT e TXT são exatamente derivados
            # dos mesmos final segments salvos no disco.
            # ------------------------------------------------

            "G_json_srt_txt_same_final_segments": (
                saved_output_validation[
                    "valid"
                ]
            ),


            # ------------------------------------------------
            # H
            # Existe validação automática funcional.
            # ------------------------------------------------

            "H_automatic_timeline_validation": (
                "timeline_validation"
                in acceptance_result

                and

                timeline_valid
            ),
        }


        phase1_acceptance_passed = all(
            acceptance.values()
        )


        # ====================================================
        # RELATÓRIO
        # ====================================================

        print()
        print("=" * 80)
        print(
            "RESULTADO FINAL DO TESTE DE ACEITAÇÃO"
        )
        print("=" * 80)


        for name, value in (
            acceptance.items()
        ):

            print(
                (
                    "✓"
                    if value
                    else "✗"
                ),
                name,
            )


        print()
        print("-" * 80)

        print(
            "Baseline single-chunk:",
            baseline_chunks,
            "chunk"
        )

        print(
            "Chunks multi:",
            multichunk_validation[
                "chunks"
            ]
        )

        print(
            "Palavras oficiais nos checkpoints:",
            multichunk_validation[
                "checkpoint_words"
            ]
        )

        print(
            "Palavras verificadas nas bordas:",
            multichunk_validation[
                "boundary_words_checked"
            ]
        )

        print(
            "Palavras perdidas checkpoint -> merge:",
            multichunk_validation[
                "missing_after_merge"
            ]
        )

        print(
            "Palavras perdidas nas bordas:",
            multichunk_validation[
                "boundary_words_missing"
            ]
        )


        print()
        print(
            "Baseline words verificadas "
            "single x multi:",
            baseline_boundary_validation[
                "baseline_words_checked"
            ]
        )

        print(
            "Baseline words sem correspondência forte:",
            baseline_boundary_validation[
                "unmatched_baseline_words"
            ]
        )

        print(
            "Buracos significativos novos:",
            len(
                baseline_boundary_validation[
                    "significant_holes"
                ]
            )
        )


        # ====================================================
        # DETALHAR BURACOS TEMPORAIS REAIS
        # ====================================================

        if (
            baseline_boundary_validation[
                "significant_holes"
            ]
        ):

            print()
            print(
                "BURACOS TEMPORAIS DETECTADOS:"
            )


            for hole in (
                baseline_boundary_validation[
                    "significant_holes"
                ]
            ):

                print(
                    "  boundary="
                    f"{hole['boundary']:.3f}s | "
                    "interval="
                    f"{hole['start']:.3f}"
                    " -> "
                    f"{hole['end']:.3f}s | "
                    "duration="
                    f"{hole['duration']:.3f}s | "
                    "coverage_multi="
                    f"{hole['multi_segment_coverage']:.3f} | "
                    "nearby_words="
                    f"{hole.get('nearby_multi_words', 0)}"
                )

                print(
                    "    baseline words:",
                    " ".join(
                        hole.get(
                            "words",
                            []
                        )
                    )
                )

                print(
                    "    reason:",
                    hole.get(
                        "reason"
                    )
                )


        print()
        print(
            "Repairs replace na auditoria:",
            repair_absolute_validation[
                "replacements_in_audit"
            ]
        )

        print(
            "Segmentos repair verificados:",
            repair_absolute_validation[
                "repair_segments_checked"
            ]
        )

        print(
            "Words repair verificadas:",
            repair_absolute_validation[
                "repair_words_checked"
            ]
        )

        print(
            "Repairs próximos de fronteira:",
            repair_absolute_validation[
                "near_boundary_repairs_count"
            ]
        )


        print()
        print(
            "Schema final:",
            acceptance_result.get(
                "schema_version"
            )
        )

        print(
            "Segmentos schema verificados:",
            schema_validation[
                "segments_checked"
            ]
        )

        print(
            "IDs únicos:",
            schema_validation[
                "unique_ids"
            ]
        )


        print()
        print(
            "JSON hash:",
            (
                "VALID"
                if (
                    saved_output_validation[
                        "json_matches_segments"
                    ]
                )
                else "INVALID"
            )
        )

        print(
            "JSON ↔ TXT:",
            (
                "VALID"
                if (
                    saved_output_validation[
                        "txt_matches_json"
                    ]
                )
                else "INVALID"
            )
        )

        print(
            "JSON ↔ SRT:",
            (
                "VALID"
                if (
                    saved_output_validation[
                        "srt_matches_json"
                    ]
                )
                else "INVALID"
            )
        )


        print()
        print(
            "Timeline errors:",
            len(
                timeline_validation.get(
                    "errors",
                    []
                )
            )
        )

        print(
            "Timeline warnings:",
            len(
                timeline_validation.get(
                    "warnings",
                    []
                )
            )
        )


        # ====================================================
        # ERROS DETALHADOS
        # ====================================================

        validations_to_report = (
            (
                "MULTI-CHUNK",
                multichunk_validation,
            ),

            (
                "BASELINE x MULTI",
                baseline_boundary_validation,
            ),

            (
                "REPAIR ABSOLUTO",
                repair_absolute_validation,
            ),

            (
                "SCHEMA",
                schema_validation,
            ),

            (
                "OUTPUTS",
                saved_output_validation,
            ),
        )


        for validation_name, validation in (
            validations_to_report
        ):

            errors = (
                validation.get(
                    "errors",
                    []
                )
            )

            warnings = (
                validation.get(
                    "warnings",
                    []
                )
            )


            if errors:

                print()
                print(
                    f"ERROS — {validation_name}:"
                )

                for error in (
                    errors[
                        :30
                    ]
                ):

                    print(
                        "-",
                        error
                    )


            if warnings:

                print()
                print(
                    f"WARNINGS — {validation_name}:"
                )

                for warning in (
                    warnings[
                        :20
                    ]
                ):

                    print(
                        "-",
                        warning
                    )


        # ====================================================
        # REPAIRS DE FRONTEIRA
        # ====================================================

        if (
            repair_absolute_validation[
                "near_boundary_repairs"
            ]
        ):

            print()
            print(
                "REPAIRS PRÓXIMOS DE FRONTEIRA:"
            )


            for repair in (
                repair_absolute_validation[
                    "near_boundary_repairs"
                ]
            ):

                print(
                    "  repair_id="
                    f"{repair['repair_id']} | "
                    "boundary="
                    f"{repair['boundary']:.3f}s | "
                    "core="
                    f"{repair['core_start']:.3f}"
                    " -> "
                    f"{repair['core_end']:.3f}s | "
                    "dist="
                    f"{repair['distance']:.3f}s"
                )


        # ====================================================
        # RESULTADO FINAL
        # ====================================================

        print()
        print("=" * 80)


        if (
            phase1_acceptance_passed
        ):

            print(
                "✓ FASE 1 APROVADA"
            )

            print(
                "✓ TIMELINE ABSOLUTA VALIDADA"
            )

            print(
                "✓ MULTI-CHUNK VALIDADO"
            )

            print(
                "✓ OVERLAP VALIDADO CONTRA BASELINE"
            )

            print(
                "✓ REPAIR ABSOLUTO VALIDADO"
            )

            print(
                "✓ SCHEMA 1.1 VALIDADO"
            )

            print(
                "✓ JSON / TXT / SRT VALIDADOS"
            )

            print()
            print(
                "✓ RARU REVIVAL — FASE 1 "
                "PRONTA PARA A FASE 2"
            )


        else:

            print(
                "✗ FASE 1 AINDA NÃO PODE "
                "SER CONSIDERADA ENCERRADA"
            )


            if (
                repair_absolute_validation[
                    "near_boundary_repairs_count"
                ]
                ==
                0
            ):

                print()
                print(
                    "Nenhum repair ocorreu próximo "
                    "de uma fronteira."
                )

                print(
                    "Neste caso o pipeline pode estar "
                    "correto, mas o requisito "
                    "OVERLAP + REPAIR ainda não foi "
                    "efetivamente exercitado."
                )


        print("=" * 80)


    finally:

        # ====================================================
        # RESTAURAR CONFIGURAÇÃO NORMAL
        # ====================================================

        CHUNK_SECONDS = (
            _original_chunk_seconds
        )


        OVERLAP_SECONDS = (
            _original_overlap_seconds
        )


        TEST_SECONDS = (
            _original_test_seconds
        )


        print()
        print(
            "Configuração normal restaurada:"
        )

        print(
            "CHUNK_SECONDS =",
            CHUNK_SECONDS
        )

        print(
            "OVERLAP_SECONDS =",
            OVERLAP_SECONDS
        )

        print(
            "TEST_SECONDS =",
            TEST_SECONDS
        )

Lives encontradas:
[0] 1002 (1).mp4

FASE 1 — BASELINE SINGLE-CHUNK RAW
Arquivo: 1002 (1).mp4
Duração: 763.310998
CHUNK_SECONDS baseline: 900.0
Chunks esperados: 1
FORCE: checkpoints e outputs anteriores deste job foram removidos.
Liberando modelo antigo da Passada 2 antes da Passada 1...

Preparando runtime da Passada 1...
Tentando carregar: model=large-v3-turbo device=cuda compute=float16
OK.

PASSADA 1 — TRANSCRIÇÃO
Arquivo: 1002 (1).mp4
Modelo: large-v3-turbo
Device: cuda
Compute: float16
Modo teste: True
Repair após Passada 1: False

VRAM — antes da Passada 1
  total=15360 MiB | usado=1677 MiB | livre=13236 MiB

[1/1] 00:00:00.000 -> 00:12:43.311
  salvo | 391 segmentos | RTF=0.131 | cuda/float16

CONCLUÍDO
JSON: /content/drive/MyDrive/RaruRevival/Transcriptions/1002_1_.TEST.json
TXT : /content/drive/MyDrive/RaruRevival/Transcriptions/1002_1_.TEST.txt
SRT : /content/drive/MyDrive/RaruRevival/Transcriptions/1002_1_.TEST.srt
RTF : 0.131

✓ Baseline single-chunk preservado em memória

In [20]:
# ============================================================
# CÉLULA 13 — Processar uma única live inteira
# ============================================================

TEST_MODE = False

# Exemplo:
#
# result = process_live(
#     LIVES[0],
#     force=False,
#     test_mode=False,
#     repair=True,
# )


In [21]:
# ============================================================
# CÉLULA 14 — Processamento em lote
# ============================================================

def process_all_lives(
    force=False,
    repair=True
):
    lives = sorted([
        p
        for p in DIRS["lives"].iterdir()
        if (
            p.suffix.lower()
            in VIDEO_EXTS
        )
    ])

    results = []

    for index, live in enumerate(
        lives,
        start=1
    ):
        print()
        print(
            "#" * 80
        )

        print(
            f"[{index}/{len(lives)}] "
            f"{live.name}"
        )

        print(
            "#" * 80
        )

        try:
            result = process_live(
                live,
                force=force,
                test_mode=False,
                repair=repair,
            )

            results.append({
                "source": live.name,
                "ok": (
                    result
                    is not None
                ),
            })

        except Exception as exc:
            print(
                f"ERRO: "
                f"{type(exc).__name__}: "
                f"{exc}"
            )

            results.append({
                "source": live.name,
                "ok": False,
                "error": str(exc),
            })

    return results

# Exemplo:
# batch_results = process_all_lives(
#     force=False,
#     repair=True,
# )


In [22]:
raw_path = job_paths(
    LIVES[0],
    test_mode=True
)["final_json"]

raw = load_json(
    raw_path,
    {}
)

result = run_repair_pass(
    source_path=LIVES[0],
    transcription=raw,
    test_mode=True,
)


PASSADA 2 — QUALITY ANALYZER / REPAIR GROUPS V3
Segmentos suspeitos: 45
Repair Groups: 42
Segmentos absorvidos por agrupamento: 3

REPAIR GROUP 001
Contexto: 0.00s → 6.98s
CORE: 1.58s → 2.98s
Segmentos agrupados: [1]
Motivos: low_confidence_near_chunk_boundary, unexpected_cyrillic_script, unexpected_non_latin_script
Score original: 0.6234
Original: Опаска

[AUTO] VALID score=0.7555
eu posso ficar

[PT] VALID score=0.7555
eu posso ficar

[EN] VALID score=0.7575
I can stay

✓ REPAIR ACEITO
Vencedor: auto
Score: 0.7555
Confiança: medium

REPAIR GROUP 002
Contexto: 17.00s → 27.98s
CORE: 21.00s → 23.98s
Segmentos agrupados: [10]
Motivos: low_confidence_near_chunk_boundary
Score original: 0.8681
Original: Final Fall, jogo que ainda nem lançou, vai lançar quinto.

[AUTO] VALID score=0.9701
jogo que ainda nem lançou, vai lançar quinta

[PT] VALID score=0.9701
jogo que ainda nem lançou, vai lançar quinta

[EN] VALID score=0.7896
a game that hasn't even been released yet. It will be released on

## Fluxo operacional recomendado

### Primeiro teste
Execute as células em ordem e, na Célula 12:

```python
result = process_live(
    LIVES[0],
    force=True,
    test_mode=True,
    repair=True,
)
```

### Retomada
Se a Passada 1 terminar e a Passada 2 falhar:

```python
result = process_live(
    LIVES[0],
    force=False,
    test_mode=True,
    repair=True,
)
```

O RAW pronto será reutilizado.

### Produção
Depois de validar o teste:

```python
TEST_MODE = False

result = process_live(
    LIVES[0],
    force=False,
    test_mode=False,
    repair=True,
)
```

### Arquivos gerados

RAW:

- `NomeDaLive.json`
- `NomeDaLive.txt`
- `NomeDaLive.srt`

Repair v3:

- `NomeDaLive.REPAIRED.json`
- `NomeDaLive.REPAIRED.txt`
- `NomeDaLive.REPAIRED.srt`

O JSON `.REPAIRED` preserva auditoria de cada Repair Group, candidatos AUTO/PT/EN, decisão final e `confidence_class`.
